In [3]:
import arcpy, os, pandas as pd
aprx = arcpy.mp.ArcGISProject("CURRENT")
m = aprx.activeMap
gdb = aprx.defaultGeodatabase
arcpy.env.workspace = gdb
arcpy.env.overwriteOutput = True
print("Project gdb:", gdb, "| Spatial Analyst:", arcpy.CheckExtension("Spatial"))

for lyr in m.listLayers():
    if lyr.isGroupLayer or lyr.isBasemapLayer:
        continue
    try:
        d = arcpy.Describe(lyr); sr = d.spatialReference
        if lyr.isRasterLayer:
            r = arcpy.Raster(d.catalogPath)
            info = f"cell {r.meanCellWidth:.4g} {sr.linearUnitName or sr.angularUnitName}"
        else:
            info = f"n = {arcpy.management.GetCount(lyr)[0]}"
        print(f"{lyr.name:26s} | {sr.name} ({sr.factoryCode}) | {info}\n    {d.catalogPath}")
    except Exception as e:
        print(f"{lyr.name}: {e}")

Project gdb: C:\Users\DCifelli2\OneDrive - mercercounty.org\Documents\ArcGIS\Projects\Fernow-FM-SOM-DC-092026\Fernow-FM-SOM-DC-092026.gdb | Spatial Analyst: Available
fernow_lidar_2018.lasd: Failed to execute. Parameters are not valid.
ERROR 000840: The value is not a Table View.
ERROR 000840: The value is not a Raster Layer.
Failed to execute (GetCount).

wsb: ERROR 000229: Cannot open wsb
Failed to execute (GetCount).

tmp_wsx: ERROR 000229: Cannot open tmp_wsx
Failed to execute (GetCount).

sjx: ERROR 000229: Cannot open sjx
Failed to execute (GetCount).

nr: ERROR 000229: Cannot open nr
Failed to execute (GetCount).

ref_rain_gauges            | NAD_1983_UTM_Zone_17N (26917) | n = 14
    C:\Users\DCifelli2\OneDrive - mercercounty.org\Documents\ArcGIS\Projects\Fernow-FM-SOM-DC-092026\Fernow_SOM_master.gdb\ref_rain_gauges
ref_weirs                  | NAD_1983_UTM_Zone_17N (26917) | n = 10
    C:\Users\DCifelli2\OneDrive - mercercounty.org\Documents\ArcGIS\Projects\Fernow-FM-SOM-DC-09

In [2]:
where = "Scomp_name LIKE 'WS2%' OR Scomp_name LIKE 'WS4%' OR Scomp_name LIKE 'WS5%'"
arcpy.conversion.ExportFeatures("dc_growth_plots", "study_plots", where_clause=where)

# plot_id matching the tree inventory labels
code = "def lbl(s, n):\n    s = s.strip()\n    return ('WS5' if s.startswith('WS5') else s) + '-' + str(int(n))"
arcpy.management.CalculateField("study_plots", "plot_id", "lbl(!Scomp_name!, !Plot_num!)", "PYTHON3", code, field_type="TEXT")

# location source: exact USFS point vs field GPS
arcpy.analysis.Near("study_plots", "growth_plots")
arcpy.management.AlterField("study_plots", "NEAR_DIST", "usfs_offset_m", "usfs_offset_m")
arcpy.management.CalculateField("study_plots", "xy_source", "'USFS' if !usfs_offset_m! < 1 else 'GPS'", "PYTHON3", field_type="TEXT")

# watershed membership check
arcpy.analysis.SpatialJoin("study_plots", "watersheds_all", "study_plots_ws", "JOIN_ONE_TO_ONE", "KEEP_ALL", match_option="INTERSECT")

def to_df(fc):
    f = [x.name for x in arcpy.ListFields(fc) if x.type not in ("Geometry", "OID", "Blob", "Raster")]
    return pd.DataFrame(arcpy.da.SearchCursor(fc, f), columns=f)

df = to_df("study_plots_ws")
print(len(df), "plots")
df[["plot_id", "Ws_name", "xy_source", "usfs_offset_m"]]

17 plots


,plot_id,Ws_name,xy_source,usfs_offset_m
0,WS5-1,WS5,USFS,0.000038
1,WS5-2,WS5,USFS,0.000051
2,WS5-3,WS5,USFS,0.000047
3,WS5-4,WS5,USFS,0.000052
4,WS5-5,WS5,USFS,0.000042
5,WS4A-11,WS4,USFS,0.000016
6,WS2B-1,WS2,USFS,0.000050
7,WS2A-5,WS2,USFS,0.000059
8,WS4A-13,WS4,USFS,0.000057
9,WS4A-10,WS4,USFS,0.000051


In [3]:
import re
lyr = m.listLayers("dc_growth_plots")[0]
src = arcpy.Describe(lyr).catalogPath
oidf = arcpy.Describe(src).OIDFieldName
sel = lyr.getSelectionSet() or []
print("Layer filter:", lyr.definitionQuery or "none", "| selected:", len(sel))

rows = list(arcpy.da.SearchCursor(src, [oidf, "Scomp_name", "Plot_num"]))
for r in rows: print(r)
norm = lambda s: re.sub(r"[^A-Z0-9]", "", str(s or "").upper())
keep = [r[0] for r in rows if norm(r[1])[:3] in ("WS2", "WS4", "WS5")]
print(len(keep), "study plots kept")

arcpy.conversion.ExportFeatures(src, "study_plots", where_clause=f"{oidf} IN ({','.join(map(str, keep))})")
arcpy.management.DefineProjection("study_plots", arcpy.SpatialReference(26917))

code = ("import re\ndef lbl(s, n):\n    s = re.sub(r'[^A-Z0-9]', '', str(s).upper())\n"
        "    return ('WS5' if s.startswith('WS5') else s) + '-' + str(int(n))")
arcpy.management.CalculateField("study_plots", "plot_id", "lbl(!Scomp_name!, !Plot_num!)", "PYTHON3", code, field_type="TEXT")
arcpy.analysis.Near("study_plots", "growth_plots")
arcpy.management.AlterField("study_plots", "NEAR_DIST", "usfs_offset_m", "usfs_offset_m")
arcpy.management.CalculateField("study_plots", "xy_source", "'USFS' if !usfs_offset_m! < 1 else 'GPS'", "PYTHON3", field_type="TEXT")
arcpy.analysis.SpatialJoin("study_plots", "watersheds_all", "study_plots_ws", "JOIN_ONE_TO_ONE", "KEEP_ALL", match_option="INTERSECT")

df = to_df("study_plots_ws")
print(len(df), "plots | GPS:", df.loc[df.xy_source == "GPS", "plot_id"].tolist())
df[["plot_id", "Ws_name", "xy_source", "usfs_offset_m"]]

Layer filter: Scomp_name <> 'WS1' And Scomp_name <> '48-1' | selected: 0
(0, 'WS5A', 1)
(1, 'WS5A', 2)
(2, 'WS5A', 3)
(3, 'WS5A', 4)
(4, 'WS5A', 5)
(5, 'WS4A', 11)
(6, 'WS2B', 1)
(7, 'WS2A', 5)
(8, 'WS4A', 13)
(9, 'WS4A', 10)
(10, 'WS2B', 2)
(11, 'WS4A', 9)
(12, 'WS2A', 3)
(13, 'WS2B', 4)
(14, 'WS4C', 1)
(15, 'WS2A', 1)
(16, 'WS1', 7)
(17, 'WS1', 13)
(18, 'WS1', 3)
(19, 'WS4B', 1)
(20, 'WS1', 9)
(21, 'WS1', 4)
(22, 'WS1', 14)
(23, ' WS5', 6)
(24, '48-1', 1)
(25, '48-1', 1)
(26, '48-1', 1)
(27, '48-1', 1)
(28, '48-1', 1)
(29, '48-1', 1)
18 study plots kept
18 plots | GPS: ['WS4C-1', 'WS4B-1', 'WS5-6']


,plot_id,Ws_name,xy_source,usfs_offset_m
0,WS5-1,WS5,USFS,0.000038
1,WS5-2,WS5,USFS,0.000051
2,WS5-3,WS5,USFS,0.000047
3,WS5-4,WS5,USFS,0.000052
4,WS5-5,WS5,USFS,0.000042
5,WS4A-11,WS4,USFS,0.000016
6,WS2B-1,WS2,USFS,0.000050
7,WS2A-5,WS2,USFS,0.000059
8,WS4A-13,WS4,USFS,0.000057
9,WS4A-10,WS4,USFS,0.000051


In [4]:
import math
from arcpy.sa import *
dem_src = arcpy.Describe(m.listLayers("DEM_Tucker_County.TIF")[0]).catalogPath

arcpy.analysis.PairwiseBuffer("watersheds_all", "ws_buf300", "300 Meters", dissolve_option="ALL")
arcpy.env.extent = arcpy.Describe("ws_buf300").extent
dem1 = ExtractByMask(dem_src, "ws_buf300")
dem5 = Aggregate(dem1, 5, "MEAN");                        dem5.save("dem5")
slp = Slope(dem5, "DEGREE");                               slp.save("slope5")
asp = Aspect(dem5)
Con(asp >= 0, Cos(asp * math.pi / 180)).save("north5")
Con(asp >= 0, Sin(asp * math.pi / 180)).save("east5")
(dem5 - FocalStatistics(dem5, NbrCircle(50, "MAP"), "MEAN")).save("tpi50")
(dem5 - FocalStatistics(dem5, NbrCircle(200, "MAP"), "MEAN")).save("tpi200")
fdir = FlowDirection(Fill(dem5), "NORMAL", None, "DINF")
facc = FlowAccumulation(fdir, None, "FLOAT", "DINF")
Ln(((facc + 1) * 5) / Tan(Con(slp < 0.1, 0.1, slp) * math.pi / 180)).save("twi5")

rasters = ["dem5", "slope5", "north5", "east5", "tpi50", "tpi200", "twi5"]
pts = {}
with arcpy.da.SearchCursor("study_plots", ["plot_id", "SHAPE@X", "SHAPE@Y"]) as cur:
    for pid, x, y in cur:
        pts[pid] = {r: float(arcpy.management.GetCellValue(r, f"{x} {y}").getOutput(0).replace(",", "")) for r in rasters}
terr = pd.DataFrame(pts).T.add_suffix("_pt")

arcpy.analysis.PairwiseBuffer("study_plots", "plot_fp", "11.35 Meters")
for r in rasters:
    ZonalStatisticsAsTable("plot_fp", "plot_id", r, "zs_tmp", "DATA", "MEAN")
    z = pd.DataFrame(arcpy.da.SearchCursor("zs_tmp", ["plot_id", "MEAN"]), columns=["plot_id", f"{r}_fp"]).set_index("plot_id")
    terr = terr.join(z)
arcpy.env.extent = None

terr = df.set_index("plot_id")[["Ws_name", "xy_source"]].join(terr)
terr.to_csv(os.path.join(os.path.dirname(gdb), "study_plots_terrain.csv"))
print(terr.groupby("Ws_name")[[c for c in terr if c.endswith("_fp")]].mean().round(2).T)
terr.round(2)

Ws_name       WS2     WS4     WS5
dem5_fp    759.13  829.94  778.72
slope5_fp   20.90    9.36   17.34
north5_fp   -0.40   -0.51    0.12
east5_fp    -0.14    0.73    0.35
tpi50_fp    -0.79   -0.15    0.96
tpi200_fp   -7.79    1.93   -0.92
twi5_fp      5.45    6.26    5.35


,Ws_name,xy_source,dem5_pt,slope5_pt,north5_pt,east5_pt,tpi50_pt,tpi200_pt,twi5_pt,dem5_fp,slope5_fp,north5_fp,east5_fp,tpi50_fp,tpi200_fp,twi5_fp
plot_id,,,,,,,,,,,,,,,,
WS5-1,WS5,USFS,779.58,8.98,-0.26,-0.96,1.42,-10.02,5.50,779.52,9.09,-0.18,-0.98,1.45,-9.85,5.59
WS5-2,WS5,USFS,780.70,19.32,0.57,0.82,1.36,1.49,4.29,780.18,19.73,0.57,0.81,1.17,1.02,4.40
WS5-3,WS5,USFS,782.44,19.52,-0.04,1.00,2.17,0.48,5.48,782.66,19.64,-0.07,0.98,1.48,0.39,5.65
WS5-4,WS5,USFS,767.76,17.07,0.40,0.92,-3.95,-10.27,5.88,768.03,14.74,0.03,0.85,-3.85,-10.14,7.29
WS5-5,WS5,USFS,748.18,19.97,-0.57,0.82,0.37,-4.57,5.31,747.62,18.59,-0.54,0.84,0.22,-5.28,5.51
WS4A-11,WS4,USFS,831.34,4.31,-0.31,0.95,1.11,4.51,4.19,830.88,5.13,-0.37,0.90,0.84,4.33,6.56
WS2B-1,WS2,USFS,733.05,18.22,-0.27,-0.96,-0.55,-5.71,5.56,732.35,20.11,-0.26,-0.97,-0.86,-6.59,5.18
WS2A-5,WS2,USFS,745.42,26.18,-0.43,0.90,0.47,-5.13,5.47,744.99,25.71,-0.43,0.90,0.63,-5.67,5.37
WS4A-13,WS4,USFS,765.35,9.20,-0.72,0.69,-4.10,-21.64,7.71,765.94,10.33,-0.75,0.63,-3.92,-21.35,7.37


In [6]:


# ---- Cleanup: shapefile attributes are unfinished manual edits -> keep geometry only ----
for fc, flds in [("ref_usfs_growth_plots", ["C_gm2", "N_gm2", "Plot_type", "Size_ac"]),
                 ("ref_dc_growth_plots_all", ["LF_C", "Plot_type", "Size_ac"]),
                 ("usfs_plot_terrain", ["C_gm2", "N_gm2", "Size_ac"])]:
    p = os.path.join(GDB, fc)
    if arcpy.Exists(p):
        have = [f.name for f in arcpy.ListFields(p) if f.name in flds]
        if have: arcpy.management.DeleteField(p, have)
        print(fc, "removed:", have)

qc = to_df(os.path.join(GDB, "meta_qc_log"))
qc = qc[qc["dataset"] != "ref_usfs_growth_plots"]
qc.loc[qc["issue"].str.startswith("Plot size"), "issue"] = "Plot size 0.1 ac (404.7 m2, r = 11.35 m), USFS permanent growth-plot standard"
qc = pd.concat([qc, pd.DataFrame([(
    "ref layers",
    "Attributes in growth_plots.shp (C_gm2, N_gm2, Size_ac, Plot_type) and dc_growth_plots.shp (LF_C) are unfinished manual edits "
    "by DC, not USFS data, and match no record in Cifelli_Fernow.xlsx. Removed; USFS plot geometry retained.",
    "resolved 2026-09-11; Cifelli_Fernow.xlsx is the authoritative soil source")], columns=qc.columns)], ignore_index=True)
write_df(qc, "meta_qc_log")

ms = to_df(os.path.join(GDB, "meta_sources"))
ms.loc[ms["dataset"] == "ref_usfs_growth_plots", "notes"] = "USFS growth-plot locations only; draft attributes removed"
write_df(ms, "meta_sources")

<class 'AssertionError'>: Run Cell 1 (config + helpers) in this notebook first

In [8]:
import arcpy, os
aprx = arcpy.mp.ArcGISProject("CURRENT")
proj = os.path.dirname(aprx.filePath)
print("Project folder:", proj)
print("Master gdb exists:", arcpy.Exists(os.path.join(proj, "Fernow_SOM_master.gdb")))
print("Tree CSV present:", os.path.exists(os.path.join(proj, "fernow_trees_WS2_WS4_WS5.csv")))

Project folder: C:\Users\DCifelli2\OneDrive - mercercounty.org\Documents\ArcGIS\Projects\Fernow-FM-SOM-DC-092026
Master gdb exists: False
Tree CSV present: False


In [10]:
import arcpy, os, re, numpy as np, pandas as pd
arcpy.env.overwriteOutput = True
aprx = arcpy.mp.ArcGISProject("CURRENT")
PROJ = os.path.dirname(aprx.filePath)
WORK = aprx.defaultGeodatabase          # has study_plots_ws + terrain rasters from earlier
SHP_ROOT = r"C:\Users\DCifelli2\OneDrive - mercercounty.org\Desktop\FernowShapeFiles"
XLSX = "https://raw.githubusercontent.com/dac0009/Forest-Management-Carbon/main/Cifelli_Fernow.xlsx"
TREES_CSV = os.path.join(PROJ, "fernow_trees_WS2_WS4_WS5.csv")
GDB = os.path.join(PROJ, "Fernow_SOM_master.gdb")
if not arcpy.Exists(GDB): arcpy.management.CreateFileGDB(PROJ, "Fernow_SOM_master.gdb")
SR = arcpy.SpatialReference(26917)
TRT = {"WS2": "Diameter-limit", "WS4": "Reference", "WS5": "Single-tree selection"}
P = os.path.join(GDB, "plots_study")

def find_shp(name):
    hits = [os.path.join(r, name + ".shp") for r, _, f in os.walk(SHP_ROOT) if name + ".shp" in f]
    if len(hits) > 1: print(f"  note: {len(hits)} copies of {name}.shp, using {hits[0]}")
    return hits[0] if hits else None

def to_df(fc, xy=False):
    f = [x.name for x in arcpy.ListFields(fc) if x.type not in ("Geometry", "OID", "Blob", "Raster", "GlobalID")]
    q = f + (["SHAPE@X", "SHAPE@Y"] if xy else [])
    return pd.DataFrame(arcpy.da.SearchCursor(fc, q), columns=q)

def _ftype(s):
    if pd.api.types.is_bool_dtype(s): return "SHORT"
    if pd.api.types.is_integer_dtype(s): return "LONG"
    if pd.api.types.is_float_dtype(s): return "DOUBLE"
    return "TEXT"

def write_df(df, name, xy=None):
    out = os.path.join(GDB, name)
    if arcpy.Exists(out): arcpy.management.Delete(out)
    if xy: arcpy.management.CreateFeatureclass(GDB, name, "POINT", spatial_reference=SR)
    else:  arcpy.management.CreateTable(GDB, name)
    cols, names = list(df.columns), []
    for c in cols:
        fn, ft = arcpy.ValidateFieldName(c, GDB), _ftype(df[c])
        L = max(16, int(df[c].astype(str).str.len().max()) + 8) if ft == "TEXT" else None
        arcpy.management.AddField(out, fn, ft, field_length=L); names.append(fn)
    def clean(v):
        if v is None or (not isinstance(v, str) and pd.isna(v)): return None
        if isinstance(v, (bool, np.bool_)): return int(v)
        return v.item() if isinstance(v, np.generic) else v
    with arcpy.da.InsertCursor(out, names + (["SHAPE@XY"] if xy else [])) as cur:
        for _, r in df.iterrows():
            row = [clean(r[c]) for c in cols]
            if xy: row.append((float(r[xy[0]]), float(r[xy[1]])))
            cur.insertRow(row)
    print(f"{name}: {arcpy.management.GetCount(out)[0]} rows, {len(names)} fields")

print("Master gdb:", GDB, "| tree CSV found:", os.path.exists(TREES_CSV))

Master gdb: C:\Users\DCifelli2\OneDrive - mercercounty.org\Documents\ArcGIS\Projects\Fernow-FM-SOM-DC-092026\Fernow_SOM_master.gdb | tree CSV found: False


In [11]:
import arcpy, os, re, numpy as np, pandas as pd
arcpy.env.overwriteOutput = True
aprx = arcpy.mp.ArcGISProject("CURRENT")
PROJ = os.path.dirname(aprx.filePath)
WORK = aprx.defaultGeodatabase          # has study_plots_ws + terrain rasters from earlier
SHP_ROOT = r"C:\Users\DCifelli2\OneDrive - mercercounty.org\Desktop\FernowShapeFiles"
XLSX = "https://raw.githubusercontent.com/dac0009/Forest-Management-Carbon/main/Cifelli_Fernow.xlsx"
TREES_CSV = os.path.join(PROJ, "fernow_trees_WS2_WS4_WS5.csv")
GDB = os.path.join(PROJ, "Fernow_SOM_master.gdb")
if not arcpy.Exists(GDB): arcpy.management.CreateFileGDB(PROJ, "Fernow_SOM_master.gdb")
SR = arcpy.SpatialReference(26917)
TRT = {"WS2": "Diameter-limit", "WS4": "Reference", "WS5": "Single-tree selection"}
P = os.path.join(GDB, "plots_study")

def find_shp(name):
    hits = [os.path.join(r, name + ".shp") for r, _, f in os.walk(SHP_ROOT) if name + ".shp" in f]
    if len(hits) > 1: print(f"  note: {len(hits)} copies of {name}.shp, using {hits[0]}")
    return hits[0] if hits else None

def to_df(fc, xy=False):
    f = [x.name for x in arcpy.ListFields(fc) if x.type not in ("Geometry", "OID", "Blob", "Raster", "GlobalID")]
    q = f + (["SHAPE@X", "SHAPE@Y"] if xy else [])
    return pd.DataFrame(arcpy.da.SearchCursor(fc, q), columns=q)

def _ftype(s):
    if pd.api.types.is_bool_dtype(s): return "SHORT"
    if pd.api.types.is_integer_dtype(s): return "LONG"
    if pd.api.types.is_float_dtype(s): return "DOUBLE"
    return "TEXT"

def write_df(df, name, xy=None):
    out = os.path.join(GDB, name)
    if arcpy.Exists(out): arcpy.management.Delete(out)
    if xy: arcpy.management.CreateFeatureclass(GDB, name, "POINT", spatial_reference=SR)
    else:  arcpy.management.CreateTable(GDB, name)
    cols, names = list(df.columns), []
    for c in cols:
        fn, ft = arcpy.ValidateFieldName(c, GDB), _ftype(df[c])
        L = max(16, int(df[c].astype(str).str.len().max()) + 8) if ft == "TEXT" else None
        arcpy.management.AddField(out, fn, ft, field_length=L); names.append(fn)
    def clean(v):
        if v is None or (not isinstance(v, str) and pd.isna(v)): return None
        if isinstance(v, (bool, np.bool_)): return int(v)
        return v.item() if isinstance(v, np.generic) else v
    with arcpy.da.InsertCursor(out, names + (["SHAPE@XY"] if xy else [])) as cur:
        for _, r in df.iterrows():
            row = [clean(r[c]) for c in cols]
            if xy: row.append((float(r[xy[0]]), float(r[xy[1]])))
            cur.insertRow(row)
    print(f"{name}: {arcpy.management.GetCount(out)[0]} rows, {len(names)} fields")

print("Master gdb:", GDB, "| tree CSV found:", os.path.exists(TREES_CSV))

Master gdb: C:\Users\DCifelli2\OneDrive - mercercounty.org\Documents\ArcGIS\Projects\Fernow-FM-SOM-DC-092026\Fernow_SOM_master.gdb | tree CSV found: False


In [12]:
refs = {"ref_watersheds": "watersheds_all", "ref_usfs_growth_plots": "growth_plots",
        "ref_dc_growth_plots_all": "dc_growth_plots", "ref_streams": "streams",
        "ref_elt_phase": "ecological_land_type_phase", "ref_soil_series": "soil_series_NRCS2008",
        "ref_soil_map_units": "soil_map_units_NRCS2008", "ref_fernow_boundary": "fernow_boundary",
        "ref_land_type_assoc": "land_type_association"}
drafts = ["C_gm2", "N_gm2", "LF_C", "Plot_type", "Size_ac"]
for out, shp in refs.items():
    p = find_shp(shp)
    if not p:
        print(f"{out}: NOT FOUND ({shp}.shp)"); continue
    o = os.path.join(GDB, out)
    arcpy.conversion.ExportFeatures(p, o)
    drop = [f.name for f in arcpy.ListFields(o) if f.name in drafts]
    if drop: arcpy.management.DeleteField(o, drop)
    print(f"{out}: {arcpy.management.GetCount(o)[0]} features | dropped {drop} | {p}")
for fc in ["ref_usfs_growth_plots", "ref_dc_growth_plots_all"]:
    if arcpy.Exists(os.path.join(GDB, fc)):
        arcpy.management.DefineProjection(os.path.join(GDB, fc), SR)

ref_watersheds: 4 features | dropped [] | C:\Users\DCifelli2\OneDrive - mercercounty.org\Desktop\FernowShapeFiles\FEF_Shapefiles\watersheds_all\watersheds_all.shp
ref_usfs_growth_plots: 43 features | dropped ['Plot_type', 'Size_ac', 'C_gm2', 'N_gm2'] | C:\Users\DCifelli2\OneDrive - mercercounty.org\Desktop\FernowShapeFiles\FEF_Shapefiles\growth_plots\growth_plots.shp
ref_dc_growth_plots_all: 30 features | dropped ['Plot_type', 'Size_ac', 'LF_C'] | C:\Users\DCifelli2\OneDrive - mercercounty.org\Desktop\FernowShapeFiles\growth_plots\dc_growth_plots.shp
ref_streams: 382 features | dropped [] | C:\Users\DCifelli2\OneDrive - mercercounty.org\Desktop\FernowShapeFiles\FEF_Shapefiles\streams\streams.shp
ref_elt_phase: 126 features | dropped [] | C:\Users\DCifelli2\OneDrive - mercercounty.org\Desktop\FernowShapeFiles\FEF_Shapefiles\ecological_land_type_phase\ecological_land_type_phase.shp
ref_soil_series: 11 features | dropped [] | C:\Users\DCifelli2\OneDrive - mercercounty.org\Desktop\FernowSh

In [13]:
s = to_df(os.path.join(WORK, "study_plots_ws"), xy=True)
sp = pd.DataFrame({
    "plot_id": s["plot_id"], "watershed": s["Ws_name"], "treatment": s["Ws_name"].map(TRT),
    "usfs_subcomp": s["Scomp_name"].str.strip(), "usfs_plot_num": s["Plot_num"].astype(int),
    "xy_source": s["xy_source"], "usfs_offset_m": s["usfs_offset_m"].round(3),
    "plot_area_ac": 0.1, "plot_radius_m": 11.35,
    "x_utm17n": s["SHAPE@X"].round(2), "y_utm17n": s["SHAPE@Y"].round(2)}).sort_values("plot_id")
write_df(sp, "plots_study", xy=("x_utm17n", "y_utm17n"))
arcpy.analysis.PairwiseBuffer(P, os.path.join(GDB, "plots_footprint"), "11.35 Meters")

def poly_attrs(poly, keep, prefix):
    arcpy.analysis.SpatialJoin(P, os.path.join(GDB, poly), r"memory\sj", "JOIN_ONE_TO_ONE", "KEEP_ALL", match_option="INTERSECT")
    d = to_df(r"memory\sj"); have = [k for k in keep if k in d.columns]
    return d[["plot_id"] + have].rename(columns={k: prefix + k for k in have}).set_index("plot_id")

def near_dist(fc, where=None):
    lyr = arcpy.management.MakeFeatureLayer(os.path.join(GDB, fc), "near_lyr", where)
    arcpy.management.CopyFeatures(P, r"memory\nr")
    arcpy.analysis.Near(r"memory\nr", lyr)
    arcpy.management.Delete("near_lyr")
    return to_df(r"memory\nr").set_index("plot_id")["NEAR_DIST"].round(1)

parts = [poly_attrs("ref_watersheds", ["Ws_name"], "ws_poly_"),
         poly_attrs("ref_elt_phase", ["ELT", "ELT_NAME", "ELTP", "SOIL", "SLOPE", "LNDFM", "ASPECT", "PLASS1", "PLASS2", "LTACODE"], "elt_"),
         poly_attrs("ref_soil_series", ["SoilSeries"], "nrcs_")]
if arcpy.Exists(os.path.join(GDB, "ref_soil_map_units")):
    parts.append(poly_attrs("ref_soil_map_units", ["MUSYM", "MUKEY", "descriptio"], "nrcs_mu_"))
site = pd.concat(parts, axis=1)
site["stream_dist_any_m"] = near_dist("ref_streams")
site["stream_dist_perennial_m"] = near_dist("ref_streams", "Perennial = 'Yes'")
write_df(site.reset_index(), "plot_site")

terr = pd.read_csv(os.path.join(PROJ, "study_plots_terrain.csv"))
terr = terr.rename(columns={terr.columns[0]: "plot_id"}).drop(columns=["Ws_name", "xy_source"], errors="ignore")
write_df(terr, "plot_terrain")
for r in ["dem5", "slope5", "north5", "east5", "tpi50", "tpi200", "twi5"]:
    arcpy.management.CopyRaster(os.path.join(WORK, r), os.path.join(GDB, "terrain_" + r))
print("terrain rasters copied")
site

plots_study: 18 rows, 11 fields
plot_site: 18 rows, 18 fields
plot_terrain: 18 rows, 15 fields
terrain rasters copied


,ws_poly_Ws_name,elt_ELT,elt_ELT_NAME,elt_ELTP,elt_SOIL,elt_SLOPE,elt_LNDFM,elt_ASPECT,elt_PLASS1,elt_PLASS2,elt_LTACODE,nrcs_SoilSeries,nrcs_mu_MUSYM,nrcs_mu_MUKEY,nrcs_mu_descriptio,stream_dist_any_m,stream_dist_perennial_m
plot_id,,,,,,,,,,,,,,,,,
WS2A-1,WS2,Ba10-23,Allegheny Front sugar maple/red oak,210050.0,Ca,CD,42.0,SW,239.0,888,Ba10,Calvin,CaD,513981,"CALVIN CHANNERY SILT LOAM, 15 TO 25 PERCENT SL...",31.1,572.5
WS2A-3,WS2,Ba10-23,Allegheny Front sugar maple/red oak,210050.0,Ca,CD,42.0,SW,239.0,888,Ba10,Calvin,CaF,513985,"CALVIN CHANNERY SILT LOAM, 35 TO 65 PERCENT SL...",33.5,354.2
WS2A-5,WS2,Ba10-23,Allegheny Front sugar maple/red oak,210050.0,Ca,CD,42.0,SW,239.0,888,Ba10,Calvin,CaF,513985,"CALVIN CHANNERY SILT LOAM, 35 TO 65 PERCENT SL...",46.9,186.5
WS2B-1,WS2,Ba10-23,Allegheny Front sugar maple/red oak,210050.0,Ca,CD,42.0,SW,239.0,888,Ba10,Calvin,CaF,513985,"CALVIN CHANNERY SILT LOAM, 35 TO 65 PERCENT SL...",35.7,163.4
WS2B-2,WS2,Ba10-23,Allegheny Front sugar maple/red oak,210050.0,Ca,CD,42.0,SW,239.0,888,Ba10,Calvin,CaF,513985,"CALVIN CHANNERY SILT LOAM, 35 TO 65 PERCENT SL...",26.1,259.0
WS2B-4,WS2,Ba10-23,Allegheny Front sugar maple/red oak,210050.0,Ca,CD,42.0,SW,239.0,888,Ba10,Calvin,CaF,513985,"CALVIN CHANNERY SILT LOAM, 35 TO 65 PERCENT SL...",34.6,475.7
WS4A-10,WS4,Ba10-22,Allegheny Front sugar maple/basswood,210051.0,Ca,CE,42.0,NE,220.0,238,Ba10,Dekalb,DaC,513997,"DEKALB CHANNERY LOAM, 8 TO 15 PERCENT SLOPES",169.8,662.2
WS4A-11,WS4,Ba10-22,Allegheny Front sugar maple/basswood,210051.0,Ca,CE,42.0,NE,220.0,238,Ba10,Calvin,CaC,513979,"CALVIN CHANNERY SILT LOAM, 8 TO 15 PERCENT SLOPES",28.1,532.0
WS4A-13,WS4,Ba10-23,Allegheny Front sugar maple/red oak,210050.0,Ca,CD,42.0,SW,239.0,888,Ba10,Calvin,CaE,513983,"CALVIN CHANNERY SILT LOAM, 25 TO 35 PERCENT SL...",11.3,253.2


In [15]:
trees = pd.read_csv(TREES_CSV)
trees["tag_number"] = trees["tag_number"].astype("Int64")
write_df(trees, "trees")
spl = (trees.groupby(["genus", "species"]).agg(common_name=("common_name", "first"), jenkins_group=("jenkins_group", "first"),
       myc_type=("myc_type", "first"), n_stems=("tree_uid", "size")).reset_index())
write_df(spl, "species_lookup")
print("tree plot_ids not in plots_study:", set(trees.plot_id) - set(sp.plot_id) or "none")

u = pd.read_excel(XLSX, sheet_name="Used_Data")
u = u[u["watershed"].isin(["WS2", "WS4", "WS5"])].copy()
dmap = {"OH": ("OH", np.nan, np.nan), "0 to 20": ("0-20", 0.0, 20.0), "20 to 40": ("20-40", 20.0, 40.0)}
frac_cols = list(u.columns[u.columns.get_loc("frac_soil_wt_g"):])

b = u.drop(columns=frac_cols).rename(columns={"ID": "sample_id", "plot": "soil_plot", "subcomp": "soil_subcomp"})
b.insert(4, "subplot_letter", b["subplot"].str[-1])
b.insert(6, "depth_code", b["depth"].map(lambda d: dmap[d][0]))
b.insert(7, "depth_top_cm", b["depth"].map(lambda d: dmap[d][1]))
b.insert(8, "depth_bottom_cm", b["depth"].map(lambda d: dmap[d][2]))
b["treatment"] = b["watershed"].map(TRT)
b["bd_note"] = np.where(b["depth"] == "OH", "organic horizon, 20x20 cm frame; no BD",
                        "BD from one core per plot x depth, applied to all subplots")
write_df(b, "soil_samples")

f = u.loc[u[frac_cols].notna().any(axis=1), ["ID", "watershed", "subcomp", "plot", "depth"] + frac_cols]
f = f.rename(columns={"ID": "source_sample_id", "subcomp": "soil_subcomp", "plot": "soil_plot"})
f.insert(5, "depth_code", f["depth"].map(lambda d: dmap[d][0]))
f["treatment"] = f["watershed"].map(TRT)
f["composite_note"] = "5 g from each subplot (A-C) pooled and density-fractionated; stored on subplot A row in source"
write_df(f, "soil_fractions")

<class 'FileNotFoundError'>: [Errno 2] No such file or directory: 'C:\\Users\\DCifelli2\\OneDrive - mercercounty.org\\Documents\\ArcGIS\\Projects\\Fernow-FM-SOM-DC-092026\\fernow_trees_WS2_WS4_WS5.csv'

In [16]:
from arcpy.sa import ZonalStatisticsAsTable
arcpy.CheckOutExtension("Spatial")

write_df(pd.DataFrame({
    "watershed": ["WS2", "WS4", "WS5"], "treatment": [TRT[w] for w in ["WS2", "WS4", "WS5"]],
    "harvest_notes": ["Diameter-limit; limit reported as 43 cm (17 in) DBH (Schuler 2004) - VERIFY; last harvest ~4 yr before 2022 - VERIFY",
                      "Unmanaged reference since early-1900s logging (Guillen et al. 2021)",
                      "Single-tree selection; last harvest ~10 yr before 2022 - VERIFY"],
    "soil_sampling": "July 2022; 6 plots x 3 subplots x 3 depths",
    "tree_inventory": "0.1-ac USFS permanent growth plots; DBH in inches; stems >= 1.0 in"}), "watersheds_info")

rows = []
for r in ["dem5", "slope5", "north5", "tpi200", "twi5"]:
    ZonalStatisticsAsTable(os.path.join(GDB, "ref_watersheds"), "Ws_name", os.path.join(GDB, "terrain_" + r), r"memory\zw", "DATA", "ALL")
    z = to_df(r"memory\zw")[["Ws_name", "MIN", "MEDIAN", "MEAN", "MAX", "STD"]]; z.insert(1, "variable", r); rows.append(z)
write_df(pd.concat(rows), "watershed_terrain")

arcpy.analysis.SpatialJoin(os.path.join(GDB, "ref_usfs_growth_plots"), os.path.join(GDB, "ref_watersheds"), r"memory\uw",
                           "JOIN_ONE_TO_ONE", "KEEP_COMMON", match_option="INTERSECT")
blk = "import re\ndef lbl(s, n):\n    return re.sub(r'[^A-Z0-9]', '', str(s).upper()) + '-' + str(int(n))"
arcpy.management.CalculateField(r"memory\uw", "usfs_id", "lbl(!Scomp_name!, !Plot_num!)", "PYTHON3", blk, field_type="TEXT")
arcpy.analysis.Near(r"memory\uw", P)
arcpy.analysis.PairwiseBuffer(r"memory\uw", r"memory\ufp", "11.35 Meters")
ut = to_df(r"memory\uw").set_index("usfs_id")
ut = ut.assign(is_study_plot=ut["NEAR_DIST"] < 1)[["Ws_name", "is_study_plot"]]
for r in ["dem5", "slope5", "north5", "east5", "tpi50", "tpi200", "twi5"]:
    ZonalStatisticsAsTable(r"memory\ufp", "usfs_id", os.path.join(GDB, "terrain_" + r), r"memory\zu", "DATA", "MEAN")
    ut = ut.join(to_df(r"memory\zu").set_index("usfs_id")["MEAN"].rename(r + "_fp"))
write_df(ut.reset_index(), "usfs_plot_terrain")

write_df(pd.DataFrame([
 ("plots_study", "USFS growth-plot locations (15) + DC field GPS (WS4B-1, WS4C-1, WS5-6)", "shapefile attributes were draft edits; not used"),
 ("trees", "DC tree inventory -> fernow_trees_WS2_WS4_WS5.csv", "0.1-ac plots; DBH in inches; Jenkins et al. 2003 biomass"),
 ("soil_samples / soil_fractions", "Cifelli_Fernow.xlsx, sheet Used_Data (authoritative)", "July 2022; CASIF elemental analysis"),
 ("plot_terrain / terrain_*", "Tucker County 2021 1-m lidar DEM mosaic", "aggregated to 5 m; footprint r = 11.35 m"),
 ("ref_elt_phase / ref_land_type_assoc", "FEF shapefiles (USFS)", "ecological land type phases"),
 ("ref_soil_series / ref_soil_map_units", "NRCS 2008 soil survey (FEF shapefiles)", "map units carry MUKEY"),
 ("ref_streams / ref_watersheds / ref_fernow_boundary", "FEF shapefiles", "")], columns=["dataset", "source", "notes"]), "meta_sources")

write_df(pd.DataFrame([
 ("trees", "'WS4-A-11' = 9 untagged stems of plot WS4A-11 entered under a variant label; merged", "confirmed by DC 2026-09-11"),
 ("trees", "DBH in inches; legacy agb column treated DBH as cm (kept as agb_kg_legacy_cm_assumed, do not use)", "confirmed by DC 2026-09-11"),
 ("trees", "Plot size 0.1 ac (404.7 m2, r = 11.35 m), USFS permanent growth-plot standard", "confirmed by DC 2026-09-11"),
 ("trees", "9 stems lack DBH (kept, flagged); mycorrhizal type assigned at genus level", ""),
 ("plots_study", "WS5-6 Scomp_name stored as ' WS5' (leading space) in dc_growth_plots.shp; normalized", "fix in source shapefile"),
 ("plots_study", "WS4B-1, WS4C-1, WS5-6 have no USFS point; field GPS used", "offsets 108-201 m"),
 ("ref layers", "Shapefile attributes (C_gm2, N_gm2, LF_C, Size_ac, Plot_type) were unfinished manual edits by DC; removed", "resolved 2026-09-11"),
 ("soil_samples", "Subplots A-C lie 5-15 ft from plot center, inside the 0.1-ac footprint; no subplot coordinates", "footprint values represent soil locations"),
 ("soil_samples", "Soil plots (7-24) not linked to plot_id (DC decision)", "soil analyzed at watershed/treatment level"),
 ("soil_samples", "Bulk density from one core per plot x depth", "applied to all subplots"),
 ("soil_fractions", "Fractions are plot composites stored on subplot A rows in source", ""),
 ("excluded", "DC_Fernow_Data.xlsx = different design (0-100 cm pit, no plot IDs)", "not study data"),
 ("excluded", "Fernow_pH2023.csv = WS3/WS7 acidification study", "not study watersheds")],
 columns=["dataset", "issue", "status"]), "meta_qc_log")

def rel(origin, dest, okey, dkey, card="ONE_TO_MANY"):
    out = os.path.join(GDB, f"rel_{os.path.basename(origin)}_{dest}")
    if arcpy.Exists(out): arcpy.management.Delete(out)
    arcpy.management.CreateRelationshipClass(origin, os.path.join(GDB, dest), out, "SIMPLE", dest,
                                             os.path.basename(origin), "NONE", card, "NONE", okey, dkey)
rel(P, "trees", "plot_id", "plot_id")
rel(P, "plot_site", "plot_id", "plot_id", "ONE_TO_ONE")
rel(P, "plot_terrain", "plot_id", "plot_id", "ONE_TO_ONE")
W = os.path.join(GDB, "ref_watersheds")
for d in ["soil_samples", "soil_fractions"]: rel(W, d, "Ws_name", "watershed")

arcpy.env.workspace = GDB
inv = [(n, arcpy.Describe(n).dataType, int(arcpy.management.GetCount(n)[0])) for n in arcpy.ListFeatureClasses() + arcpy.ListTables()]
inv += [(n, "Raster", None) for n in arcpy.ListRasters()]
pd.DataFrame(inv, columns=["name", "type", "rows"]).sort_values("name")

watersheds_info: 3 rows, 5 fields
watershed_terrain: 20 rows, 7 fields
usfs_plot_terrain: 43 rows, 10 fields
meta_sources: 7 rows, 3 fields
meta_qc_log: 13 rows, 3 fields


<class 'arcgisscripting.ExecuteError'>: Failed to execute. Parameters are not valid.
ERROR 000732: Destination Table: Dataset C:\Users\DCifelli2\OneDrive - mercercounty.org\Documents\ArcGIS\Projects\Fernow-FM-SOM-DC-092026\Fernow_SOM_master.gdb\trees does not exist or is not supported
Failed to execute (CreateRelationshipClass).


In [17]:
import lzma, base64, hashlib, io
B64 = """
/Td6WFoAAATm1rRGAgAhARwAAAAQz1jM4NbbHO1dADuYSu7uxsJOxJ1UctImbURHZc4uk8v9C46hT1RoaG/82Zg4lJqhaft9kAg3
h6ekje8rQtCmRQpb+ZOMdEw9v2kVx8mlPVI2cpN9Wr2yohQrRD9cxlUI96DxW5FfLZqn38WreNsff2FEJfL5OsudTzn7pv5XuY3M
3BehIc9UA060SVmBaoMDwrCMjANb2pZepRUqehnae+OkJ4r9XAep5axCUkp+WtWOXPWRgq8YZIZHq1yoKMRlIdWp7abVX6xoWRxS
Cqbo77ujKoITLSSjo8F1+Td+Gbaa4scUFKwtZZfXEzf50N/RJTOxS+QWGSEUG6JpPq+2IcRiy253XCBUcgDPA7igNoErBcEZE4py
5alfTPegj4/tJZdvapoY1OOAYWS2EuvGeESGRxwINl/w7kivgrDe9qQvmQQshXq8CIXzl11qyjWEtoILSrvJzu/BjQeKEm3IjC1j
+lP/XK3VOVx83KmszIBIIKjfF47kSZ4w3pmLIFn6mZOlD6ImTrY6QNDEGaQguvDdsli5pPs41aSFXxlEmWI0vRIMwCU17FKGl539
s3mNo+UjsCNR/RH9G5FXnOl0e63tJZ0xcFgOHZkMc21KidQtRy+0NPg0V5QcLnxBgBDUL/ahB9CnrFasNA7lQvhInYRsThsgXL97
CBEviOzhNdIcCNNJFDi/LSnAmrXE/gi3ZEzRGYF3Yd2E2XANuruiMVpQ+S6JNQ4qYLSc4VHJ0kxWOxECSzf6m6Cr6e24S0Byxpsg
ZdWSgmb+lVOfbzCfVx5Yzn88C3xFOHalubYjz4eD4zgEZs+7TzQ0ZxO1IQxzUCya64rGTZvkLncSjQa6N2ywF1UJOVXnBO7Mafi5
VgErk3SimIdhTxMKomhgQPNY7h5QfVdIfTFZpzHxfwHDGy4T+8TiXeAaOlElZQxpl0CLY9o2TRiyOwUTMbGM45X0dXoyq2iM0/Oh
yI5+aEIlPgwVjljCleAh9uqkKtvwCzErTUAtieSfINXY5KJiC+3MEBFCxX/jPuhAHT/t+eXUWAwe87ykEB4Tgds3HB/ekWV6EQ7K
k4xJE4pOyavty2Uj3OE7xN+GSWb3UkE5HDG6V3kD4A1Zop3HfTo+IGmoWl4JWmIzoL/8ZfcgL2Kw7CfTvn+cX3duYIXuxk48PRgD
aifes96IqL5I6NO614dzQK21hiAA1Ngd1yw2V8lrWWiCqYp1KuKpBXuOQvOStmRhylt0jl6Hpe+FdnBJCf1aZ+XVVGs7IkUlckui
QHFSKYVkZHCZrFjMn2gktAdRQNAjZ12p0cs/WVTy/sneC9KRti8IhXpZnkiavHPN+SXWxitee29JgMk4M+JWPsCYIZkg16rMlkmk
TKouN/1fOZTOw3aKrgHQt4YpgfZ/Gk2HQowX1Ai6hQpiHRdSzUO7ycgiCJW/fpPY/4Zlt2kFqMjv82IKgFxAz8udw9G9kiNn9jpi
zfSQgGG2BvwCDk8QxSVPTEdH1UadZMIGqcvJVLo1IUWvFi1r5k18A8FCxAyfH1uuZqnRMXNv8AhdE+76EgXGvhfhjtd2tPt+vs+P
AZ2380eaFuVuGf+Lm4zdbiQpIRXc0XK/koHHmFUWcqrIanVjRF8lggW+U7vpCVMgYNa+OI0RYMntd2Gwd/qfBUoHi0+pZ72ujxB3
ncFzDKyG0HF+WF4WJbkWwLSjwp053FCfWnKebe3vSvHJST9+yLz1cgdWU+o9cH/uSgwpisLJYQH6uzY4K5EvLcsIjoyQmOR4sr1k
uXkbA7rrGEXvYRkkAgcrb6Inp7w7NEEMtxWtQBCb52ACrwqNhBO29t5fF2V2/k1TYOwnHMhrLpwc0B6RlSLFJkrSzkb3kAW/T+pV
G3ahkYvpYGTrETJ/ZnBnP+lG+RnS9dxq0PlWSMMt8T/xt0sOoaqSPUHRnq/RJGefFqdBlrBZiUmOHC+Ice6E70Dic51jyz00//Mv
DXlj1ZL8j0u1CWEVWENzoF1qVK0Dk8tfEH+TkL8T/UXhBsvx8MfOmXsw4FntWnsvIhgnxlMpsrvS9LYD8LBAMWvlG28n0NWo7AOo
+rbv0kWucpbN7ZwqX5++MOTKIZ7wdtR6thqJX0CgygVnL7MFHVinZiwN6FrTGifxRFCdIDTy55Wy1dSk+1tIc9lPljJMJ6D5b2mW
zOSsfo0kaqai8Vlup5LICXPIbc9bz+sWbSS5K9+ftxookMbfEZhjIo6c3AcyJ9rPlkEMmfRFK4rxl4CKxlnzMpVQuiY8MY6exE9Y
01j8gkCJjCQVPhKj2oMARe5Fb+9OyDEvmUNgtXaujwfgPzRnVR5fHZLqfCAK5P6VHr5uveFmkHyPIlLIdsn4kuQmAw5DFZpbY+zI
zJRt/HcHnlx1Lc570A7oiq7Ax8EF9tzL7CrLG347TCz1HQNyLlKm08Yvyjhrfw+hlUzkoBvbmYs8FCWWV1WuZOCt8pi1c6V722gt
Ip/CKC4Gg4i2d4wLaCkiMFyMoOqnq18QxApcOqLdpqIMzpHAHad3hse9+QUmhbItjrQoQAOXkbINOP4xwi+QmIiNxYdB80URe4Iq
juDvMHPz18VEzQk5eTW97DVkJmyc/g3s3Wr2k96MotjDYfQdaZYP9EPvAypiyevrubFwiVB+yKjU/ii38/3yOR3fpgcxaezVE8Fu
mjxtqsfrU1zRtfNltLg6kSbkg5qWMob78yfjSx9I43p4gS+1Q7OYWO5r1byhGtKdxJ+2AziKOoH5uFHjK/i2CjSHGCFl6MnfIdIo
ZGBCB9R9Is6wpnd6ZNZA6HEA0fwTNZQEq5w3J03QnrVnF1/k2V15NU1G9B7sBrmm9tYmYmm+rix9lO5YJ5M75bWlSkIID664uzgX
9eBvArwhW74/nHZrohGoYZ77SpzSWGIkl9HiRznRdT/JSJeEa7I+sPC96rT1nyLE0gsYVwe7+ks4VdFgeQptIipXDxwmdDqxOlnC
GRMBv3cr2uq08ndO6MUiC8tUSqPR8gTlFZx4QgggtnyxAf2oRR7rWR/IwhagAwFcGnilCY9RF/OFCDPLWy04TLgSYxQJ201nsxeY
+RJXWGyPIPchCpZ4fXCW7rVTobh02VCmJUSfPczbeTjM3squt+8CpcOPwbbw3NmrQwaWp0ya39HOaxV3OYvmCecL7qwMpWi4l82b
+Yhz1H3cOHd0Mlr+rfbtfvf4uYnQqonjijmNzPW0wD/G0CLfmqF6tAbumD7fNRVXvJT0nbxogLliHtIrwa5pid7AhMzxfGYZ6HgM
ud4IglxvTPinPDj4TV/PCuq0au+BzsTyxg1T3m40cMdFgbzyFfhtsR+2YRd7M18kc8a9NvgHbGevcfsZv6T4WrmdysFjHc7zuYlN
QBQ8p0ORgB/aN4czuqNV4E26OaNOtr6x0ca78LQEwLIcPNDBXLigTza8NvaIZ87EF9jXmWTp6q73C4k6U1hmIkaDx8T8WZcCe03x
GXq2z6PNWif/Pex/4eHXO3WhplnGgBu2koUN8/ROV+eduOT3KfuYU/LV26mUl1J1vyiKtU0ZQGb4DW1AhoBGOsl2mwQZLXoWyOoA
lvb04OgMzoINQJQeyAeCr0JaWLEeFuYFVIzxJCfiMF1/CxXaopyy71mAWdETox/gBTQpUJNjlHwpZLD2yAZJABPMsFjrXgvaY2N6
HoX30KW+wPAHacMG9CmsJDabBv6sEYtG8wwkp8Oi+F8bf0rmBe7qWqBosr9oQyKANOtjyuxot6cy+W6oo5um/ig0h1wiQGPd+//R
f5PLb6rud16cVRRNUDFT6X79XXFQHKNWJOk1Xa6GGw1OpXcHatQL6ZSnFhxMMtQ58Q4V+baRX54dbmEu/iXAkjWhWUmYkbSZy0fV
6R5F7ibygyEVVkQe3mbnAv16gBT1x0Vt8C0sDjpa77bhQjjr3Rd6UxbIhcILDjJu3lu8YaCdjS2CR473GvOqcSP/cX21dORKLqzO
3CJVupY8idxVjL1JoxmVl0jNN/aOQrfYJ9V9CZ5rr9PvdhnPrcvoGX3HaSfV37hMkfonxxbNrj//+D7H/j6EJJjNQF4gzsTC2tDN
4eV/IJ8HOnXin9rlMrvCtD10AJv4DF+9qRRy6VO7Vg6l1xP3ewuhcrGddyyoJSkk2I//KMgXfdmU5jvdMsTtNKkKAYar71eKaXOA
/tpi3t2pv9akCrqAhGImVyk6lOmE79rEPwoU2j1k2ukbEWcWfyRCXnXWK2TcNsQvLEtxwN0GwlwJj5SA4UIu8ZD/puAuNW4u3Fqj
jKCKsXCvPeqiMWBzb/FBSnMsT3bk2sfD6uvqkzcn6zoPTo38eNOVtIvjjBx99lzRZRT16OMM4l03nXLEYQnN5r70Swas3vaqppgA
hAx4CP+UoAP/grxlPDfmup3YpknqefLvfN+EO5Wm+bQ8il/Lk6zABfFvF79sGjHUf5C4781oWQa7kEhgbyRnbLDMIHFZqcysI9qE
LV/43Cf1eqgEu29oSfUOEIKcmXprYg1L+1uasSGpbJ73Z+niGYvxsfrq+AA60e98fz830VEJzl5eXxUTALdgy2DsEWaFMsnQXkA7
GpwKxQdmpj4q+li0o0LQ82UFzd8kGTpb3L92B35KfVaRBWmjcgzLIggqGlijtqjCSyirP/EYZwoHeCMMcrkuxlcc5P4J+smIav/m
C+fGhhYU9N3OkDWxKI3OqR9rOE9eiowoBTkSOpuSMpupcdeYIN4aI52UUZ4Y9RIeQhk6i5IZvynot1Zh4RIlEFTHWta+gCvpDfdp
nlvtvqH5DQ4gvPH6aQI4WtXxzJDnr8Q+o6d743gpZMm7dOpSV99XmytY79NlXjeuzGhcjYupoi42xO9vC/PgNwLXZtRzvKNRsG+5
BZuz2IVuYuGnFP8+z1oiYkOPEicls3LetV+uwqIYm9GV090K10u0vn+AEzEiOHcqRzTD8dNwzx3Bj8kcQP9QKkOWUZJ5Uf6HidqM
a60VvPWcwzbzlAHlJx6cfGhap4gPqoEOLMlyZkky9ZLN/rBSp51egvX68UMJCOsaxEhA7pf3VvZnEXr6NtpCjTSwdp0/mGaUo/cq
4jhHnu9rhGZ/tF1S//0BhZ4D2tuzabqVYSoMP16Wv/YH+cmdzAl7YgrCTkxJjywD9c6lqg7PyokTA+wkzgdLPfWECVUs0Gzxo1Eq
3mIluL3YzTWWfPIaGYaG+1KGy7L3RiBKDUqigf+N0CUgVUyyOXceWi2qLAaQYo3sLGWR9xD+LVDkYyxSqz5eswJycv955/cbNdYu
YvxXG+Ulw6HhwaMpXpVTPMQSm9rqb0zwswCJEFSJmwe4UUI6gned/LZ1XYPO+AiJRueEIlOUrAc2KyOtuoXENhzAkoyD74Pt+yk6
q0QxL63RQ6b1vmynTkGRUQFw0q3G6x/OtVUuU9RAyLyT7FwKFxZBlZ7MIGDRAwcc3V2VPBuqwhdHoc3CEoI7s6Q8vheacae9tdt7
9qhrV2H3rbgNsk/dqbm1xBUbSs8kooObyIuQ5FP2dXU9PrhcHKmkQehLmtb3GiyACRnzd56Ah/FES6u5guHBvmWBrrLIUtrVi88I
lMPU/wUd98Og0yIjiN2zCNK3CQR/QURZcaFoGycVsVJ0mlfHHyUX0IYmFDpV/WEiA6GleNYv0edMl1sd8gk6Ag1MzXw/wdvZvp0b
CjtO+UTkjG2/B517DpV3cdViYBzJxKAY0F37HLCwjUDjuEie3zDiMA8E/D1lqR73hwSgPgerlhEdbryZ33eRJLYg/CXmRwKgW/6p
jwVgP3+Z/XepSrTw+HVqCDVbzZoZb2E0NzfXWh9YWRUQTZXUnJEW72mBdic0gcCL0y8YOTItR9rY8y89ra8fz1pMihgWI12McI7a
IT1mV/9JSfqsP0ZQgsIPPSc6WTsFD/PpD+9ikmz87zzJ8jCFB2WOAHuaUV710q/jgFsy9a1paGHpC4RpUuOlH5gMET9SU+HzcVus
SDP10x80J1HMgzwxAmtJKrKO1Rh22iKxP+gVAyOaUrOmcwEA2KvKELP5eyKvLM5XxGyw6KKXgEmcMzQ+09n1xg+JwCKpNPzAAWMV
uu5f4snXOwodDpeJWLwmvpbSHImDsA7OGMEz/RP/juiNrbuNasKhZQowd+VPCMhRpVlA0aDxanORmO+L4rtM3NycBr0p2fvObJ66
79Swx/GpCKo1bLq3xKwOQak13Tbk0QG9Ov6WJ7ZMeEWmdno6LjHG4P56DmGFQxTk77SuvInCHS/F/sMvqtRZFIwLXRnKL8JbLtl7
Z2pZokmvcLXJe53Zg4DH8hcC1MOUFhz8KX5WS1YH816C1YJLGEUmUUAMGXYVims9b16e6949OFU3v5YTB663mXrWpD5amhYYbLeB
Y/UkzPqZFbW1aAsbyxyN0Iejoq9fOSr3B75LSCBYMEToINZQayWM7l1/hvuwpK9PBFGZNWR14rPf/RuYSefmgsJ30Fa+xsO7SGEk
/WfOIXwRouA4BcU0EQJRsHN9o3ZZmksFuhkOMLpgzFfT7IHN0xX2QYjLXnXoO+TzlibseHWJacahU0meUv2CIZ8P7tD9vgsqcfEH
Jd2grElkKDxJiZJKEnJffLpJPhFs7983BTlR2Rdlk6GQDs1g23nOjEHEicVeNNE7vHz4rARh39x6gWIK9HWAzFVjdS9Ekm2zycAp
/iswCnLeRPD0BJPjex4wyaEwUGGPywADv7IcvURXJH0VglRydkLM/7GvMYYHA5srQpf15+/Ysx8n9WlScBHmjh12Jsxd7KI6RssN
99k+IAinFXJ9qjwfXcnhWw8O9XN9jLhxovDh/YIoZbNAkh2FnnRoJTJfRi2JjG+jXf7oopiKgUmhArNLn3y40XAa+k/O7MJxbhz9
wlzOPNYyF55ziACGNsOl7MJkk52euNF+T5QhvXnmsZrYRw47Uixmhmlju3nq1/L67J0P8LZAbTXsCwbzEXPpBpXOMBlSTtyCUdwC
BaVFPcTQ6/fR2Clk6ELIa1TN1odA1mwYM72WY4ZlXQ8dVLy/bLw9dK5MpYueLIle+BUJpurl5cuew2uFEAv/csSFpRDhpa/2vrQ4
jWXfWdwNKwaKhLTBImSt36i/iEXRA5ASB5GqzeIM65Cjjd4hODhjK44ZqKTnttgZGTqnlxmFnmVrQB/zF/mIlFj1w80IjOkqM0B+
MPblnQa0FoyqItGk5VBoHZvyjfkRq+T28vnXQaN1X3TDz9khtXMvVifCFPPXF78SzLB4Ngm8hFQAuDhBlIBJIaxFkpXy+Zj2XSQH
DO/LJo7DkWjIiS0MooPhnUQEvhAh4/4UN9V2bOLm9Ax4g1bUtFcNrC8Me7hv68wP5HeEmXHwLh2xAk5POj18SUbUj9BiG5qOuRTa
iqVoz8izDwO6WTTS57+7dBW1A6hkhHNQNBghypGxzzf5JnKHFc+07RTHe33ME93wzO2rDGKy9GBfvWRYvAwYgwVo5R1SzSUsqHfw
+zAX5eASoQ6RV8Ta8moyvrbJPYNb0Vd4gKAZKpEvBIDw47mfihgOGkrIzYuG2M3aBAP25awyXhhEPooaInTJVaAV6UvGbyplNNCi
ZCW8CHztIXhu7NLZkDNC2mR1cj0f+t8kBkaoIzCAHAFmhAcwKG9n5CM3zM1ebEuCHMU+sXHEdreNAElIFM5hlpaIeCjr06vfq9l3
lDceDHMD7vFbNUE9qdAS3gwjn+99mZHYfA1LWRcW0831XafI7hTsL1THEpMhs6ggKGT09DtpgM/I4GA6+W2ttP2u1TPZ7oX9Yjk/
Ig0t6t9ek7k1Yc5HivBgV2VgofbiKhEttp+unwbHMuALTduG+a0t0u2865fyi+inLg07wbrzGJgJlFSFdpTSQ+LRVYCOmtc0xrno
MoQuWKihyybxNYRtrdaykMMeCxOH1erdXS3o/+tOv32o3gspkuLTITeUVnft8FgI5k4vDJNd0Wd7VgaiAygkViLxr6zWuOaWdPT4
1G/QaX/qoAU7t4KO2JGyhgveOFDwvasyZi/i8Ljrffvfy7vaRWRR+3Khk3w9fCFvOTGc1LH7i/KB2pOOz6kK0anxVw1yoSrwLzkO
qaEW3QPa8Zy115+54AcY49GJxv+Sz8KhYsFBW9u0pAoZ3ADVGY928les4yTuQPL38Yj151h1kCfSQ2OBgNR/BjYGBxAC7OHXNY+K
rEFGT14WXOzo2NrdMr6TEmPhCPnTbJaJ7WlwhKDAfNiZMqmMpUMpAq1hBE4SfqGhOpWqjbjQesWUnydNVUgvMZvENix+qfsayFxA
2HEmq/0iz0k8mRNqM2PEr1XFmI8+NqdKFFQed/4u6TuKu8IjcV5wbImq2lF/be5qRAF4ANzJ6vhaEYJ4BqeizOGhd6QKLf7mzGRi
9LqRVUsAdYM9rfWrk0kDd0kX+9r8SmZc7z0FpOUp109nUlofZxuakixMw7uR1BRBeFRrFZqtMtcBKmH5pD/5sQmbATehArmfUQll
VxK4f936BSx60yqpGIOvQ+voIk/h/gnBZaozsYFiP8V9edRObAzJWVhiliYptrn5OASPEYGre2LZ44I/wpUwpnEwOzRaMSLpyKtw
XkXZFl85UZzDaV3KPaJJlvQRuNzrYeBecpieSkUcIztjn+6hjO+sqZHxD0if02tPA3P0WMEoocyktosd1i/aWgP5mnddWIlxkg2i
LE4AAbgClCyDlTuksLKwYFNLnRbP1ocqRZbJyW/RmAzZX9OyKCLduh2JfRL6uLOL7N5iyLJrQUDzGjBvyaohXggi1kIR1AkFHOBg
3bjpZ+XTSGaMWK5TUpqcTds4h5oJvhx43B09gW3CymXurGUS7Iv6lpFXLj4YMfBVs59QpczHbYrgs2im/TK0pqwZgihLdJelCMNq
T0ZWwJPeBlTADk+cmlQUganoG64CtsDILr0iiSa60DeT8H/YgiLSyhDyBkaJ1fPaxQnVrCciLWePSwRWkta6Xy97C8yffJdev/Ep
ojlnKWiIAGjIi+ruQh61QIduN/tms4FvAmwRKZNpikZgLx/y24zYMb5B9AD4lI0XdKosZ4DkYiii6Vrk0a2yw4Fgv10f3BMkMsIu
UkaB66BE4FP2NX6jbBnQwx48nUpq7Lix3rpdovowMNIsN3+irFiF58UfpSK+4gMM2HGia8oiM7Hv4eabqSg+Icq6DvjUjkMxOjYt
tSODdBcjwgehYKyXK09d/X62WOY1t7eA3ZNG6GDOKcSufr/U4UXbRSdUS1lyldQNHO6HYVo6BwLFDkXkKgqpF7/dircNAM5NuBV3
mQtamc490E+T5hQTIee1XdN0aoKfu27Z5vEuxol6lYnZlcUZadpoKSI8RE1SFQJJcWOI/SLs3kR0dhvocC9rMK2kMtDmeB1xczAE
7oyIhRCoStMiG7bJ7MV2NeMMj1l+5MjC4HdBty1oDtDt682AsPfjVx2miPuXBJ2ypFql8ZjQJX8ROKmjXkgdzcCj2d8bkNs0k6ra
GTsBGkyo+w3bqv7O5YO76pDSCDg1xS1x95ghbJPBbvKLe68VDok6hGkt4e74Fr+eDL3fHNsQ9sNoC4+RuhlTkfYSv18bCtgyRb9k
uTPFvIftRQuP7lpd0sgMp0AglaymNWYwpmlGrMIDL8kx5f/NuqAIvQncqUh5z2tUlimEClz2LElWQeJUez09Q1aYyWBRXtkm7jGY
DwYFmivFuWYCMgndNuizqNfdbIokiSuQO4a8typDhu1eWyI6SNelYENRQfH8w5VSeF/TrBMSxq7BkcasjtG+JbjXT3gc0uNiW7+H
/RSUUHCwwH/9s7jdgE4AED9YSBob/TpMQwmD3deMdVQFNuG0u2PrShOKsajy6zaryGgnZLWwxBHzRUQbsKB9d8Ld9E4fMQO8h0Sk
ziH7BrcDDm/CaLIAAAAA2+6RQ4t5DAcAAYk63K0DADuqdIexxGf7AgAAAAAEWVo=
"""
txt = lzma.decompress(base64.b64decode("".join(B64.split())))
assert hashlib.sha256(txt).hexdigest().startswith("1d7da3d62521ebd7"), "Tree data corrupted in copy-paste"
raw = pd.read_csv(io.BytesIO(txt))

J = {"hard maple/oak/hickory/beech": (-2.0705, 2.4410), "soft maple/birch": (-1.9123, 2.3651),
     "mixed hardwood": (-2.4800, 2.4835), "true fir/hemlock": (-2.5384, 2.4814)}   # Jenkins et al. 2003
def jgroup(g, s):
    if g in ("Quercus", "Carya", "Fagus") or (g == "Acer" and s == "saccharum"): return "hard maple/oak/hickory/beech"
    if g in ("Acer", "Betula"): return "soft maple/birch"
    if g == "Tsuga": return "true fir/hemlock"
    return "mixed hardwood"
MYC = {"Fagus": "ECM", "Quercus": "ECM", "Betula": "ECM", "Carya": "ECM", "Tsuga": "ECM", "Castanea": "ECM", "Ostrya": "ECM",
       "Tilia": "ECM", "Acer": "AM", "Liriodendron": "AM", "Prunus": "AM", "Fraxinus": "AM", "Magnolia": "AM", "Nyssa": "AM",
       "Sassafras": "AM", "Robinia": "AM", "Ulmus": "AM", "Cornus": "AM", "Aralia": "AM", "Asimina": "AM", "Amelanchier": "AM",
       "Oxydendrum": "ERM"}
t = pd.DataFrame({
    "tree_uid": [f"T{i+1:04d}" for i in range(len(raw))],
    "plot_id": raw["plot"].replace({"WS4-A-11": "WS4A-11"}),
    "plot_label_raw": raw["plot"], "watershed": raw["watershed"],
    "tag_number": pd.to_numeric(raw["tag_number"], errors="coerce").astype("Int64"),
    "common_name": raw["name"], "genus": raw["genus"], "species": raw["species"],
    "dbh_in": raw["diameter"], "dbh_cm": (raw["diameter"] * 2.54).round(3),
    "crown_class": raw["crown_class"], "origin": raw["origin"]})
t["ba_m2"] = (np.pi * (t.dbh_cm / 200) ** 2).round(6)
t["jenkins_group"] = [jgroup(g, s) for g, s in zip(t.genus, t.species)]
t["agb_kg_jenkins"] = [round(float(np.exp(J[k][0] + J[k][1] * np.log(d))), 3) if pd.notna(d) else np.nan
                       for k, d in zip(t.jenkins_group, t.dbh_cm)]
t["agb_kg_legacy_cm_assumed"] = raw["agb"]
t["myc_type"] = t.genus.map(MYC).fillna("uncertain")
t["qc_note"] = np.select([raw["plot"].eq("WS4-A-11"), raw["diameter"].isna()],
    ["untagged stems entered as WS4-A-11; merged into WS4A-11 (confirmed by DC 2026-09-11)", "DBH missing"], default="")

write_df(t, "trees")
spl = (t.groupby(["genus", "species"]).agg(common_name=("common_name", "first"), jenkins_group=("jenkins_group", "first"),
       myc_type=("myc_type", "first"), n_stems=("tree_uid", "size")).reset_index())
write_df(spl, "species_lookup")
sp_ids = {r[0] for r in arcpy.da.SearchCursor(P, ["plot_id"])}
print("tree plot_ids not in plots_study:", set(t.plot_id) - sp_ids or "none")
t.groupby("plot_id").size()

trees: 712 rows, 18 fields
species_lookup: 26 rows, 6 fields
tree plot_ids not in plots_study: none


plot_id
WS2A-1     29
WS2A-3     30
WS2A-5     55
WS2B-1     43
WS2B-2     34
WS2B-4     40
WS4A-10    22
WS4A-11    28
WS4A-13    30
WS4A-9     19
WS4B-1     28
WS4C-1     33
WS5-1      67
WS5-2      43
WS5-3      63
WS5-4      43
WS5-5      52
WS5-6      53
dtype: int64

In [18]:
u = pd.read_excel(XLSX, sheet_name="Used_Data")
u = u[u["watershed"].isin(["WS2", "WS4", "WS5"])].copy()
dmap = {"OH": ("OH", np.nan, np.nan), "0 to 20": ("0-20", 0.0, 20.0), "20 to 40": ("20-40", 20.0, 40.0)}
frac_cols = list(u.columns[u.columns.get_loc("frac_soil_wt_g"):])

b = u.drop(columns=frac_cols).rename(columns={"ID": "sample_id", "plot": "soil_plot", "subcomp": "soil_subcomp"})
b.insert(4, "subplot_letter", b["subplot"].str[-1])
b.insert(6, "depth_code", b["depth"].map(lambda d: dmap[d][0]))
b.insert(7, "depth_top_cm", b["depth"].map(lambda d: dmap[d][1]))
b.insert(8, "depth_bottom_cm", b["depth"].map(lambda d: dmap[d][2]))
b["treatment"] = b["watershed"].map(TRT)
b["bd_note"] = np.where(b["depth"] == "OH", "organic horizon, 20x20 cm frame; no BD",
                        "BD from one core per plot x depth, applied to all subplots")
write_df(b, "soil_samples")

f = u.loc[u[frac_cols].notna().any(axis=1), ["ID", "watershed", "subcomp", "plot", "depth"] + frac_cols]
f = f.rename(columns={"ID": "source_sample_id", "subcomp": "soil_subcomp", "plot": "soil_plot"})
f.insert(5, "depth_code", f["depth"].map(lambda d: dmap[d][0]))
f["treatment"] = f["watershed"].map(TRT)
f["composite_note"] = "5 g from each subplot (A-C) pooled and density-fractionated; stored on subplot A row in source"
write_df(f, "soil_fractions")

soil_samples: 162 rows, 26 fields
soil_fractions: 36 rows, 34 fields


In [19]:
import requests
SDA = "https://sdmdataaccess.sc.egov.usda.gov/Tabular/post.rest"
def sda(q):
    r = requests.post(SDA, data={"query": q, "format": "JSON+COLUMNNAME"}, timeout=120); r.raise_for_status()
    t = r.json().get("Table", [])
    return pd.DataFrame(t[1:], columns=t[0]) if t else pd.DataFrame()

with arcpy.da.SearchCursor(P, ["plot_id", "SHAPE@XY"], spatial_reference=arcpy.SpatialReference(4326)) as cur:
    ll = {pid: xy for pid, xy in cur}
pm = []
for pid, (lon, lat) in ll.items():
    d = sda(f"SELECT * FROM SDA_Get_Mukey_from_intersection_with_WktWgs84('point({lon:.7f} {lat:.7f})')")
    pm.append((pid, str(d.iloc[0, 0]) if len(d) else None))
pm = pd.DataFrame(pm, columns=["plot_id", "mukey"])
keys = ",".join(f"'{k}'" for k in sorted(pm.mukey.dropna().unique()))
print(pm.groupby("mukey").plot_id.apply(list))

mu = sda(f"""SELECT m.mukey, m.musym, m.muname, a.brockdepmin, a.drclassdcd, a.hydgrpdcd, a.aws0100wta
             FROM mapunit m JOIN muaggatt a ON a.mukey = m.mukey WHERE m.mukey IN ({keys})""")
hz = sda(f"""SELECT c.mukey, c.cokey, c.compname, c.comppct_r, c.majcompflag, c.taxclname, c.drainagecl,
             h.chkey, h.hzname, h.hzdept_r, h.hzdepb_r, h.sandtotal_r, h.silttotal_r, h.claytotal_r, h.om_r,
             h.dbthirdbar_r, h.awc_r, h.ph1to1h2o_r, h.cec7_r, h.ksat_r,
             (SELECT SUM(f.fragvol_r) FROM chfrags f WHERE f.chkey = h.chkey) AS fragvol_r
             FROM component c JOIN chorizon h ON h.cokey = c.cokey WHERE c.mukey IN ({keys})""")
rs = sda(f"""SELECT c.mukey, c.cokey, c.compname, r.reskind, r.resdept_r
             FROM component c JOIN corestrictions r ON r.cokey = c.cokey WHERE c.mukey IN ({keys})""")
print("map units:", len(mu), "| horizons:", len(hz), "| restrictions:", len(rs))
mu

mukey
513979                          [WS4A-11, WS4A-9, WS5-3]
513981                    [WS2A-1, WS2A-3, WS5-4, WS5-5]
513983                                 [WS4A-13, WS4C-1]
513985    [WS2A-5, WS2B-1, WS2B-2, WS2B-4, WS5-2, WS5-6]
513997                                         [WS4A-10]
514003                                          [WS4B-1]
514008                                           [WS5-1]
Name: plot_id, dtype: object
map units: 7 | horizons: 37 | restrictions: 8


,mukey,musym,muname,brockdepmin,drclassdcd,hydgrpdcd,aws0100wta
0,513979,CaC,"Calvin channery silt loam, 8 to 15 percent slopes",119,Well drained,A,11.29
1,513981,CaD,"Calvin channery silt loam, 15 to 25 percent sl...",119,Well drained,A,11.29
2,513983,CaE,"Calvin channery silt loam, 25 to 35 percent sl...",119,Well drained,A,11.29
3,513985,CaF,"Calvin channery silt loam, 35 to 65 percent sl...",119,Well drained,A,11.29
4,513997,DaC,"Dekalb channery loam, 8 to 15 percent slopes",86,Well drained,A,7.44
5,514003,DmC,"Dekalb very cobbly loam, 3 to 15 percent slope...",86,Well drained,A,7.44
6,514008,EnD,"Ernest silt loam, moist, 15 to 25 percent slopes",None,Moderately well drained,D,12.81


In [20]:
props = ["sandtotal_r", "silttotal_r", "claytotal_r", "om_r", "dbthirdbar_r", "awc_r", "ph1to1h2o_r", "cec7_r", "ksat_r", "fragvol_r"]
for c in props + ["hzdept_r", "hzdepb_r", "comppct_r"]:
    hz[c] = pd.to_numeric(hz[c], errors="coerce")
for c in ["brockdepmin", "aws0100wta"]:
    mu[c] = pd.to_numeric(mu[c], errors="coerce")

def wavg(v, w):
    m = v.notna() & (w > 0)
    return float(np.average(v[m], weights=w[m])) if m.any() else np.nan

def layer(top, bot):
    h = hz.assign(ov=(np.minimum(hz.hzdepb_r, bot) - np.maximum(hz.hzdept_r, top)).clip(lower=0))
    h = h[h.ov > 0]
    comp = h.groupby(["mukey", "cokey"]).apply(
        lambda g: pd.Series({"comppct_r": g.comppct_r.iloc[0], **{p: wavg(g[p], g.ov) for p in props}}), include_groups=False).reset_index()
    m = comp.groupby("mukey").apply(lambda g: pd.Series({p: wavg(g[p], g.comppct_r) for p in props}), include_groups=False)
    return m.add_suffix(f"_{top}_{bot}")

mu_props = mu.set_index("mukey").join(pd.concat([layer(0, 20), layer(20, 40)], axis=1))
ssurgo_plot = pm.merge(mu_props.reset_index(), on="mukey", how="left")
write_df(ssurgo_plot, "ssurgo_plot")
write_df(hz, "ssurgo_horizons")
write_df(rs, "ssurgo_restrictions")

out = os.path.join(GDB, "rel_plots_study_ssurgo_plot")
if arcpy.Exists(out): arcpy.management.Delete(out)
arcpy.management.CreateRelationshipClass(P, os.path.join(GDB, "ssurgo_plot"), out, "SIMPLE", "ssurgo_plot", "plots_study",
                                         "NONE", "ONE_TO_ONE", "NONE", "plot_id", "plot_id")
ssurgo_plot.round(2)

<class 'TypeError'>: layer.<locals>.<lambda>() got an unexpected keyword argument 'include_groups'

In [21]:
props = ["sandtotal_r", "silttotal_r", "claytotal_r", "om_r", "dbthirdbar_r", "awc_r", "ph1to1h2o_r", "cec7_r", "ksat_r", "fragvol_r"]
for c in props + ["hzdept_r", "hzdepb_r", "comppct_r"]:
    hz[c] = pd.to_numeric(hz[c], errors="coerce")
for c in ["brockdepmin", "aws0100wta"]:
    mu[c] = pd.to_numeric(mu[c], errors="coerce")

def layer(top, bot):
    """Horizon values weighted by overlap with [top, bot] cm, then components weighted by comppct_r."""
    h = hz.assign(ov=(np.minimum(hz.hzdepb_r, bot) - np.maximum(hz.hzdept_r, top)).clip(lower=0))
    h = h[h.ov > 0]
    pct = h.groupby(["mukey", "cokey"])["comppct_r"].first()
    out = {}
    for p in props:
        w = h["ov"].where(h[p].notna(), 0.0)
        num = (h[p].fillna(0) * w).groupby([h["mukey"], h["cokey"]]).sum()
        den = w.groupby([h["mukey"], h["cokey"]]).sum()
        comp = num / den.replace(0, np.nan)
        ok = comp.notna() & (pct > 0)
        out[p] = (comp[ok] * pct[ok]).groupby(level="mukey").sum() / pct[ok].groupby(level="mukey").sum()
    return pd.DataFrame(out).add_suffix(f"_{top}_{bot}")

mu_props = mu.set_index("mukey").join(pd.concat([layer(0, 20), layer(20, 40)], axis=1))
ssurgo_plot = pm.merge(mu_props.reset_index(), on="mukey", how="left")
write_df(ssurgo_plot, "ssurgo_plot")
write_df(hz, "ssurgo_horizons")
write_df(rs, "ssurgo_restrictions")

out = os.path.join(GDB, "rel_plots_study_ssurgo_plot")
if arcpy.Exists(out): arcpy.management.Delete(out)
arcpy.management.CreateRelationshipClass(P, os.path.join(GDB, "ssurgo_plot"), out, "SIMPLE", "ssurgo_plot", "plots_study",
                                         "NONE", "ONE_TO_ONE", "NONE", "plot_id", "plot_id")
ssurgo_plot.round(2)

ssurgo_plot: 18 rows, 28 fields
ssurgo_horizons: 37 rows, 21 fields
ssurgo_restrictions: 8 rows, 5 fields


,plot_id,mukey,musym,muname,brockdepmin,drclassdcd,hydgrpdcd,aws0100wta,sandtotal_r_0_20,silttotal_r_0_20,...,sandtotal_r_20_40,silttotal_r_20_40,claytotal_r_20_40,om_r_20_40,dbthirdbar_r_20_40,awc_r_20_40,ph1to1h2o_r_20_40,cec7_r_20_40,ksat_r_20_40,fragvol_r_20_40
0,WS2A-1,513981,CaD,"Calvin channery silt loam, 15 to 25 percent sl...",119.0,Well drained,A,11.29,26.76,53.99,...,22.40,55.10,22.50,0.26,1.50,0.12,5.30,16.00,28.00,26.0
1,WS2A-3,513981,CaD,"Calvin channery silt loam, 15 to 25 percent sl...",119.0,Well drained,A,11.29,26.76,53.99,...,22.40,55.10,22.50,0.26,1.50,0.12,5.30,16.00,28.00,26.0
2,WS2A-5,513985,CaF,"Calvin channery silt loam, 35 to 65 percent sl...",119.0,Well drained,A,11.29,26.76,53.99,...,22.40,55.10,22.50,0.26,1.50,0.12,5.30,16.00,28.00,26.0
3,WS2B-1,513985,CaF,"Calvin channery silt loam, 35 to 65 percent sl...",119.0,Well drained,A,11.29,26.76,53.99,...,22.40,55.10,22.50,0.26,1.50,0.12,5.30,16.00,28.00,26.0
4,WS2B-2,513985,CaF,"Calvin channery silt loam, 35 to 65 percent sl...",119.0,Well drained,A,11.29,26.76,53.99,...,22.40,55.10,22.50,0.26,1.50,0.12,5.30,16.00,28.00,26.0
5,WS2B-4,513985,CaF,"Calvin channery silt loam, 35 to 65 percent sl...",119.0,Well drained,A,11.29,26.76,53.99,...,22.40,55.10,22.50,0.26,1.50,0.12,5.30,16.00,28.00,26.0
6,WS4A-10,513997,DaC,"Dekalb channery loam, 8 to 15 percent slopes",86.0,Well drained,A,7.44,44.30,40.70,...,45.76,42.24,12.00,1.46,1.35,0.09,4.60,7.38,92.00,42.6
7,WS4A-11,513979,CaC,"Calvin channery silt loam, 8 to 15 percent slopes",119.0,Well drained,A,11.29,26.76,53.99,...,22.40,55.10,22.50,0.26,1.50,0.12,5.30,16.00,28.00,26.0
8,WS4A-13,513983,CaE,"Calvin channery silt loam, 25 to 35 percent sl...",119.0,Well drained,A,11.29,26.76,53.99,...,22.40,55.10,22.50,0.26,1.50,0.12,5.30,16.00,28.00,26.0
9,WS4A-9,513979,CaC,"Calvin channery silt loam, 8 to 15 percent slopes",119.0,Well drained,A,11.29,26.76,53.99,...,22.40,55.10,22.50,0.26,1.50,0.12,5.30,16.00,28.00,26.0


In [22]:
from arcpy.sa import AreaSolarRadiation, TimeWholeYear, ZonalStatisticsAsTable
arcpy.CheckOutExtension("Spatial")
dem = os.path.join(GDB, "terrain_dem5")
arcpy.env.extent = arcpy.Describe(dem).extent
sol = AreaSolarRadiation(dem, sky_size=200, time_configuration=TimeWholeYear(2022),
                         day_interval=14, hour_interval=0.5, each_interval="NOINTERVAL",
                         slope_aspect_input_type="FROM_DEM", calculation_directions=32)
(sol / 1000).save(os.path.join(GDB, "terrain_solar_kwh"))       # kWh m-2 yr-1
arcpy.env.extent = None

ZonalStatisticsAsTable(os.path.join(GDB, "plots_footprint"), "plot_id", os.path.join(GDB, "terrain_solar_kwh"), r"memory\zs", "DATA", "MEAN")
solar = to_df(r"memory\zs")[["plot_id", "MEAN"]].rename(columns={"MEAN": "solar_kwh_m2_yr_fp"})
write_df(solar, "plot_solar")
out = os.path.join(GDB, "rel_plots_study_plot_solar")
if arcpy.Exists(out): arcpy.management.Delete(out)
arcpy.management.CreateRelationshipClass(P, os.path.join(GDB, "plot_solar"), out, "SIMPLE", "plot_solar", "plots_study",
                                         "NONE", "ONE_TO_ONE", "NONE", "plot_id", "plot_id")
ZonalStatisticsAsTable(os.path.join(GDB, "ref_watersheds"), "Ws_name", os.path.join(GDB, "terrain_solar_kwh"), r"memory\zsw", "DATA", "ALL")
print(to_df(r"memory\zsw")[["Ws_name", "MIN", "MEAN", "MAX"]].round(0))
solar.round(0)

plot_solar: 18 rows, 2 fields
  Ws_name     MIN    MEAN     MAX
0     WS1   890.0  1256.0  1422.0
1     WS4   868.0  1348.0  1483.0
2     WS2  1153.0  1364.0  1476.0
3     WS5   823.0  1256.0  1464.0


,plot_id,solar_kwh_m2_yr_fp
0,WS2A-1,1462.0
1,WS2A-3,1358.0
2,WS2A-5,1335.0
3,WS2B-1,1305.0
4,WS2B-2,1341.0
5,WS2B-4,1225.0
6,WS4A-10,1359.0
7,WS4A-11,1374.0
8,WS4A-13,1391.0
9,WS4A-9,1372.0


In [23]:
ms = to_df(os.path.join(GDB, "meta_sources"))
ms = pd.concat([ms, pd.DataFrame([
    ("ssurgo_plot / ssurgo_horizons / ssurgo_restrictions", "USDA-NRCS SSURGO via Soil Data Access, queried by plot coordinate (2026-09-11)",
     "mapped estimates; 0-20 and 20-40 cm depth-weighted, component-%-weighted; 3 distinct profiles (Calvin, Dekalb, Ernest)"),
    ("terrain_solar_kwh / plot_solar", "ArcGIS Area Solar Radiation on 5-m lidar DEM",
     "whole year, 14-day / 0.5-h intervals, 32 directions; horizon shading limited to DEM extent (watersheds + 300 m)")],
    columns=ms.columns)])
write_df(ms, "meta_sources")
qc = to_df(os.path.join(GDB, "meta_qc_log"))
qc = pd.concat([qc, pd.DataFrame([
    ("ssurgo_plot", "Current SSURGO map units (by coordinate) may differ from NRCS 2008 units in plot_site; both retained", ""),
    ("ssurgo_plot", "Mapped slope classes disagree with lidar slope at some plots (e.g., WS5-3 mapped 8-15%, lidar ~36%)", "use lidar for slope")],
    columns=qc.columns)])
write_df(qc, "meta_qc_log")

meta_sources: 9 rows, 3 fields
meta_qc_log: 15 rows, 3 fields


In [24]:
arcpy.env.workspace = GDB
inv = [(n, arcpy.Describe(n).dataType, int(arcpy.management.GetCount(n)[0])) for n in arcpy.ListFeatureClasses() + arcpy.ListTables()]
inv += [(n, "Raster", None) for n in arcpy.ListRasters()]
pd.DataFrame(inv, columns=["name", "type", "rows"]).sort_values("name")

,name,type,rows
25,meta_qc_log,TableView,15.0
24,meta_sources,TableView,9.0
11,plot_site,TableView,18.0
23,plot_solar,TableView,18.0
12,plot_terrain,TableView,18.0
10,plots_footprint,FeatureLayer,18.0
9,plots_study,FeatureLayer,18.0
2,ref_dc_growth_plots_all,FeatureLayer,30.0
4,ref_elt_phase,FeatureLayer,126.0
7,ref_fernow_boundary,FeatureLayer,1.0


In [25]:
exp = os.path.join(PROJ, "gdb_export"); os.makedirs(exp, exist_ok=True)
arcpy.env.workspace = GDB
rels = [c.name for c in arcpy.Describe(GDB).children if c.dataType == "RelationshipClass"]
print("relationship classes:", len(rels), rels)

for n in arcpy.ListFeatureClasses() + arcpy.ListTables():
    is_pt = arcpy.Describe(n).dataType in ("FeatureClass", "FeatureLayer") and arcpy.Describe(n).shapeType == "Point"
    to_df(n, xy=is_pt).to_csv(os.path.join(exp, f"{n}.csv"), index=False)
print(len(os.listdir(exp)), "CSVs written to", exp)

relationship classes: 2 ['rel_plots_study_ssurgo_plot', 'rel_plots_study_plot_solar']
26 CSVs written to C:\Users\DCifelli2\OneDrive - mercercounty.org\Documents\ArcGIS\Projects\Fernow-FM-SOM-DC-092026\gdb_export


In [26]:
# --- subcompartment prescriptions (Schuler 2014, Table 7.1) ---
sc = pd.DataFrame([
    ("WS2A", "WS2", "Diameter-limit",        80, "excellent", 15.0,   "cut all trees >= 43 cm (17 in) DBH", np.nan, np.nan),
    ("WS2B", "WS2", "Diameter-limit",        60, "fair",      20.0,   "cut all trees >= 43 cm (17 in) DBH", np.nan, np.nan),
    ("WS4A", "WS4", "Reference",             80, "excellent", np.nan, "unmanaged",                          np.nan, np.nan),
    ("WS4B", "WS4", "Reference",             70, "good",      np.nan, "unmanaged",                          np.nan, np.nan),
    ("WS4C", "WS4", "Reference",             60, "fair",      np.nan, "unmanaged",                          np.nan, np.nan),
    ("WS5A", "WS5", "Single-tree selection", 80, "excellent", 10.0,   "BDq single-tree selection, Q = 1.3", 15.0, 81.0),
    ("WS5B", "WS5", "Single-tree selection", 60, "fair",      15.0,   "BDq single-tree selection, Q = 1.3",  8.0, 51.0)],
    columns=["subcomp", "watershed", "treatment", "site_index_nro", "site_class", "cutting_cycle_yr",
             "prescription", "rba_m2ha_trees_ge28cm", "largest_tree_retained_cm"])
sc["source"] = "Schuler 2014 (Springer, Experimental Forests and Ranges, pp 149-171), Table 7.1; LAMP study est. 1950"
write_df(sc, "subcompartments_info")

wsi = to_df(os.path.join(GDB, "watersheds_info"))
wsi["harvest_notes"] = ["Diameter-limit (LAMP): all trees >= 43 cm (17 in) cut; 15-yr cycle in WS2A (SI 80), 20-yr in WS2B (SI 60). Schuler 2014",
                        "Unmanaged reference since 1903-1911 railroad logging; WS4A/B/C = SI 80/70/60. Schuler 2014",
                        "Single-tree selection (LAMP, BDq Q=1.3): WS5A SI 80, 10-yr cycle; WS5B SI 60, 15-yr cycle. Schuler 2014"]
write_df(wsi, "watersheds_info")

qc = to_df(os.path.join(GDB, "meta_qc_log"))
qc = pd.concat([qc, pd.DataFrame([
    ("plots_study", "WS5-6 label lacks subcompartment; thesis states one WS5 plot was in WS5B (15-yr cycle)", "OPEN: confirm WS5A vs WS5B"),
    ("plots_study", "Thesis states one WS2 plot in WS2B; USFS/tree labels place 3 plots (B-1, B-2, B-4) in WS2B", "OPEN: labels treated as authoritative")],
    columns=qc.columns)])
write_df(qc, "meta_qc_log")

# --- all relationship classes ---
def rel(origin, dest, okey, dkey, card="ONE_TO_MANY"):
    out = os.path.join(GDB, f"rel_{os.path.basename(origin)}_{dest}")
    if arcpy.Exists(out): arcpy.management.Delete(out)
    arcpy.management.CreateRelationshipClass(origin, os.path.join(GDB, dest), out, "SIMPLE", dest,
                                             os.path.basename(origin), "NONE", card, "NONE", okey, dkey)
rel(P, "trees", "plot_id", "plot_id")
for d in ["plot_site", "plot_terrain"]: rel(P, d, "plot_id", "plot_id", "ONE_TO_ONE")
W = os.path.join(GDB, "ref_watersheds")
for d in ["soil_samples", "soil_fractions"]: rel(W, d, "Ws_name", "watershed")
rel(os.path.join(GDB, "subcompartments_info"), "plots_study", "subcomp", "usfs_subcomp")
print("relationship classes:", [c.name for c in arcpy.Describe(GDB).children if c.dataType == "RelationshipClass"])

# --- what other shapefiles do you have? (looking for compartment boundaries) ---
print("\nshapefiles under", SHP_ROOT)
for r, _, fs in os.walk(SHP_ROOT):
    for f in fs:
        if f.endswith(".shp"): print("  ", os.path.relpath(os.path.join(r, f), SHP_ROOT))

# --- re-export CSVs for GitHub ---
exp = os.path.join(PROJ, "gdb_export"); os.makedirs(exp, exist_ok=True)
arcpy.env.workspace = GDB
for n in arcpy.ListFeatureClasses() + arcpy.ListTables():
    is_pt = arcpy.Describe(n).dataType in ("FeatureClass", "FeatureLayer") and arcpy.Describe(n).shapeType == "Point"
    to_df(n, xy=is_pt).to_csv(os.path.join(exp, f"{n}.csv"), index=False)
print(len(os.listdir(exp)), "CSVs in", exp)

subcompartments_info: 7 rows, 10 fields
watersheds_info: 3 rows, 5 fields
meta_qc_log: 17 rows, 3 fields
relationship classes: ['rel_plots_study_ssurgo_plot', 'rel_plots_study_plot_solar', 'rel_plots_study_trees', 'rel_plots_study_plot_site', 'rel_plots_study_plot_terrain', 'rel_ref_watersheds_soil_samples', 'rel_ref_watersheds_soil_fractions', 'rel_subcompartments_info_plots_study']

shapefiles under C:\Users\DCifelli2\OneDrive - mercercounty.org\Desktop\FernowShapeFiles
   FEF_Shapefiles\Compartments\compartments.shp
   FEF_Shapefiles\cover_type_1932\cover_type_1932.shp
   FEF_Shapefiles\decks\decks.shp
   FEF_Shapefiles\dendro_plots\Dendroplots.shp
   FEF_Shapefiles\ecological_land_type_phase\ecological_land_type_phase.shp
   FEF_Shapefiles\elklick_drainage\elklick_drainage.shp
   FEF_Shapefiles\fernow_boundary\CopyOffernow_boundary.shp
   FEF_Shapefiles\fernow_boundary\fernow_boundary.shp
   FEF_Shapefiles\growth_plots\growth_plots.shp
   FEF_Shapefiles\landtype_association\land_ty

In [27]:
ws3 = arcpy.management.MakeFeatureLayer(os.path.join(GDB, "ref_watersheds"), "ws3", "Ws_name IN ('WS2','WS4','WS5')")
rows = []
for r, _, fs in os.walk(os.path.join(SHP_ROOT, "FEF_Shapefiles")):
    for f in fs:
        if not f.endswith(".shp"): continue
        p = os.path.join(r, f)
        try:
            d = arcpy.Describe(p)
            lyr = arcpy.management.MakeFeatureLayer(p, "survey_lyr")
            arcpy.management.SelectLayerByLocation(lyr, "WITHIN_A_DISTANCE", ws3, "100 Meters", "NEW_SELECTION")
            n_in = int(arcpy.management.GetCount(lyr)[0])
            arcpy.management.Delete("survey_lyr")
            flds = [x.name for x in arcpy.ListFields(p) if x.type not in ("Geometry", "OID")]
            rows.append((f[:-4], d.shapeType, int(arcpy.management.GetCount(p)[0]), n_in, d.spatialReference.name, ", ".join(flds)))
        except Exception as e:
            rows.append((f[:-4], "ERROR", None, None, str(e)[:80], ""))
survey = pd.DataFrame(rows, columns=["layer", "geom", "n_total", "n_near_WS245", "crs", "fields"]).sort_values("n_near_WS245", ascending=False)
survey.to_csv(os.path.join(PROJ, "gdb_export", "shapefile_survey.csv"), index=False)
pd.set_option("display.max_colwidth", 120); pd.set_option("display.max_rows", 80)
print(survey[["layer", "geom", "n_total", "n_near_WS245", "crs"]].to_string(index=False))

# subcompartment + compartment membership of the 18 plots
for name in ["subcompartments", "compartments"]:
    p = find_shp(name)
    arcpy.analysis.SpatialJoin(P, p, r"memory\sj_" + name, "JOIN_ONE_TO_ONE", "KEEP_ALL", match_option="INTERSECT")
    d = to_df(r"memory\sj_" + name)
    extra = [c for c in d.columns if c not in to_df(P).columns and c not in ("Join_Count", "TARGET_FID")]
    print(f"\n{name}: fields {extra}")
    print(d[["plot_id", "usfs_subcomp"] + extra].to_string(index=False))

                            layer     geom  n_total  n_near_WS245                                                                              crs
                       WS4Biomass    Point    112.0         112.0                                                            NAD_1983_UTM_Zone_17N
                        road_skid Polyline    784.0         107.0                                                            NAD_1983_UTM_Zone_17N
                   WS3_Soils_2011    Point    100.0          41.0                                                            NAD_1983_UTM_Zone_17N
                       leaf_traps    Point     75.0          40.0                                                            NAD_1983_UTM_Zone_17N
                          streams Polyline    382.0          35.0                                                            NAD_1983_UTM_Zone_17N
                 USGS_contours_ft Polyline    345.0          34.0                                                     

In [28]:
arcpy.env.outputCoordinateSystem = SR

def add_cols(fc, df):
    """Add/replace columns on an existing table keyed by plot_id (keeps relationship classes intact)."""
    have = [f.name for f in arcpy.ListFields(fc)]
    for c in df.columns:
        if c in have: arcpy.management.DeleteField(fc, c)
        ft = _ftype(df[c]); L = max(16, int(df[c].astype(str).str.len().max()) + 8) if ft == "TEXT" else None
        arcpy.management.AddField(fc, c, ft, field_length=L)
    with arcpy.da.UpdateCursor(fc, ["plot_id"] + list(df.columns)) as cur:
        for row in cur:
            if row[0] in df.index:
                vals = [None if (not isinstance(v, str) and pd.isna(v)) else (v.item() if isinstance(v, np.generic) else v)
                        for v in df.loc[row[0]].tolist()]
                cur.updateRow([row[0]] + vals)

# 1. mapped subcompartment for each plot
arcpy.conversion.ExportFeatures(find_shp("subcompartments"), os.path.join(GDB, "ref_subcompartments"))
arcpy.analysis.SpatialJoin(P, os.path.join(GDB, "ref_subcompartments"), r"memory\sjs", "JOIN_ONE_TO_ONE", "KEEP_ALL", match_option="INTERSECT")
m = to_df(r"memory\sjs").set_index("plot_id")
lab = to_df(P).set_index("plot_id")["usfs_subcomp"]
sub = pd.DataFrame({"subcomp_mapped": m["Scomp_name"], "site_index_mapped": m["Site_index"],
                    "cutting_cycle_mapped": m["Rotation"], "treatment_mapped": m["Treatment_1"]})
sub["label_matches_map"] = (lab.reindex(sub.index) == sub["subcomp_mapped"]).astype(int)
add_cols(P, sub)
print(sub)

# 2. subcompartment table: add the layer's measured values alongside Schuler 2014
rs = to_df(os.path.join(GDB, "ref_subcompartments"))
rs = rs[rs["Scomp_name"].isin(sc["subcomp"])][["Scomp_name", "Site_index", "Rotation", "Treatment_1", "Survey_ac", "Establish"]]
rs.columns = ["subcomp", "site_index_layer", "cutting_cycle_layer", "treatment_layer", "area_ac_layer", "established_layer"]
sc2 = sc.merge(rs, on="subcomp", how="left")
write_df(sc2, "subcompartments_info")
out = os.path.join(GDB, "rel_subcompartments_info_plots_study")
if arcpy.Exists(out): arcpy.management.Delete(out)
arcpy.management.CreateRelationshipClass(os.path.join(GDB, "subcompartments_info"), P, out, "SIMPLE", "plots_study",
                                         "subcompartments_info", "NONE", "ONE_TO_MANY", "NONE", "subcomp", "subcomp_mapped")

# 3. reference layers into the gdb (reprojected to UTM 17N)
ref_list = ["road_skid", "road_truck", "decks", "cover_type_1932", "surficial_geology", "potential_natural_vegetation",
            "soil_map_units_NRCS1971", "soil_series_NRCS1971", "WS4Biomass", "WS4_Soils_1997", "WS4_leaf_mass", "WS4_leaf_mass2",
            "WS4_Gilliam_Plots", "leaf_traps", "lysimeters", "mean_lysimeter_chem_A_horizon", "mean_lysimeter_chem_B_horizon",
            "mean_lysimeter_chem_C_horizon", "mean_WS4_lysimeter_chem_A_horizon", "LTREB_Damage_Plots", "LTREB_Sites",
            "Dendroplots", "Ten_Four_Project_Plots", "weirs", "rain_gauges"]
for n in ref_list:
    try:
        o = os.path.join(GDB, "ref_" + n.lower())
        arcpy.conversion.ExportFeatures(find_shp(n), o)
        print(f"ref_{n.lower()}: {arcpy.management.GetCount(o)[0]}")
    except Exception as e:
        print(f"ref_{n.lower()}: FAILED - {str(e)[:100]}")
arcpy.env.outputCoordinateSystem = None

# 4. what's inside the point datasets and the key polygon layers
for n in ["WS4Biomass", "WS4_Soils_1997", "WS4_leaf_mass", "WS4_Gilliam_Plots", "leaf_traps", "LTREB_Damage_Plots",
          "Ten_Four_Project_Plots", "Dendroplots", "cover_type_1932", "surficial_geology", "potential_natural_vegetation",
          "road_skid", "decks"]:
    o = os.path.join(GDB, "ref_" + n.lower())
    if arcpy.Exists(o):
        d = to_df(o)
        print(f"\n=== {n} ({len(d)} rows) ===")
        print(d.head(3).to_string(max_colwidth=30))

        subcomp_mapped  site_index_mapped cutting_cycle_mapped  \
plot_id                                                          
WS2A-1            WS2A                 75                 15yr   
WS2A-3            WS2A                 75                 15yr   
WS2A-5            WS2A                 75                 15yr   
WS2B-1            WS2A                 75                 15yr   
WS2B-2            WS2A                 75                 15yr   
WS2B-4            WS2B                 59                 20yr   
WS4A-10           WS4A                 81              control   
WS4A-11           WS4A                 81              control   
WS4A-13           WS4A                 81              control   
WS4A-9            WS4A                 81              control   
WS4B-1            WS4B                 70              control   
WS4C-1            WS4C                 64              control   
WS5-1             WS5A                 75                 10yr   
WS5-2     

<class 'KeyError'>: "['Treatment_1'] not in index"

In [29]:
# 2. subcompartment table: layer values alongside Schuler 2014
rs = to_df(os.path.join(GDB, "ref_subcompartments"))
rs = rs[rs["Scomp_name"].isin(sc["subcomp"])][["Scomp_name", "Site_index", "Rotation", "Treatment", "Survey_ac", "Establish"]]
rs.columns = ["subcomp", "site_index_layer", "cutting_cycle_layer", "treatment_layer", "area_ac_layer", "established_layer"]
write_df(sc.merge(rs, on="subcomp", how="left"), "subcompartments_info")
out = os.path.join(GDB, "rel_subcompartments_info_plots_study")
if arcpy.Exists(out): arcpy.management.Delete(out)
arcpy.management.CreateRelationshipClass(os.path.join(GDB, "subcompartments_info"), P, out, "SIMPLE", "plots_study",
                                         "subcompartments_info", "NONE", "ONE_TO_MANY", "NONE", "subcomp", "subcomp_mapped")

# QC log: WS5-6 resolved; drop the WS2B item
qc = to_df(os.path.join(GDB, "meta_qc_log"))
qc = qc[~qc["issue"].str.contains("WS2B|WS5-6 label lacks", regex=True)]
qc = pd.concat([qc, pd.DataFrame([("plots_study", "Subcompartment assigned from mapped boundaries (subcompartments.shp): WS5-6 = WS5B",
                                   "resolved 2026-09-11")], columns=qc.columns)])
write_df(qc, "meta_qc_log")

# 3. reference layers into the gdb (reprojected to UTM 17N)
arcpy.env.outputCoordinateSystem = SR
ref_list = ["road_skid", "road_truck", "decks", "cover_type_1932", "surficial_geology", "potential_natural_vegetation",
            "soil_map_units_NRCS1971", "soil_series_NRCS1971", "WS4Biomass", "WS4_Soils_1997", "WS4_leaf_mass", "WS4_leaf_mass2",
            "WS4_Gilliam_Plots", "leaf_traps", "lysimeters", "mean_lysimeter_chem_A_horizon", "mean_lysimeter_chem_B_horizon",
            "mean_lysimeter_chem_C_horizon", "mean_WS4_lysimeter_chem_A_horizon", "LTREB_Damage_Plots", "LTREB_Sites",
            "Dendroplots", "Ten_Four_Project_Plots", "weirs", "rain_gauges"]
for n in ref_list:
    try:
        o = os.path.join(GDB, "ref_" + n.lower())
        arcpy.conversion.ExportFeatures(find_shp(n), o)
        print(f"ref_{n.lower()}: {arcpy.management.GetCount(o)[0]}")
    except Exception as e:
        print(f"ref_{n.lower()}: FAILED - {str(e)[:100]}")
arcpy.env.outputCoordinateSystem = None

# 4. what's inside the point datasets and key polygon layers
for n in ["WS4Biomass", "WS4_Soils_1997", "WS4_leaf_mass", "WS4_Gilliam_Plots", "leaf_traps", "LTREB_Damage_Plots",
          "Ten_Four_Project_Plots", "Dendroplots", "cover_type_1932", "surficial_geology", "potential_natural_vegetation",
          "road_skid", "decks"]:
    o = os.path.join(GDB, "ref_" + n.lower())
    if arcpy.Exists(o):
        d = to_df(o)
        print(f"\n=== {n} ({len(d)} rows) ===")
        print(d.head(3).to_string(max_colwidth=30))

subcompartments_info: 7 rows, 15 fields
meta_qc_log: 16 rows, 3 fields
ref_road_skid: 784
ref_road_truck: 104
ref_decks: 63
ref_cover_type_1932: 63
ref_surficial_geology: 263
ref_potential_natural_vegetation: 81
ref_soil_map_units_nrcs1971: 282
ref_soil_series_nrcs1971: 79
ref_ws4biomass: 112
ref_ws4_soils_1997: 16
ref_ws4_leaf_mass: 25
ref_ws4_leaf_mass2: 25
ref_ws4_gilliam_plots: 7
ref_leaf_traps: 75
ref_lysimeters: 30
ref_mean_lysimeter_chem_a_horizon: 28
ref_mean_lysimeter_chem_b_horizon: 27
ref_mean_lysimeter_chem_c_horizon: 16
ref_mean_ws4_lysimeter_chem_a_horizon: 15
ref_ltreb_damage_plots: 64
ref_ltreb_sites: 6
ref_dendroplots: 20
ref_ten_four_project_plots: 43
ref_weirs: 10
ref_rain_gauges: 14

=== WS4Biomass (112 rows) ===
   Id  Biomass        POINT_X       POINT_Y
0   0      335  613311.075890  4.324345e+06
1   0      227  613351.617877  4.324311e+06
2   0      393  613423.386250  4.324293e+06

=== WS4_Soils_1997 (16 rows) ===
   Plot  Easting  Northing   NNP_Sum  ffdep    

In [30]:
arcpy.env.workspace = GDB
pts = to_df(P, xy=True).set_index("plot_id")[["watershed", "SHAPE@X", "SHAPE@Y"]]
Pcols = set(to_df(P).columns)

def near_dist(fc, where=None):
    lyr = arcpy.management.MakeFeatureLayer(os.path.join(GDB, fc), "near_lyr", where)
    arcpy.management.CopyFeatures(P, r"memory\nr")
    arcpy.analysis.Near(r"memory\nr", lyr)
    arcpy.management.Delete("near_lyr")
    return to_df(r"memory\nr").set_index("plot_id")["NEAR_DIST"].round(1)

def nearest_pts(fc, cols, prefix, within=None):
    d = to_df(os.path.join(GDB, fc), xy=True)
    X, Y = d["SHAPE@X"].to_numpy(), d["SHAPE@Y"].to_numpy()
    out = {}
    for pid, r in pts.iterrows():
        dist = np.hypot(X - r["SHAPE@X"], Y - r["SHAPE@Y"]); i = int(np.argmin(dist))
        rec = {f"{prefix}dist_m": round(float(dist[i]), 1), **{f"{prefix}{c}": d[c].iloc[i] for c in cols}}
        if within:
            m = dist <= within; rec[f"{prefix}n_within_{within}m"] = int(m.sum())
            for c in cols:
                v = pd.to_numeric(d.loc[m, c], errors="coerce")
                rec[f"{prefix}{c}_mean_{within}m"] = float(v.mean()) if m.any() else np.nan
        out[pid] = rec
    return pd.DataFrame.from_dict(out, orient="index")

def poly_all(fc, prefix, drop=("AREA", "PERIMETER", "Shape_Length", "Shape_Area", "Shape_Leng", "OBJECTID")):
    arcpy.analysis.SpatialJoin(P, os.path.join(GDB, fc), r"memory\sjx", "JOIN_ONE_TO_ONE", "KEEP_ALL", match_option="INTERSECT")
    d = to_df(r"memory\sjx").set_index("plot_id")
    src = [f.name for f in arcpy.ListFields(os.path.join(GDB, fc)) if f.type not in ("Geometry", "OID")]
    return d[[c for c in src if c in d.columns and c not in drop and c not in Pcols]].add_prefix(prefix)

def rel(origin, dest, okey, dkey, card="ONE_TO_MANY"):
    out = os.path.join(GDB, f"rel_{os.path.basename(origin)}_{dest}")
    if arcpy.Exists(out): arcpy.management.Delete(out)
    arcpy.management.CreateRelationshipClass(origin, os.path.join(GDB, dest), out, "SIMPLE", dest,
                                             os.path.basename(origin), "NONE", card, "NONE", okey, dkey)

# 1. harvest infrastructure at each plot
dt = pd.DataFrame({"dist_skid_trail_m": near_dist("ref_road_skid"), "dist_truck_road_m": near_dist("ref_road_truck"),
                   "dist_log_deck_m": near_dist("ref_decks")})
for r, lab in [(11.35, "footprint"), (25, "25m"), (50, "50m"), (100, "100m")]:
    buf, ix = os.path.join(GDB, "tmp_buf"), os.path.join(GDB, "tmp_ix")
    arcpy.analysis.PairwiseBuffer(P, buf, f"{r} Meters")
    arcpy.analysis.PairwiseIntersect([buf, os.path.join(GDB, "ref_road_skid")], ix)
    L = pd.DataFrame(arcpy.da.SearchCursor(ix, ["plot_id", "SHAPE@LENGTH"]), columns=["plot_id", "len"]).groupby("plot_id")["len"].sum()
    dt[f"skid_m_per_ha_{lab}"] = (L.reindex(dt.index).fillna(0) / (np.pi * r**2 / 1e4)).round(1)
    arcpy.management.Delete(buf); arcpy.management.Delete(ix)
write_df(dt.reset_index(), "plot_disturbance")

# 2. history, geology, vegetation potential, 1971 soils at each plot
hist = pd.concat([poly_all("ref_cover_type_1932", "cover1932_"), poly_all("ref_surficial_geology", "geo_"),
                  poly_all("ref_potential_natural_vegetation", "pnv_"), poly_all("ref_soil_series_nrcs1971", "nrcs1971_"),
                  poly_all("ref_soil_map_units_nrcs1971", "nrcs1971mu_")], axis=1)
write_df(hist.reset_index(), "plot_history_geology")

# 3. links to earlier studies (nearest record; means within 50 m where useful)
s97 = ["Plot", "NNP_Sum", "ffdep", "phw0", "phc0", "c0", "n0", "cton0", "bs0", "cec0", "cl0", "si0", "sa0",
       "phw1", "phc1", "c1", "n1", "cton1", "bs1", "cec1"]
prior = pd.concat([nearest_pts("ref_ws4biomass", ["Biomass"], "ws4bio_", within=50),
                   nearest_pts("ref_ws4_soils_1997", s97, "ws4soil97_"),
                   nearest_pts("ref_ws4_leaf_mass", ["Leaf_trap", "Meanmass"], "ws4leaf_", within=50),
                   nearest_pts("ref_ws4_gilliam_plots", ["ID", "Plot"], "gilliam_"),
                   nearest_pts("ref_ltreb_damage_plots", ["Name"], "ltreb_"),
                   nearest_pts("ref_dendroplots", ["Name"], "dendro_")], axis=1)
write_df(prior.reset_index().rename(columns={"index": "plot_id"}), "plot_prior_studies")
for d in ["plot_disturbance", "plot_history_geology", "plot_prior_studies"]: rel(P, d, "plot_id", "plot_id", "ONE_TO_ONE")

# 4. watershed-level composition (area or length by class, WS2/WS4/WS5)
W3 = arcpy.management.MakeFeatureLayer(os.path.join(GDB, "ref_watersheds"), "w3", "Ws_name IN ('WS2','WS4','WS5')")
for fc, fld, name in [("ref_cover_type_1932", "COVER_TYPE", "watershed_cover1932"), ("ref_surficial_geology", "SURF_GEO_T", "watershed_surfgeo"),
                      ("ref_potential_natural_vegetation", "CLASS_SHOR", "watershed_pnv"), ("ref_road_skid", None, "watershed_skid"),
                      ("ref_road_truck", None, "watershed_truckroad"), ("ref_decks", None, "watershed_decks")]:
    arcpy.analysis.TabulateIntersection(W3, "Ws_name", os.path.join(GDB, fc), os.path.join(GDB, name), fld)
ws_ha = {r[0]: r[1] / 1e4 for r in arcpy.da.SearchCursor(W3, ["Ws_name", "SHAPE@AREA"])}

# 5. documentation
qc = to_df(os.path.join(GDB, "meta_qc_log"))
qc = pd.concat([qc, pd.DataFrame([
    ("ref_ten_four_project_plots", "Latitude/Longitude attributes are not coordinates (e.g., 10, 77); geometry CRS 'assumed'", "not used"),
    ("ref_ws4biomass", "Biomass units and year not in file; likely WS4 biomass survey 1990/1993 (WVU Fernow LTREB data page)", "OPEN: confirm units"),
    ("ref_ws4_soils_1997", "Suffix 0/1 = two sampling depths; depth definitions not in file", "OPEN: confirm depths (metadata)")],
    columns=qc.columns)])
write_df(qc, "meta_qc_log")

# ---- printouts ----
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
print("PLOT DISTURBANCE\n", dt.round(1))
print("\nPLOT HISTORY/GEOLOGY\n", hist[[c for c in hist.columns if c.endswith(("COVER_TYPE", "SURF_GEO_T", "CLASS_SHOR"))]])
print("\nPRIOR-STUDY LINKS (WS4 plots)\n", prior.loc[pts.watershed == "WS4",
      ["ws4bio_dist_m", "ws4bio_Biomass", "ws4bio_n_within_50m", "ws4bio_Biomass_mean_50m",
       "ws4soil97_dist_m", "ws4soil97_Plot", "ws4soil97_phw0", "ws4soil97_c0",
       "gilliam_dist_m", "gilliam_ID", "ltreb_dist_m", "ltreb_Name"]])
print("\nWS4Biomass values:", to_df(os.path.join(GDB, "ref_ws4biomass"))["Biomass"].describe().round(1).to_dict())
print("\nwatershed areas (ha):", {k: round(v, 1) for k, v in ws_ha.items()})
for name in ["watershed_cover1932", "watershed_surfgeo", "watershed_pnv", "watershed_skid", "watershed_truckroad", "watershed_decks"]:
    print(f"\n{name}\n", to_df(os.path.join(GDB, name)).round(1).to_string(index=False, max_colwidth=45))

plot_disturbance: 18 rows, 8 fields
plot_history_geology: 18 rows, 29 fields
plot_prior_studies: 18 rows, 39 fields


<class 'arcgisscripting.ExecuteError'>: Failed to execute. Parameters are not valid.
ERROR 000824: The tool is not licensed.
Failed to execute (TabulateIntersection).


In [31]:
# 4. watershed-level composition without Tabulate Intersection (Basic license)
if arcpy.Exists("w3"): arcpy.management.Delete("w3")
W3 = arcpy.management.MakeFeatureLayer(os.path.join(GDB, "ref_watersheds"), "w3", "Ws_name IN ('WS2','WS4','WS5')")
ws_ha = {r[0]: r[1] / 1e4 for r in arcpy.da.SearchCursor(W3, ["Ws_name", "SHAPE@AREA"])}

def ws_intersect(fc):
    out = os.path.join(GDB, "tmp_wsx")
    if arcpy.Exists(out): arcpy.management.Delete(out)
    arcpy.analysis.PairwiseIntersect([W3, os.path.join(GDB, fc)], out)
    return out

for fc, fld, name in [("ref_cover_type_1932", "COVER_TYPE", "watershed_cover1932"),
                      ("ref_surficial_geology", "SURF_GEO_T", "watershed_surfgeo"),
                      ("ref_potential_natural_vegetation", "CLASS_SHOR", "watershed_pnv")]:
    x = ws_intersect(fc)
    d = pd.DataFrame(arcpy.da.SearchCursor(x, ["Ws_name", fld, "SHAPE@AREA"]), columns=["watershed", "class", "area_m2"])
    t = d.groupby(["watershed", "class"], as_index=False)["area_m2"].sum()
    t["area_ha"] = (t["area_m2"] / 1e4).round(2)
    t["pct_of_watershed"] = (100 * t["area_ha"] / t["watershed"].map(ws_ha)).round(1)
    write_df(t.drop(columns="area_m2"), name)

rows = []
for fc, kind in [("ref_road_skid", "skid_trail"), ("ref_road_truck", "truck_road"), ("ref_decks", "log_deck")]:
    x = ws_intersect(fc)
    geo = "SHAPE@AREA" if kind == "log_deck" else "SHAPE@LENGTH"
    d = pd.DataFrame(arcpy.da.SearchCursor(x, ["Ws_name", geo]), columns=["watershed", "v"]).groupby("watershed")["v"].agg(["sum", "count"])
    for w in ws_ha:
        s, n = (d.loc[w, "sum"], d.loc[w, "count"]) if w in d.index else (0.0, 0)
        rows.append((w, kind, int(n), round(float(s), 1), round(float(s) / ws_ha[w], 2)))
arcpy.management.Delete(os.path.join(GDB, "tmp_wsx"))
write_df(pd.DataFrame(rows, columns=["watershed", "feature", "n_pieces", "total_m_or_m2", "per_ha"]), "watershed_harvest_infrastructure")

# 5. documentation
qc = to_df(os.path.join(GDB, "meta_qc_log"))
qc = pd.concat([qc, pd.DataFrame([
    ("ref_ten_four_project_plots", "Latitude/Longitude attributes are not coordinates (e.g., 10, 77); geometry CRS 'assumed'", "not used"),
    ("ref_ws4biomass", "Biomass units and year not in file; likely WS4 biomass survey 1990/1993 (WVU Fernow LTREB data page)", "OPEN: confirm units"),
    ("ref_ws4_soils_1997", "Suffix 0/1 = two sampling depths; depth definitions not in file", "OPEN: confirm depths (metadata)")],
    columns=qc.columns)])
write_df(qc, "meta_qc_log")

# ---- printouts ----
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
print("PLOT DISTURBANCE\n", dt.round(1))
print("\nPLOT HISTORY/GEOLOGY\n", hist[[c for c in hist.columns if c.endswith(("COVER_TYPE", "SURF_GEO_T", "CLASS_SHOR"))]])
print("\nPRIOR-STUDY LINKS (WS4 plots)\n", prior.loc[pts.watershed == "WS4",
      ["ws4bio_dist_m", "ws4bio_Biomass", "ws4bio_n_within_50m", "ws4bio_Biomass_mean_50m",
       "ws4soil97_dist_m", "ws4soil97_Plot", "ws4soil97_phw0", "ws4soil97_c0",
       "gilliam_dist_m", "gilliam_ID", "ltreb_dist_m", "ltreb_Name"]])
print("\nWS4Biomass values:", to_df(os.path.join(GDB, "ref_ws4biomass"))["Biomass"].describe().round(1).to_dict())
print("\nwatershed areas (ha):", {k: round(v, 1) for k, v in ws_ha.items()})
for name in ["watershed_cover1932", "watershed_surfgeo", "watershed_pnv", "watershed_harvest_infrastructure"]:
    print(f"\n{name}\n", to_df(os.path.join(GDB, name)).to_string(index=False, max_colwidth=45))

watershed_cover1932: 13 rows, 4 fields
watershed_surfgeo: 10 rows, 4 fields
watershed_pnv: 10 rows, 4 fields
watershed_harvest_infrastructure: 9 rows, 5 fields
meta_qc_log: 19 rows, 3 fields
PLOT DISTURBANCE
          dist_skid_trail_m  dist_truck_road_m  dist_log_deck_m  skid_m_per_ha_footprint  skid_m_per_ha_25m  skid_m_per_ha_50m  skid_m_per_ha_100m
plot_id                                                                                                                                          
WS2A-1                28.3              227.5            578.7                      0.0                0.0              128.6               156.3
WS2A-3                 7.8              370.6            352.7                    409.7              240.5              154.0               114.9
WS2A-5                21.4              177.2            171.8                      0.0              133.8              121.3               115.1
WS2B-1                19.2              154.3            128.

In [32]:
qc = to_df(os.path.join(GDB, "meta_qc_log"))
qc = pd.concat([qc, pd.DataFrame([
    ("plots_study", "WS4B-1 and WS4C-1 GPS locations lie 0.3 m from LTREB plots GP4B-1 and GP4C-1", "locations independently corroborated"),
    ("plot_disturbance", "Skid trails cross the footprints of WS2A-3 and WS5-4; context for bulk density only", "stored, not modeled"),
    ("ref_potential_natural_vegetation", "Layer does not cover WS2 (99% unclassified)", "not usable for watershed comparisons"),
    ("plot_history_geology", "WS2B-2 falls on channel-way alluvium; all other plots on cobble-boulder diamict veneer", ""),
    ("ref_ws4biomass", "Grid mean 321 (range 106-595); consistent with corrected reference AGB if units are Mg/ha", "OPEN: confirm units")],
    columns=qc.columns)])
write_df(qc, "meta_qc_log")

exp = os.path.join(PROJ, "gdb_export"); os.makedirs(exp, exist_ok=True)
arcpy.env.workspace = GDB
for n in arcpy.ListFeatureClasses() + arcpy.ListTables():
    if n.startswith("tmp_"): continue
    d = arcpy.Describe(n)
    is_pt = d.dataType in ("FeatureClass", "FeatureLayer") and d.shapeType == "Point"
    to_df(n, xy=is_pt).to_csv(os.path.join(exp, f"{n}.csv"), index=False)
print(len(os.listdir(exp)), "CSVs in", exp)

meta_qc_log: 24 rows, 3 fields
61 CSVs in C:\Users\DCifelli2\OneDrive - mercercounty.org\Documents\ArcGIS\Projects\Fernow-FM-SOM-DC-092026\gdb_export


In [33]:
import requests, json
print("ArcGIS Pro version:", arcpy.GetInstallInfo()["Version"])

# bounding box of the three watersheds + 300 m, in lon/lat
arcpy.analysis.PairwiseBuffer(W3 if arcpy.Exists("w3") else os.path.join(GDB, "ref_watersheds"), r"memory\wsb", "300 Meters", dissolve_option="ALL")
ext = arcpy.Describe(r"memory\wsb").extent.projectAs(arcpy.SpatialReference(4326))
bbox = f"{ext.XMin:.5f},{ext.YMin:.5f},{ext.XMax:.5f},{ext.YMax:.5f}"
print("bbox:", bbox)

r = requests.get("https://tnmaccess.nationalmap.gov/api/v1/products",
                 params={"datasets": "Lidar Point Cloud (LPC)", "bbox": bbox, "max": 200, "outputFormat": "JSON"}, timeout=120)
r.raise_for_status()
items = r.json().get("items", [])
def laz_url(it):
    for k in ("downloadLazURL", "downloadURL"):
        u = it.get(k)
        if isinstance(u, str) and u.lower().endswith(".laz"): return u
    for v in (it.get("urls") or {}).values():
        if isinstance(v, str) and v.lower().endswith(".laz"): return v
    return None
lpc = pd.DataFrame([{"title": it.get("title"), "published": it.get("publicationDate"),
                     "size_MB": round((it.get("sizeInBytes") or 0) / 1e6, 1), "format": it.get("format"),
                     "laz_url": laz_url(it), "meta_url": it.get("metaUrl")} for it in items])
lpc["project"] = lpc["laz_url"].str.extract(r"/Projects/([^/]+)/")[0]
print(f"{len(lpc)} tiles returned; with .laz links: {lpc['laz_url'].notna().sum()}")
print(lpc.groupby("project", dropna=False).agg(tiles=("title", "size"), MB=("size_MB", "sum"),
                                               first_pub=("published", "min")).to_string())
lpc.to_csv(os.path.join(PROJ, "lidar_tile_list.csv"), index=False)

ArcGIS Pro version: 3.4
bbox: -79.69903,39.04276,-79.67706,39.06400
10 tiles returned; with .laz links: 10
                    tiles     MB   first_pub
project                                     
WV_FEMAHQ_2018_D18     10  471.6  2020-12-10


In [34]:
# 1. collection dates from each tile's ScienceBase record
dates = []
for mu in lpc["meta_url"].dropna().unique()[:3]:
    try:
        j = requests.get(mu, params={"format": "json"}, timeout=60).json()
        dates.append((j.get("title", "")[-30:], [(d.get("type"), d.get("dateString")) for d in j.get("dates", [])]))
    except Exception as e:
        dates.append((mu, f"failed: {e}"))
for t, d in dates: print(t, d)

# 2. download tiles (outside OneDrive; skips files already present)
LAS_DIR = os.path.join(os.path.expanduser("~"), "fernow_lidar_2018")
os.makedirs(LAS_DIR, exist_ok=True)
for u in lpc["laz_url"].dropna():
    fn = os.path.join(LAS_DIR, u.split("/")[-1])
    if os.path.exists(fn) and os.path.getsize(fn) > 0: continue
    with requests.get(u, stream=True, timeout=900) as resp:
        resp.raise_for_status()
        with open(fn + ".part", "wb") as f:
            for chunk in resp.iter_content(1 << 20): f.write(chunk)
    os.replace(fn + ".part", fn)
    print("downloaded", os.path.basename(fn), round(os.path.getsize(fn) / 1e6, 1), "MB")
print(len([f for f in os.listdir(LAS_DIR) if f.endswith(".laz")]), "tiles in", LAS_DIR)

# 3. LAS dataset + summary (Pro 3.4 reads .laz directly)
LASD = os.path.join(PROJ, "fernow_lidar_2018.lasd")
arcpy.management.CreateLasDataset(LAS_DIR, LASD, folder_recursion="NO_RECURSION", compute_stats="COMPUTE_STATS")
d = arcpy.Describe(LASD)
print("files:", d.fileCount, "| points:", d.pointCount, "| CRS:", d.spatialReference.name, "| units:", d.spatialReference.linearUnitName)
stats_csv = os.path.join(PROJ, "lidar_stats.csv")
arcpy.management.LasDatasetStatistics(LASD, "OVERWRITE_EXISTING_STATS", stats_csv, "DATASET")
st = pd.read_csv(stats_csv)
print(st[st.iloc[:, 0].astype(str).str.contains("Class|Return", case=False)].to_string(index=False))

 WV_FEMAHQ_2018_D18 e1390n1899 [('Publication', '2020-12-10'), ('Start', '2018-11-08'), ('End', '2019-05-01'), ('Info', '2021-06-30 06:42:22')]
 WV_FEMAHQ_2018_D18 e1390n1900 [('Publication', '2020-12-10'), ('Start', '2018-11-08'), ('End', '2019-05-01'), ('Info', '2021-06-30 06:42:22')]
 WV_FEMAHQ_2018_D18 e1390n1901 [('Publication', '2020-12-10'), ('Start', '2018-11-08'), ('End', '2019-05-01'), ('Info', '2021-06-30 06:42:21')]
downloaded USGS_LPC_WV_FEMAHQ_2018_D18_e1390n1899.laz 56.7 MB
downloaded USGS_LPC_WV_FEMAHQ_2018_D18_e1390n1900.laz 52.6 MB
downloaded USGS_LPC_WV_FEMAHQ_2018_D18_e1390n1901.laz 38.1 MB
downloaded USGS_LPC_WV_FEMAHQ_2018_D18_e1391n1899.laz 41.5 MB
downloaded USGS_LPC_WV_FEMAHQ_2018_D18_e1391n1900.laz 33.2 MB
downloaded USGS_LPC_WV_FEMAHQ_2018_D18_e1391n1901.laz 41.9 MB
downloaded USGS_LPC_WV_FEMAHQ_2018_D18_e1392n1899.laz 47.6 MB
downloaded USGS_LPC_WV_FEMAHQ_2018_D18_e1392n1900.laz 59.2 MB
downloaded USGS_LPC_WV_FEMAHQ_2018_D18_e1392n1901.laz 56.9 MB
downloaded

In [1]:
laz = [os.path.join(LAS_DIR, f) for f in sorted(os.listdir(LAS_DIR)) if f.lower().endswith(".laz")]
print(len(laz), "laz files;", round(sum(os.path.getsize(f) for f in laz) / 1e6, 1), "MB total")

LASD = os.path.join(PROJ, "fernow_lidar_2018.lasd")
if arcpy.Exists(LASD): arcpy.management.Delete(LASD)

def describe_lasd():
    d = arcpy.Describe(LASD)
    print(f"   -> files: {d.fileCount} | points: {d.pointCount} | CRS: {d.spatialReference.name}")
    return d.fileCount > 0

ok = False
# A: explicit file list
try:
    arcpy.management.CreateLasDataset(laz, LASD, compute_stats="COMPUTE_STATS")
    ok = describe_lasd()
except Exception as e:
    print("A failed:", str(e)[:150])

# B: folder + explicit CRS (the .laz headers may not carry one)
if not ok:
    if arcpy.Exists(LASD): arcpy.management.Delete(LASD)
    try:
        arcpy.management.CreateLasDataset(LAS_DIR, LASD, folder_recursion="NO_RECURSION",
                                          spatial_reference=arcpy.SpatialReference(26917), compute_stats="COMPUTE_STATS")
        ok = describe_lasd()
    except Exception as e:
        print("B failed:", str(e)[:150])

# C: convert .laz -> .las, then build from those
if not ok:
    LAS_OUT = os.path.join(os.path.expanduser("~"), "fernow_lidar_2018_las")
    os.makedirs(LAS_OUT, exist_ok=True)
    print("converting to .las (several minutes)...")
    arcpy.conversion.ConvertLas(laz, LAS_OUT, "SAME_AS_INPUT", "NO_COMPRESSION")
    if arcpy.Exists(LASD): arcpy.management.Delete(LASD)
    arcpy.management.CreateLasDataset(LAS_OUT, LASD, folder_recursion="NO_RECURSION", compute_stats="COMPUTE_STATS")
    ok = describe_lasd()

if ok:
    d = arcpy.Describe(LASD)
    print("\nCRS:", d.spatialReference.name, "| XY units:", d.spatialReference.linearUnitName,
          "| Z units:", d.spatialReference.VCS.linearUnitName if d.spatialReference.VCS else "unknown")
    stats_csv = os.path.join(PROJ, "lidar_stats.csv")
    arcpy.management.LasDatasetStatistics(LASD, "OVERWRITE_EXISTING_STATS", stats_csv, "DATASET")
    print(pd.read_csv(stats_csv).to_string(index=False, max_colwidth=40))

<class 'NameError'>: name 'os' is not defined

In [4]:
LAS_DIR = os.path.join(os.path.expanduser("~"), "fernow_lidar_2018")
laz = [os.path.join(LAS_DIR, f) for f in sorted(os.listdir(LAS_DIR)) if f.lower().endswith(".laz")]
print(len(laz), "laz files;", round(sum(os.path.getsize(f) for f in laz) / 1e6, 1), "MB total")

LASD = os.path.join(PROJ, "fernow_lidar_2018.lasd")
if arcpy.Exists(LASD): arcpy.management.Delete(LASD)

def describe_lasd():
    d = arcpy.Describe(LASD)
    print(f"   -> files: {d.fileCount} | points: {d.pointCount} | CRS: {d.spatialReference.name}")
    return d.fileCount > 0

ok = False
# A: explicit file list
try:
    arcpy.management.CreateLasDataset(laz, LASD, compute_stats="COMPUTE_STATS")
    ok = describe_lasd()
except Exception as e:
    print("A failed:", str(e)[:150])

# B: folder + explicit CRS
if not ok:
    if arcpy.Exists(LASD): arcpy.management.Delete(LASD)
    try:
        arcpy.management.CreateLasDataset(LAS_DIR, LASD, folder_recursion="NO_RECURSION",
                                          spatial_reference=arcpy.SpatialReference(26917), compute_stats="COMPUTE_STATS")
        ok = describe_lasd()
    except Exception as e:
        print("B failed:", str(e)[:150])

# C: convert .laz -> .las, then build
if not ok:
    LAS_OUT = os.path.join(os.path.expanduser("~"), "fernow_lidar_2018_las")
    os.makedirs(LAS_OUT, exist_ok=True)
    print("converting to .las (several minutes)...")
    arcpy.conversion.ConvertLas(laz, LAS_OUT, "SAME_AS_INPUT", "NO_COMPRESSION")
    if arcpy.Exists(LASD): arcpy.management.Delete(LASD)
    arcpy.management.CreateLasDataset(LAS_OUT, LASD, folder_recursion="NO_RECURSION", compute_stats="COMPUTE_STATS")
    ok = describe_lasd()

if ok:
    d = arcpy.Describe(LASD)
    print("\nCRS:", d.spatialReference.name, "| XY units:", d.spatialReference.linearUnitName,
          "| Z units:", d.spatialReference.VCS.linearUnitName if d.spatialReference.VCS else "unknown")
    stats_csv = os.path.join(PROJ, "lidar_stats.csv")
    arcpy.management.LasDatasetStatistics(LASD, "OVERWRITE_EXISTING_STATS", stats_csv, "DATASET")
    print(pd.read_csv(stats_csv).to_string(index=False, max_colwidth=40))

10 laz files; 471.5 MB total


<class 'NameError'>: name 'PROJ' is not defined

In [5]:
import arcpy, os, re, numpy as np, pandas as pd
arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False
aprx = arcpy.mp.ArcGISProject("CURRENT")
PROJ = os.path.dirname(aprx.filePath)
WORK = aprx.defaultGeodatabase
SHP_ROOT = r"C:\Users\DCifelli2\OneDrive - mercercounty.org\Desktop\FernowShapeFiles"
GDB = os.path.join(PROJ, "Fernow_SOM_master.gdb")
SR = arcpy.SpatialReference(26917)
TRT = {"WS2": "Diameter-limit", "WS4": "Reference", "WS5": "Single-tree selection"}
P = os.path.join(GDB, "plots_study")
LAS_DIR = os.path.join(os.path.expanduser("~"), "fernow_lidar_2018")

def find_shp(name):
    hits = [os.path.join(r, name + ".shp") for r, _, f in os.walk(SHP_ROOT) if name + ".shp" in f]
    return hits[0] if hits else None

def to_df(fc, xy=False):
    f = [x.name for x in arcpy.ListFields(fc) if x.type not in ("Geometry", "OID", "Blob", "Raster", "GlobalID")]
    q = f + (["SHAPE@X", "SHAPE@Y"] if xy else [])
    return pd.DataFrame(arcpy.da.SearchCursor(fc, q), columns=q)

def _ftype(s):
    if pd.api.types.is_bool_dtype(s): return "SHORT"
    if pd.api.types.is_integer_dtype(s): return "LONG"
    if pd.api.types.is_float_dtype(s): return "DOUBLE"
    return "TEXT"

def write_df(df, name, xy=None):
    out = os.path.join(GDB, name)
    if arcpy.Exists(out): arcpy.management.Delete(out)
    if xy: arcpy.management.CreateFeatureclass(GDB, name, "POINT", spatial_reference=SR)
    else:  arcpy.management.CreateTable(GDB, name)
    cols, names = list(df.columns), []
    for c in cols:
        fn, ft = arcpy.ValidateFieldName(c, GDB), _ftype(df[c])
        L = max(16, int(df[c].astype(str).str.len().max()) + 8) if ft == "TEXT" else None
        arcpy.management.AddField(out, fn, ft, field_length=L); names.append(fn)
    def clean(v):
        if v is None or (not isinstance(v, str) and pd.isna(v)): return None
        if isinstance(v, (bool, np.bool_)): return int(v)
        return v.item() if isinstance(v, np.generic) else v
    with arcpy.da.InsertCursor(out, names + (["SHAPE@XY"] if xy else [])) as cur:
        for _, r in df.iterrows():
            row = [clean(r[c]) for c in cols]
            if xy: row.append((float(r[xy[0]]), float(r[xy[1]])))
            cur.insertRow(row)
    print(f"{name}: {arcpy.management.GetCount(out)[0]} rows, {len(names)} fields")

print("gdb exists:", arcpy.Exists(GDB), "| plots:", arcpy.management.GetCount(P)[0] if arcpy.Exists(P) else "n/a",
      "| laz files:", len([f for f in os.listdir(LAS_DIR) if f.lower().endswith(".laz")]))

gdb exists: True | plots: 18 | laz files: 10


In [6]:
laz = [os.path.join(LAS_DIR, f) for f in sorted(os.listdir(LAS_DIR)) if f.lower().endswith(".laz")]
LASD = os.path.join(PROJ, "fernow_lidar_2018.lasd")
if arcpy.Exists(LASD): arcpy.management.Delete(LASD)

def describe_lasd():
    d = arcpy.Describe(LASD)
    print(f"   -> files: {d.fileCount} | points: {d.pointCount} | CRS: {d.spatialReference.name}")
    return d.fileCount > 0

ok = False
try:                                    # A: explicit file list
    arcpy.management.CreateLasDataset(laz, LASD, compute_stats="COMPUTE_STATS")
    ok = describe_lasd()
except Exception as e:
    print("A failed:", str(e)[:150])

if not ok:                              # B: folder + explicit CRS
    if arcpy.Exists(LASD): arcpy.management.Delete(LASD)
    try:
        arcpy.management.CreateLasDataset(LAS_DIR, LASD, folder_recursion="NO_RECURSION",
                                          spatial_reference=SR, compute_stats="COMPUTE_STATS")
        ok = describe_lasd()
    except Exception as e:
        print("B failed:", str(e)[:150])

if not ok:                              # C: convert .laz -> .las
    LAS_OUT = os.path.join(os.path.expanduser("~"), "fernow_lidar_2018_las")
    os.makedirs(LAS_OUT, exist_ok=True)
    print("converting to .las (several minutes)...")
    arcpy.conversion.ConvertLas(laz, LAS_OUT, "SAME_AS_INPUT", "NO_COMPRESSION")
    if arcpy.Exists(LASD): arcpy.management.Delete(LASD)
    arcpy.management.CreateLasDataset(LAS_OUT, LASD, folder_recursion="NO_RECURSION", compute_stats="COMPUTE_STATS")
    ok = describe_lasd()

if ok:
    d = arcpy.Describe(LASD)
    print("\nCRS:", d.spatialReference.name, "| XY units:", d.spatialReference.linearUnitName,
          "| Z units:", d.spatialReference.VCS.linearUnitName if d.spatialReference.VCS else "unknown")
    stats_csv = os.path.join(PROJ, "lidar_stats.csv")
    arcpy.management.LasDatasetStatistics(LASD, "OVERWRITE_EXISTING_STATS", stats_csv, "DATASET")
    print(pd.read_csv(stats_csv).to_string(index=False, max_colwidth=40))

A failed: Failed to execute. Parameters are not valid.
ERROR 000814: Invalid file type
ERROR 000814: Invalid file type
ERROR 000814: Invalid file type
ERROR 000
   -> files: 0 | points: 0 | CRS: NAD_1983_UTM_Zone_17N
converting to .las (several minutes)...


<class 'RuntimeError'>: Object: Error in executing tool

In [7]:
print(f"{'file':<45} {'MB':>7}  header")
for f in sorted(os.listdir(LAS_DIR)):
    p = os.path.join(LAS_DIR, f)
    with open(p, "rb") as fh: head = fh.read(8)
    sig = head[:4].decode("latin1")
    print(f"{f:<45} {os.path.getsize(p)/1e6:>7.1f}  {sig!r} {'OK' if sig == 'LASF' else '<-- NOT A LAS/LAZ FILE'}")
    if sig != "LASF":
        with open(p, "rb") as fh: print("      starts with:", fh.read(200))

file                                               MB  header
USGS_LPC_WV_FEMAHQ_2018_D18_e1390n1899.laz       56.7  'LASF' OK
USGS_LPC_WV_FEMAHQ_2018_D18_e1390n1900.laz       52.6  'LASF' OK
USGS_LPC_WV_FEMAHQ_2018_D18_e1390n1901.laz       38.1  'LASF' OK
USGS_LPC_WV_FEMAHQ_2018_D18_e1391n1899.laz       41.5  'LASF' OK
USGS_LPC_WV_FEMAHQ_2018_D18_e1391n1900.laz       33.2  'LASF' OK
USGS_LPC_WV_FEMAHQ_2018_D18_e1391n1901.laz       41.9  'LASF' OK
USGS_LPC_WV_FEMAHQ_2018_D18_e1392n1899.laz       47.6  'LASF' OK
USGS_LPC_WV_FEMAHQ_2018_D18_e1392n1900.laz       59.2  'LASF' OK
USGS_LPC_WV_FEMAHQ_2018_D18_e1392n1901.laz       56.9  'LASF' OK
USGS_LPC_WV_FEMAHQ_2018_D18_e1392n1902.laz       43.9  'LASF' OK


In [8]:
import subprocess, sys
py = os.path.join(sys.prefix, "python.exe")
r = subprocess.run([py, "-m", "pip", "install", "--user", "laspy[lazrs]"], capture_output=True, text=True)
print(r.stdout[-400:], r.stderr[-400:])

8.2-cp311-cp311-win_amd64.whl.metadata (99 bytes)
   ---------------------------------------- 436.9/436.9 kB 3.9 MB/s eta 0:00:00
   ---------------------------------------- 86.1/86.1 kB ? eta 0:00:00
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.



In [9]:
import importlib, laspy
importlib.reload(laspy)
LAS_OUT = os.path.join(os.path.expanduser("~"), "fernow_lidar_2018_las")
os.makedirs(LAS_OUT, exist_ok=True)

for f in sorted(os.listdir(LAS_DIR)):
    if not f.lower().endswith(".laz"): continue
    dst = os.path.join(LAS_OUT, f[:-4] + ".las")
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        print("have", os.path.basename(dst)); continue
    with laspy.open(os.path.join(LAS_DIR, f)) as src:
        with laspy.open(dst + ".part", mode="w", header=src.header) as out:
            for pts in src.chunk_iterator(2_000_000): out.append_points(pts)
        n, cls = src.header.point_count, src.header.point_format.id
    os.replace(dst + ".part", dst)
    print(f"{f} -> {n:,} points, format {cls}, {os.path.getsize(dst)/1e6:.0f} MB", flush=True)

LASD = os.path.join(PROJ, "fernow_lidar_2018.lasd")
if arcpy.Exists(LASD): arcpy.management.Delete(LASD)
arcpy.management.CreateLasDataset(LAS_OUT, LASD, folder_recursion="NO_RECURSION", compute_stats="COMPUTE_STATS")
d = arcpy.Describe(LASD)
print("\nfiles:", d.fileCount, "| points:", d.pointCount, "| CRS:", d.spatialReference.name,
      "| XY units:", d.spatialReference.linearUnitName,
      "| Z units:", d.spatialReference.VCS.linearUnitName if d.spatialReference.VCS else "unknown")
stats_csv = os.path.join(PROJ, "lidar_stats.csv")
arcpy.management.LasDatasetStatistics(LASD, "OVERWRITE_EXISTING_STATS", stats_csv, "DATASET")
print(pd.read_csv(stats_csv).to_string(index=False, max_colwidth=40))

<class 'ModuleNotFoundError'>: No module named 'laspy'

In [10]:
import sys, site, os, importlib
user_site = site.getusersitepackages()
user_site = user_site if isinstance(user_site, str) else user_site[0]
print("user site-packages:", user_site, "| exists:", os.path.isdir(user_site))
if user_site not in sys.path: sys.path.insert(0, user_site)
importlib.invalidate_caches()
import laspy
print("laspy", laspy.__version__, "|", laspy.__file__)

user site-packages: C:\Users\DCifelli2\AppData\Roaming\Python\Python311\site-packages | exists: True
laspy 2.7.0 | C:\Users\DCifelli2\AppData\Roaming\Python\Python311\site-packages\laspy\__init__.py


In [11]:
LAS_OUT = os.path.join(os.path.expanduser("~"), "fernow_lidar_2018_las")
os.makedirs(LAS_OUT, exist_ok=True)

for f in sorted(os.listdir(LAS_DIR)):
    if not f.lower().endswith(".laz"): continue
    dst = os.path.join(LAS_OUT, f[:-4] + ".las")
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        print("have", os.path.basename(dst)); continue
    with laspy.open(os.path.join(LAS_DIR, f)) as src:
        n, fmt = src.header.point_count, src.header.point_format.id
        with laspy.open(dst + ".part", mode="w", header=src.header) as out:
            for pts in src.chunk_iterator(2_000_000): out.append_points(pts)
    os.replace(dst + ".part", dst)
    print(f"{f} -> {n:,} points, format {fmt}, {os.path.getsize(dst)/1e6:.0f} MB", flush=True)

LASD = os.path.join(PROJ, "fernow_lidar_2018.lasd")
if arcpy.Exists(LASD): arcpy.management.Delete(LASD)
arcpy.management.CreateLasDataset(LAS_OUT, LASD, folder_recursion="NO_RECURSION", compute_stats="COMPUTE_STATS")
d = arcpy.Describe(LASD)
print("\nfiles:", d.fileCount, "| points:", d.pointCount, "| CRS:", d.spatialReference.name,
      "| XY units:", d.spatialReference.linearUnitName,
      "| Z units:", d.spatialReference.VCS.linearUnitName if d.spatialReference.VCS else "unknown")
stats_csv = os.path.join(PROJ, "lidar_stats.csv")
arcpy.management.LasDatasetStatistics(LASD, "OVERWRITE_EXISTING_STATS", stats_csv, "DATASET")
print(pd.read_csv(stats_csv).to_string(index=False, max_colwidth=40))

<class 'AttributeError'>: 'LasWriter' object has no attribute 'append_points'

In [12]:
LAS_OUT = os.path.join(os.path.expanduser("~"), "fernow_lidar_2018_las")
os.makedirs(LAS_OUT, exist_ok=True)

for f in sorted(os.listdir(LAS_DIR)):
    if not f.lower().endswith(".laz"): continue
    dst = os.path.join(LAS_OUT, f[:-4] + ".las")
    if os.path.exists(dst) and os.path.getsize(dst) > 0:
        print("have", os.path.basename(dst)); continue
    with laspy.open(os.path.join(LAS_DIR, f)) as src:
        n, fmt = src.header.point_count, src.header.point_format.id
        with laspy.open(dst + ".part", mode="w", header=src.header) as out:
            for pts in src.chunk_iterator(2_000_000):
                out.write_points(pts)
    os.replace(dst + ".part", dst)
    print(f"{f} -> {n:,} points, format {fmt}, {os.path.getsize(dst)/1e6:.0f} MB", flush=True)

LASD = os.path.join(PROJ, "fernow_lidar_2018.lasd")
if arcpy.Exists(LASD): arcpy.management.Delete(LASD)
arcpy.management.CreateLasDataset(LAS_OUT, LASD, folder_recursion="NO_RECURSION", compute_stats="COMPUTE_STATS")
d = arcpy.Describe(LASD)
print("\nfiles:", d.fileCount, "| points:", d.pointCount, "| CRS:", d.spatialReference.name,
      "| XY units:", d.spatialReference.linearUnitName,
      "| Z units:", d.spatialReference.VCS.linearUnitName if d.spatialReference.VCS else "unknown")
stats_csv = os.path.join(PROJ, "lidar_stats.csv")
arcpy.management.LasDatasetStatistics(LASD, "OVERWRITE_EXISTING_STATS", stats_csv, "DATASET")
print(pd.read_csv(stats_csv).to_string(index=False, max_colwidth=40))

USGS_LPC_WV_FEMAHQ_2018_D18_e1390n1899.laz -> 9,453,773 points, format 6, 284 MB
USGS_LPC_WV_FEMAHQ_2018_D18_e1390n1900.laz -> 8,886,276 points, format 6, 267 MB
USGS_LPC_WV_FEMAHQ_2018_D18_e1390n1901.laz -> 6,744,350 points, format 6, 202 MB
USGS_LPC_WV_FEMAHQ_2018_D18_e1391n1899.laz -> 7,918,896 points, format 6, 238 MB
USGS_LPC_WV_FEMAHQ_2018_D18_e1391n1900.laz -> 6,854,949 points, format 6, 206 MB
USGS_LPC_WV_FEMAHQ_2018_D18_e1391n1901.laz -> 7,901,309 points, format 6, 237 MB
USGS_LPC_WV_FEMAHQ_2018_D18_e1392n1899.laz -> 8,838,144 points, format 6, 265 MB
USGS_LPC_WV_FEMAHQ_2018_D18_e1392n1900.laz -> 10,066,966 points, format 6, 302 MB
USGS_LPC_WV_FEMAHQ_2018_D18_e1392n1901.laz -> 9,705,438 points, format 6, 291 MB
USGS_LPC_WV_FEMAHQ_2018_D18_e1392n1902.laz -> 8,322,869 points, format 6, 250 MB

files: 10 | points: 84692970 | CRS: Unknown | XY units: ﻿ | Z units: unknown


<class 'arcgisscripting.ExecuteError'>: Failed to execute. Parameters are not valid.
ERROR 000258: Output s already exists
Failed to execute (LasDatasetStatistics).


In [13]:
# 1. what CRS do the files actually declare?
with laspy.open(os.path.join(LAS_DIR, sorted(os.listdir(LAS_DIR))[0])) as src:
    print("point count:", src.header.point_count, "| scales:", src.header.scales, "| offsets:", src.header.offsets)
    try:
        crs = src.header.parse_crs()
        print("declared CRS:", crs.name if crs else None, "| EPSG:", crs.to_epsg() if crs else None)
        print("vertical unit:", crs.axis_info[2].unit_name if crs and len(crs.axis_info) > 2 else "not stated")
    except Exception as e:
        print("could not parse CRS:", e)

# 2. rebuild the LAS dataset with the CRS stated explicitly
LASD = os.path.join(PROJ, "fernow_lidar_2018.lasd")
if arcpy.Exists(LASD): arcpy.management.Delete(LASD)
arcpy.management.CreateLasDataset(LAS_OUT, LASD, folder_recursion="NO_RECURSION",
                                  spatial_reference=SR, compute_stats="COMPUTE_STATS")
d = arcpy.Describe(LASD)
print("\nfiles:", d.fileCount, "| points:", d.pointCount, "| CRS:", d.spatialReference.name,
      "| XY units:", d.spatialReference.linearUnitName)

# 3. statistics to a fresh file
stats_csv = os.path.join(PROJ, "lidar_stats_v2.csv")
if os.path.exists(stats_csv): os.remove(stats_csv)
arcpy.management.LasDatasetStatistics(LASD, "OVERWRITE_EXISTING_STATS", stats_csv, "DATASET")
print(pd.read_csv(stats_csv).to_string(index=False, max_colwidth=40))

# 4. sanity check: do point elevations match the DEM? (catches feet-vs-metres)
ext = arcpy.Describe(P).extent
print("\nplot extent Z check — DEM range at plots:",
      to_df(os.path.join(GDB, "plot_terrain"))[["dem5_fp"]].agg(["min", "max"]).round(0).to_dict())

point count: 9453773 | scales: [0.01 0.01 0.01] | offsets: [-0. -0. -0.]
could not parse CRS: No module named 'pyproj'

files: 10 | points: 84692970 | CRS: NAD_1983_UTM_Zone_17N | XY units: Meter
              Item   Category     Pt_Cnt  Percent  Z_Min   Z_Max  Intensity_Min  Intensity_Max  Synthetic_Pt_Cnt  Range_Min  Range_Max
             First    Returns 39916432.0    47.13 458.85 1145.75            NaN            NaN               NaN        NaN        NaN
            Second    Returns 23824107.0    28.13 458.43 1140.80            NaN            NaN               NaN        NaN        NaN
             Third    Returns  8770739.0    10.36 458.99 1126.26            NaN            NaN               NaN        NaN        NaN
            Fourth    Returns  1984467.0     2.34 458.14 1108.79            NaN            NaN               NaN        NaN        NaN
             Fifth    Returns   278341.0     0.33 459.11 1095.93            NaN            NaN               NaN        NaN      

In [14]:
from arcpy.sa import Con, Raster, ZonalStatisticsAsTable, IsNull, FocalStatistics, NbrRectangle
arcpy.CheckOutExtension("Spatial")
buf = os.path.join(GDB, "ws_buf300")
if not arcpy.Exists(buf):
    arcpy.analysis.PairwiseBuffer(os.path.join(GDB, "ref_watersheds"), buf, "300 Meters", dissolve_option="ALL")
arcpy.env.extent = arcpy.Describe(buf).extent
arcpy.env.cellSize = 1
arcpy.env.snapRaster = None

# canopy surface: highest first return per cell, overlap and noise excluded
first = arcpy.management.MakeLasDatasetLayer(LASD, "las_first", class_code=[1, 2, 9],
                                             return_values=["SINGLE", "FIRST_OF_MANY"], no_flag=True, synthetic=False,
                                             keypoint=True, withheld=False, overlap=False)
arcpy.conversion.LasDatasetToRaster(first, os.path.join(GDB, "lidar_dsm1"), "ELEVATION",
                                    "BINNING MAXIMUM NONE", "FLOAT", "CELLSIZE", 1)
ground = arcpy.management.MakeLasDatasetLayer(LASD, "las_ground", class_code=[2], no_flag=True,
                                              withheld=False, overlap=False)
arcpy.conversion.LasDatasetToRaster(ground, os.path.join(GDB, "lidar_dtm1"), "ELEVATION",
                                    "BINNING AVERAGE LINEAR", "FLOAT", "CELLSIZE", 1)

# fill small DSM gaps (leaf-off returns are sparse), then CHM
dsm = Raster(os.path.join(GDB, "lidar_dsm1"))
dsm_f = Con(IsNull(dsm), FocalStatistics(dsm, NbrRectangle(3, 3, "CELL"), "MAXIMUM", "DATA"), dsm)
chm = dsm_f - Raster(os.path.join(GDB, "lidar_dtm1"))
chm = Con(chm < 0, 0, Con(chm > 55, 55, chm))     # clamp artefacts
chm.save(os.path.join(GDB, "lidar_chm1"))
Con(chm >= 5, 1, 0).save(os.path.join(GDB, "lidar_cover5"))
Con(chm < 2, 1, 0).save(os.path.join(GDB, "lidar_gap2"))
for r in ["lidar_dsm1", "lidar_dtm1", "lidar_chm1"]:
    print(r, arcpy.management.GetRasterProperties(os.path.join(GDB, r), "MINIMUM")[0], "-",
          arcpy.management.GetRasterProperties(os.path.join(GDB, r), "MAXIMUM")[0])
arcpy.env.extent = None; arcpy.env.cellSize = None

# per-plot canopy metrics
def canopy(zones, key):
    ZonalStatisticsAsTable(zones, key, os.path.join(GDB, "lidar_chm1"), r"memory\zc", "DATA", "ALL",
                           percentile_values=[25, 50, 75, 90, 95, 99])
    z = to_df(r"memory\zc")
    keep = [key] + [c for c in z.columns if c in ("MEAN", "MAX", "STD", "MEDIAN", "COUNT") or c.startswith("PCT")]
    out = z[keep].set_index(key).add_prefix("chm_")
    for r, nm in [("lidar_cover5", "frac_ge5m_leafoff"), ("lidar_gap2", "frac_lt2m_leafoff")]:
        ZonalStatisticsAsTable(zones, key, os.path.join(GDB, r), r"memory\zc2", "DATA", "MEAN")
        out = out.join(to_df(r"memory\zc2").set_index(key)["MEAN"].rename(nm))
    return out.reset_index()

pc = canopy(os.path.join(GDB, "plots_footprint"), "plot_id")
write_df(pc, "plot_canopy")
out = os.path.join(GDB, "rel_plots_study_plot_canopy")
if arcpy.Exists(out): arcpy.management.Delete(out)
arcpy.management.CreateRelationshipClass(P, os.path.join(GDB, "plot_canopy"), out, "SIMPLE", "plot_canopy",
                                         "plots_study", "NONE", "ONE_TO_ONE", "NONE", "plot_id", "plot_id")

# all 43 USFS plots + whole watersheds, for context
arcpy.analysis.PairwiseBuffer(os.path.join(GDB, "ref_usfs_growth_plots"), os.path.join(GDB, "tmp_ufp"), "11.35 Meters")
blk = "import re\ndef lbl(s, n):\n    return re.sub(r'[^A-Z0-9]', '', str(s).upper()) + '-' + str(int(n))"
arcpy.management.CalculateField(os.path.join(GDB, "tmp_ufp"), "usfs_id", "lbl(!Scomp_name!, !Plot_num!)", "PYTHON3", blk, field_type="TEXT")
write_df(canopy(os.path.join(GDB, "tmp_ufp"), "usfs_id"), "usfs_plot_canopy")
arcpy.management.Delete(os.path.join(GDB, "tmp_ufp"))
ZonalStatisticsAsTable(os.path.join(GDB, "ref_watersheds"), "Ws_name", os.path.join(GDB, "lidar_chm1"),
                       r"memory\zw", "DATA", "ALL", percentile_values=[50, 90, 95, 99])
write_df(to_df(r"memory\zw"), "watershed_canopy")

# documentation
ms = to_df(os.path.join(GDB, "meta_sources"))
ms = pd.concat([ms, pd.DataFrame([("lidar_chm1 / plot_canopy / usfs_plot_canopy / watershed_canopy",
    "USGS 3DEP project WV_FEMAHQ_2018_D18 (flown 2018-11-08 to 2019-05-01), 10 tiles, 84.7 M points, NAD83 UTM 17N, metres",
    "CHM = max first-return DSM minus ground DTM at 1 m; overlap/withheld/noise excluded; clamped 0-55 m")],
    columns=ms.columns)])
write_df(ms, "meta_sources")
qc = to_df(os.path.join(GDB, "meta_qc_log"))
qc = pd.concat([qc, pd.DataFrame([
    ("plot_canopy", "Lidar flown leaf-off (Nov 2018-May 2019): height metrics reliable; cover/gap fractions biased low", "use heights, not cover"),
    ("plot_canopy", "Lidar predates the 2022 tree inventory by 3-4 yr; post-dates WS5 harvest (~2012)", "OPEN: confirm WS2 last harvest year"),
    ("plot_canopy", "ArcGIS could not read the .laz files; decompressed with laspy 2.7.0 and CRS set explicitly to EPSG 26917", "resolved")],
    columns=qc.columns)])
write_df(qc, "meta_qc_log")

pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
pc2 = pc.set_index("plot_id").join(to_df(P).set_index("plot_id")[["watershed"]])
print("\nPER-PLOT CANOPY\n", pc2[["watershed", "chm_MEAN", "chm_MAX", "chm_MEDIAN", "chm_PCT90", "chm_PCT95", "chm_STD", "frac_ge5m_leafoff"]].round(1).to_string())
print("\nBY WATERSHED (footprint means)\n", pc2.groupby("watershed")[["chm_MEAN", "chm_MAX", "chm_PCT95", "chm_STD"]].mean().round(1))
print("\nWHOLE WATERSHEDS\n", to_df(os.path.join(GDB, "watershed_canopy")).round(1).to_string(index=False))

<class 'arcgisscripting.ExecuteError'>: ERROR 001100: Failed because no statistics are available.
Failed to execute (GetRasterProperties).


In [15]:
for r in ["lidar_dsm1", "lidar_dtm1", "lidar_chm1", "lidar_cover5", "lidar_gap2"]:
    p = os.path.join(GDB, r)
    print(r, "exists:", arcpy.Exists(p))
    if arcpy.Exists(p):
        arcpy.management.CalculateStatistics(p)
        print("   range:", arcpy.management.GetRasterProperties(p, "MINIMUM")[0], "-",
              arcpy.management.GetRasterProperties(p, "MAXIMUM")[0])
arcpy.env.extent = None; arcpy.env.cellSize = None

from arcpy.sa import ZonalStatisticsAsTable
def canopy(zones, key):
    ZonalStatisticsAsTable(zones, key, os.path.join(GDB, "lidar_chm1"), r"memory\zc", "DATA", "ALL",
                           percentile_values=[25, 50, 75, 90, 95, 99])
    z = to_df(r"memory\zc")
    keep = [key] + [c for c in z.columns if c in ("MEAN", "MAX", "STD", "MEDIAN", "COUNT") or c.startswith("PCT")]
    out = z[keep].set_index(key).add_prefix("chm_")
    for r, nm in [("lidar_cover5", "frac_ge5m_leafoff"), ("lidar_gap2", "frac_lt2m_leafoff")]:
        ZonalStatisticsAsTable(zones, key, os.path.join(GDB, r), r"memory\zc2", "DATA", "MEAN")
        out = out.join(to_df(r"memory\zc2").set_index(key)["MEAN"].rename(nm))
    return out.reset_index()

pc = canopy(os.path.join(GDB, "plots_footprint"), "plot_id")
write_df(pc, "plot_canopy")
out = os.path.join(GDB, "rel_plots_study_plot_canopy")
if arcpy.Exists(out): arcpy.management.Delete(out)
arcpy.management.CreateRelationshipClass(P, os.path.join(GDB, "plot_canopy"), out, "SIMPLE", "plot_canopy",
                                         "plots_study", "NONE", "ONE_TO_ONE", "NONE", "plot_id", "plot_id")

arcpy.analysis.PairwiseBuffer(os.path.join(GDB, "ref_usfs_growth_plots"), os.path.join(GDB, "tmp_ufp"), "11.35 Meters")
blk = "import re\ndef lbl(s, n):\n    return re.sub(r'[^A-Z0-9]', '', str(s).upper()) + '-' + str(int(n))"
arcpy.management.CalculateField(os.path.join(GDB, "tmp_ufp"), "usfs_id", "lbl(!Scomp_name!, !Plot_num!)", "PYTHON3", blk, field_type="TEXT")
write_df(canopy(os.path.join(GDB, "tmp_ufp"), "usfs_id"), "usfs_plot_canopy")
arcpy.management.Delete(os.path.join(GDB, "tmp_ufp"))
ZonalStatisticsAsTable(os.path.join(GDB, "ref_watersheds"), "Ws_name", os.path.join(GDB, "lidar_chm1"),
                       r"memory\zw", "DATA", "ALL", percentile_values=[50, 90, 95, 99])
write_df(to_df(r"memory\zw"), "watershed_canopy")

ms = to_df(os.path.join(GDB, "meta_sources"))
ms = pd.concat([ms, pd.DataFrame([("lidar_chm1 / plot_canopy / usfs_plot_canopy / watershed_canopy",
    "USGS 3DEP project WV_FEMAHQ_2018_D18 (flown 2018-11-08 to 2019-05-01), 10 tiles, 84.7 M points, NAD83 UTM 17N, metres",
    "CHM = max first-return DSM minus ground DTM at 1 m; overlap/withheld/noise excluded; clamped 0-55 m")],
    columns=ms.columns)])
write_df(ms, "meta_sources")
qc = to_df(os.path.join(GDB, "meta_qc_log"))
qc = pd.concat([qc, pd.DataFrame([
    ("plot_canopy", "Lidar flown leaf-off (Nov 2018-May 2019): height metrics reliable; cover/gap fractions biased low", "use heights, not cover"),
    ("plot_canopy", "Lidar predates the 2022 tree inventory by 3-4 yr; post-dates WS5 harvest (~2012)", "OPEN: confirm WS2 last harvest year"),
    ("plot_canopy", "ArcGIS could not read the .laz files; decompressed with laspy 2.7.0 and CRS set explicitly to EPSG 26917", "resolved")],
    columns=qc.columns)])
write_df(qc, "meta_qc_log")

pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
pc2 = pc.set_index("plot_id").join(to_df(P).set_index("plot_id")[["watershed"]])
print("\nPER-PLOT CANOPY\n", pc2[["watershed", "chm_MEAN", "chm_MAX", "chm_MEDIAN", "chm_PCT90", "chm_PCT95", "chm_STD", "frac_ge5m_leafoff"]].round(1).to_string())
print("\nBY WATERSHED (footprint means)\n", pc2.groupby("watershed")[["chm_MEAN", "chm_MAX", "chm_PCT95", "chm_STD"]].mean().round(1))
print("\nWHOLE WATERSHEDS\n", to_df(os.path.join(GDB, "watershed_canopy")).round(1).to_string(index=False))

lidar_dsm1 exists: True


<class 'arcgisscripting.ExecuteError'>: ERROR 001100: Failed because no statistics are available.
Failed to execute (GetRasterProperties).


In [16]:
print("exists:", {r: arcpy.Exists(os.path.join(GDB, r)) for r in ["lidar_dsm1", "lidar_dtm1", "lidar_chm1"]})
try:
    r = arcpy.Raster(os.path.join(GDB, "lidar_dsm1"))
    print("dsm cols x rows:", r.width, "x", r.height, "| cell:", r.meanCellWidth, "| has stats:", r.hasRAT or True)
    a = arcpy.RasterToNumPyArray(os.path.join(GDB, "lidar_dsm1"), nodata_to_value=np.nan)
    print("dsm valid cells:", int(np.isfinite(a).sum()), "of", a.size, "| min/max:", np.nanmin(a), np.nanmax(a))
except Exception as e:
    print("dsm read failed:", str(e)[:200])

for nm, kw in [("ground_noflags", dict(class_code=[2])),
               ("ground_strict", dict(class_code=[2], withheld=False, overlap=False))]:
    try:
        if arcpy.Exists(nm): arcpy.management.Delete(nm)
        lyr = arcpy.management.MakeLasDatasetLayer(LASD, nm, **kw)
        print(nm, "-> points:", arcpy.Describe(lyr).pointCount)
    except Exception as e:
        print(nm, "FAILED:", str(e)[:200])

exists: {'lidar_dsm1': True, 'lidar_dtm1': True, 'lidar_chm1': True}
dsm cols x rows: 2702 x 2384 | cell: 1.0 | has stats: True


C:/Users/DCIFEL~1/AppData/Local/Temp/ArcGISProTemp17532/xpython_17532/4006948339.py:6: RuntimeWarning: All-NaN slice encountered
  print("dsm valid cells:", int(np.isfinite(a).sum()), "of", a.size, "| min/max:", np.nanmin(a), np.nanmax(a))


dsm valid cells: 0 of 6441568 | min/max: nan nan
ground_noflags -> points: 84692970
ground_strict -> points: 84692970


In [17]:
import numpy as np, laspy, os
pl = to_df(P, xy=True).set_index("plot_id")
R = 11.35
res = {}
for pid, row in pl.iterrows(): res[pid] = {"x": row["SHAPE@X"], "y": row["SHAPE@Y"], "z": [], "cls": [], "ret": [], "num": []}

for f in sorted(os.listdir(LAS_DIR)):
    if not f.lower().endswith(".laz"): continue
    with laspy.open(os.path.join(LAS_DIR, f)) as src:
        for pts in src.chunk_iterator(3_000_000):
            X, Y, Z = np.asarray(pts.x), np.asarray(pts.y), np.asarray(pts.z)
            C, RN, NR = np.asarray(pts.classification), np.asarray(pts.return_number), np.asarray(pts.number_of_returns)
            keep_noise = (C != 7) & (C != 18)
            for pid, d in res.items():
                m = (np.abs(X - d["x"]) <= R) & (np.abs(Y - d["y"]) <= R)
                if not m.any(): continue
                m &= ((X - d["x"])**2 + (Y - d["y"])**2 <= R*R) & keep_noise
                if m.any():
                    d["z"].append(Z[m]); d["cls"].append(C[m]); d["ret"].append(RN[m]); d["num"].append(NR[m])
    print("scanned", f, flush=True)

rows = []
for pid, d in res.items():
    if not d["z"]:
        rows.append({"plot_id": pid}); continue
    z = np.concatenate(d["z"]); c = np.concatenate(d["cls"]); rn = np.concatenate(d["ret"])
    gz = z[c == 2]
    if gz.size < 5:
        rows.append({"plot_id": pid, "n_points": z.size, "n_ground": gz.size}); continue
    ground = np.median(gz)                       # plot ground elevation
    h = z[(rn == 1)] - ground                    # first returns, height above ground
    h = h[(h >= 0) & (h <= 55)]
    rows.append({"plot_id": pid, "n_points": int(z.size), "n_ground": int(gz.size), "n_first_returns": int(h.size),
                 "ground_elev_m": round(float(ground), 2),
                 "chm_mean_m": round(float(h.mean()), 2), "chm_max_m": round(float(h.max()), 2),
                 "chm_p50_m": round(float(np.percentile(h, 50)), 2), "chm_p90_m": round(float(np.percentile(h, 90)), 2),
                 "chm_p95_m": round(float(np.percentile(h, 95)), 2), "chm_p99_m": round(float(np.percentile(h, 99)), 2),
                 "chm_sd_m": round(float(h.std()), 2),
                 "frac_ge5m_leafoff": round(float((h >= 5).mean()), 3),
                 "frac_ge20m_leafoff": round(float((h >= 20).mean()), 3)})
pc = pd.DataFrame(rows)
write_df(pc, "plot_canopy")
out = os.path.join(GDB, "rel_plots_study_plot_canopy")
if arcpy.Exists(out): arcpy.management.Delete(out)
arcpy.management.CreateRelationshipClass(P, os.path.join(GDB, "plot_canopy"), out, "SIMPLE", "plot_canopy",
                                         "plots_study", "NONE", "ONE_TO_ONE", "NONE", "plot_id", "plot_id")

pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
j = pc.set_index("plot_id").join(to_df(P).set_index("plot_id")[["watershed"]])
print(j[["watershed", "n_points", "n_ground", "ground_elev_m", "chm_mean_m", "chm_max_m", "chm_p95_m", "chm_sd_m", "frac_ge20m_leafoff"]].to_string())
print("\nBY WATERSHED\n", j.groupby("watershed")[["chm_mean_m", "chm_max_m", "chm_p95_m", "chm_sd_m", "frac_ge20m_leafoff"]].mean().round(1))

scanned USGS_LPC_WV_FEMAHQ_2018_D18_e1390n1899.laz
scanned USGS_LPC_WV_FEMAHQ_2018_D18_e1390n1900.laz
scanned USGS_LPC_WV_FEMAHQ_2018_D18_e1390n1901.laz
scanned USGS_LPC_WV_FEMAHQ_2018_D18_e1391n1899.laz
scanned USGS_LPC_WV_FEMAHQ_2018_D18_e1391n1900.laz
scanned USGS_LPC_WV_FEMAHQ_2018_D18_e1391n1901.laz
scanned USGS_LPC_WV_FEMAHQ_2018_D18_e1392n1899.laz
scanned USGS_LPC_WV_FEMAHQ_2018_D18_e1392n1900.laz
scanned USGS_LPC_WV_FEMAHQ_2018_D18_e1392n1901.laz
scanned USGS_LPC_WV_FEMAHQ_2018_D18_e1392n1902.laz
plot_canopy: 18 rows, 1 fields


<class 'KeyError'>: "['n_points', 'n_ground', 'ground_elev_m', 'chm_mean_m', 'chm_max_m', 'chm_p95_m', 'chm_sd_m', 'frac_ge20m_leafoff'] not in index"

In [18]:
import laspy, numpy as np
f0 = os.path.join(LAS_DIR, sorted(os.listdir(LAS_DIR))[0])
with laspy.open(f0) as src:
    h = src.header
    print("LAS mins:", np.round(h.mins, 1), "\nLAS maxs:", np.round(h.maxs, 1))
    print("offsets:", h.offsets, "| scales:", h.scales)
    for v in h.vlrs:
        print("  VLR:", getattr(v, "record_id", "?"), str(v)[:120])

pl = to_df(P, xy=True)
print("\nplots X:", round(pl['SHAPE@X'].min(), 1), "-", round(pl['SHAPE@X'].max(), 1))
print("plots Y:", round(pl['SHAPE@Y'].min(), 1), "-", round(pl['SHAPE@Y'].max(), 1))

LAS mins: [1.390e+06 1.899e+06 5.641e+02] 
LAS maxs: [1.391e+06 1.900e+06 8.778e+02]
offsets: [-0. -0. -0.] | scales: [0.01 0.01 0.01]
  VLR: 2112 <laspy.vlrs.known.WktCoordinateSystemVlr object at 0x000001D5C0322C90>
  VLR: 4 <VLR(user_id: 'NIIRS10', record_id: '4', data len: 10)>
  VLR: 1 <VLR(user_id: 'NIIRS10', record_id: '1', data len: 26)>
  VLR: 22204 <laspy.vlrs.known.LasZipVlr object at 0x000001D5C0321AD0>

plots X: 613017.4 - 614062.9
plots Y: 4322932.5 - 4324249.0


In [19]:
import laspy, numpy as np
with laspy.open(os.path.join(LAS_DIR, sorted(os.listdir(LAS_DIR))[0])) as src:
    wkt = next((v.string for v in src.header.vlrs if v.__class__.__name__ == "WktCoordinateSystemVlr"), None)
print(wkt[:600] if wkt else "no WKT found")

las_sr = arcpy.SpatialReference(); las_sr.loadFromString(wkt)
print("\nLAS CRS:", las_sr.name, "| WKID:", las_sr.factoryCode, "| XY units:", las_sr.linearUnitName)

pl = to_df(P, xy=False)[["plot_id"]]
with arcpy.da.SearchCursor(P, ["plot_id", "SHAPE@XY"], spatial_reference=las_sr) as cur:
    conv = {p: xy for p, xy in cur}
print("\nplots in LAS coordinates:")
for p, (x, y) in list(conv.items())[:3]: print(f"   {p}: {x:.1f}, {y:.1f}")
xs = [v[0] for v in conv.values()]; ys = [v[1] for v in conv.values()]
print(f"   range X {min(xs):.0f}-{max(xs):.0f}  Y {min(ys):.0f}-{max(ys):.0f}")
print("   (LAS tile coverage: X 1390000-1392000+, Y 1899000-1902000+)")

COMPD_CS["NAD83(2011) / Conus Albers + NAVD88 height - Geoid12B (metre)",PROJCS["NAD83(2011) / Conus Albers",GEOGCS["NAD83(2011)",DATUM["NAD83_National_Spatial_Reference_System_2011",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","1116"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","6318"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAM


<class 'ValueError'>: SpatialReference: Spatial reference string is invalid.

In [20]:
las_sr = arcpy.SpatialReference(6350)          # NAD83(2011) / Conus Albers
print("LAS CRS:", las_sr.name, "| units:", las_sr.linearUnitName)
with arcpy.da.SearchCursor(P, ["plot_id", "SHAPE@XY"], spatial_reference=las_sr) as cur:
    conv = {p: xy for p, xy in cur}
xs = [v[0] for v in conv.values()]; ys = [v[1] for v in conv.values()]
print(f"plots in Albers: X {min(xs):.0f}-{max(xs):.0f}  Y {min(ys):.0f}-{max(ys):.0f}")
print("LAS tiles cover:  X 1390000-1393000   Y 1899000-1902000")

LAS CRS: NAD_1983_2011_Contiguous_USA_Albers | units: Meter
plots in Albers: X 1391180-1392188  Y 1900053-1901322
LAS tiles cover:  X 1390000-1393000   Y 1899000-1902000


In [21]:
R = 11.35
res = {pid: {"x": xy[0], "y": xy[1], "z": [], "cls": [], "ret": []} for pid, xy in conv.items()}
for f in sorted(os.listdir(LAS_DIR)):
    if not f.lower().endswith(".laz"): continue
    with laspy.open(os.path.join(LAS_DIR, f)) as src:
        for pts in src.chunk_iterator(3_000_000):
            X, Y, Z = np.asarray(pts.x), np.asarray(pts.y), np.asarray(pts.z)
            C, RN = np.asarray(pts.classification), np.asarray(pts.return_number)
            ok = (C != 7) & (C != 18)
            for d in res.values():
                m = (np.abs(X - d["x"]) <= R) & (np.abs(Y - d["y"]) <= R)
                if not m.any(): continue
                m &= ((X - d["x"])**2 + (Y - d["y"])**2 <= R*R) & ok
                if m.any(): d["z"].append(Z[m]); d["cls"].append(C[m]); d["ret"].append(RN[m])
    print("scanned", f, flush=True)

rows = []
for pid, d in res.items():
    if not d["z"]: rows.append({"plot_id": pid, "n_points": 0}); continue
    z, c, rn = np.concatenate(d["z"]), np.concatenate(d["cls"]), np.concatenate(d["ret"])
    gz = z[c == 2]
    if gz.size < 5: rows.append({"plot_id": pid, "n_points": int(z.size), "n_ground": int(gz.size)}); continue
    ground = float(np.median(gz)); h = z[rn == 1] - ground; h = h[(h >= 0) & (h <= 55)]
    rows.append({"plot_id": pid, "n_points": int(z.size), "n_ground": int(gz.size), "n_first_returns": int(h.size),
                 "ground_elev_m": round(ground, 2), "chm_mean_m": round(float(h.mean()), 2),
                 "chm_max_m": round(float(h.max()), 2), "chm_p50_m": round(float(np.percentile(h, 50)), 2),
                 "chm_p90_m": round(float(np.percentile(h, 90)), 2), "chm_p95_m": round(float(np.percentile(h, 95)), 2),
                 "chm_p99_m": round(float(np.percentile(h, 99)), 2), "chm_sd_m": round(float(h.std()), 2),
                 "frac_ge5m_leafoff": round(float((h >= 5).mean()), 3),
                 "frac_ge20m_leafoff": round(float((h >= 20).mean()), 3)})
pc = pd.DataFrame(rows)
write_df(pc, "plot_canopy")
out = os.path.join(GDB, "rel_plots_study_plot_canopy")
if arcpy.Exists(out): arcpy.management.Delete(out)
arcpy.management.CreateRelationshipClass(P, os.path.join(GDB, "plot_canopy"), out, "SIMPLE", "plot_canopy",
                                         "plots_study", "NONE", "ONE_TO_ONE", "NONE", "plot_id", "plot_id")
ms = to_df(os.path.join(GDB, "meta_sources"))
ms = pd.concat([ms, pd.DataFrame([("plot_canopy", "USGS 3DEP WV_FEMAHQ_2018_D18 (flown 2018-11-08 to 2019-05-01), 84.7 M points, NAD83(2011) Conus Albers (EPSG 6350), heights in metres",
    "per-plot from point cloud via laspy: ground = median class-2 elevation in footprint; heights = first returns above ground, 0-55 m")], columns=ms.columns)])
write_df(ms, "meta_sources")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30)
j = pc.set_index("plot_id").join(to_df(P).set_index("plot_id")[["watershed"]])
print(j.to_string())
print("\nBY WATERSHED\n", j.groupby("watershed")[["chm_mean_m", "chm_max_m", "chm_p95_m", "chm_sd_m", "frac_ge20m_leafoff"]].mean().round(1))

scanned USGS_LPC_WV_FEMAHQ_2018_D18_e1390n1899.laz
scanned USGS_LPC_WV_FEMAHQ_2018_D18_e1390n1900.laz
scanned USGS_LPC_WV_FEMAHQ_2018_D18_e1390n1901.laz
scanned USGS_LPC_WV_FEMAHQ_2018_D18_e1391n1899.laz
scanned USGS_LPC_WV_FEMAHQ_2018_D18_e1391n1900.laz
scanned USGS_LPC_WV_FEMAHQ_2018_D18_e1391n1901.laz
scanned USGS_LPC_WV_FEMAHQ_2018_D18_e1392n1899.laz
scanned USGS_LPC_WV_FEMAHQ_2018_D18_e1392n1900.laz
scanned USGS_LPC_WV_FEMAHQ_2018_D18_e1392n1901.laz
scanned USGS_LPC_WV_FEMAHQ_2018_D18_e1392n1902.laz
plot_canopy: 18 rows, 14 fields
meta_sources: 10 rows, 3 fields
         n_points  n_ground  n_first_returns  ground_elev_m  chm_mean_m  chm_max_m  chm_p50_m  chm_p90_m  chm_p95_m  chm_p99_m  chm_sd_m  frac_ge5m_leafoff  frac_ge20m_leafoff watershed
plot_id                                                                                                                                                                                  
WS2A-1       2815      1306             1274         7

In [22]:
qc = to_df(os.path.join(GDB, "meta_qc_log"))
qc = pd.concat([qc, pd.DataFrame([
    ("plot_canopy", "Lidar delivered in NAD83(2011) Conus Albers (EPSG 6350), not UTM 17N; ArcGIS LAS tools returned empty rasters on this install, so metrics computed directly from the point cloud with laspy", "resolved"),
    ("plot_canopy", "Flown leaf-off (Nov 2018-May 2019): height metrics reliable; frac_ge5m/ge20m biased low, use as relative measures only", "documented"),
    ("plot_canopy", "Lidar predates the 2022 inventory by 3-4 yr; canopy ranking (WS4 >> WS2 ~ WS5) matches inventory max-DBH and AGB ranking", "independent corroboration")],
    columns=qc.columns)])
write_df(qc, "meta_qc_log")

exp = os.path.join(PROJ, "gdb_export"); os.makedirs(exp, exist_ok=True)
arcpy.env.workspace = GDB
for n in arcpy.ListFeatureClasses() + arcpy.ListTables():
    if n.startswith("tmp_"): continue
    d = arcpy.Describe(n)
    is_pt = d.dataType in ("FeatureClass", "FeatureLayer") and d.shapeType == "Point"
    to_df(n, xy=is_pt).to_csv(os.path.join(exp, f"{n}.csv"), index=False)
print(len(os.listdir(exp)), "CSVs ready for GitHub in", exp)

meta_qc_log: 27 rows, 3 fields
63 CSVs ready for GitHub in C:\Users\DCifelli2\OneDrive - mercercounty.org\Documents\ArcGIS\Projects\Fernow-FM-SOM-DC-092026\gdb_export


In [1]:
import arcpy, os, numpy as np, pandas as pd
arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False
aprx = arcpy.mp.ArcGISProject("CURRENT")
PROJ = os.path.dirname(aprx.filePath)
GDB  = os.path.join(PROJ, "Fernow_SOM_master.gdb")
SR   = arcpy.SpatialReference(26917)
SHP_ROOT = r"C:\Users\DCifelli2\OneDrive - mercercounty.org\Desktop\FernowShapeFiles"

def find_shp(n):
    h=[os.path.join(r,n+".shp") for r,_,f in os.walk(SHP_ROOT) if n+".shp" in f]
    return h[0] if h else None
def to_df(fc, xy=False):
    f=[x.name for x in arcpy.ListFields(fc) if x.type not in ("Geometry","OID","Blob","Raster","GlobalID")]
    q=f+(["SHAPE@X","SHAPE@Y"] if xy else [])
    return pd.DataFrame(arcpy.da.SearchCursor(fc,q),columns=q)

# ---- assemble one flat row per plot from the master gdb tables ----
EXP, CF = 1/0.0404686, 0.48
t = to_df(os.path.join(GDB,"trees")); v = t.dropna(subset=["dbh_cm"])
st = []
for pid,g in v.groupby("plot_id"):
    ba=g.ba_m2.sum()
    st.append(dict(plot_id=pid, agbC=g.agb_kg_jenkins.sum()/1000*EXP*CF, ba=ba*EXP, stems=len(g)*EXP,
        qmd=np.sqrt((g.dbh_cm**2).mean()), dbhmax=g.dbh_cm.max(),
        big_share=100*g.loc[g.dbh_cm>=50,"agb_kg_jenkins"].sum()/g.agb_kg_jenkins.sum(),
        rich=g.groupby(["genus","species"]).ngroups,
        ecm=100*g.loc[g.myc_type=="ECM","ba_m2"].sum()/ba,
        top_sp=(g.assign(k=g.genus+" "+g.species).groupby("k").ba_m2.sum().idxmax())))
st = pd.DataFrame(st)

p  = to_df(os.path.join(GDB,"plots_study"), xy=True)
df = (p[["plot_id","watershed","treatment","usfs_subcomp","subcomp_mapped","site_index_mapped",
         "cutting_cycle_mapped","xy_source","SHAPE@X","SHAPE@Y"]]
      .merge(st,on="plot_id")
      .merge(to_df(os.path.join(GDB,"plot_terrain"))[["plot_id","dem5_fp","slope5_fp","north5_fp","tpi200_fp","twi5_fp"]],on="plot_id",how="left")
      .merge(to_df(os.path.join(GDB,"plot_solar")),on="plot_id",how="left")
      .merge(to_df(os.path.join(GDB,"plot_canopy"))[["plot_id","chm_mean_m","chm_p95_m","chm_max_m","frac_ge20m_leafoff"]],on="plot_id",how="left")
      .merge(to_df(os.path.join(GDB,"plot_site"))[["plot_id","nrcs_SoilSeries","elt_ELT_NAME","stream_dist_any_m"]],on="plot_id",how="left")
      .merge(to_df(os.path.join(GDB,"ssurgo_plot"))[["plot_id","musym","om_r_0_20","ph1to1h2o_h2o" if False else "ph1to1h2o_r_0_20","fragvol_r_0_20","brockdepmin"]],on="plot_id",how="left")
      .merge(to_df(os.path.join(GDB,"plot_history_geology"))[["plot_id","cover1932_COVER_TYPE"]],on="plot_id",how="left")
      .merge(to_df(os.path.join(GDB,"plot_disturbance"))[["plot_id","dist_skid_trail_m"]],on="plot_id",how="left"))
df["yrs_since_harvest"] = df.watershed.map({"WS4":112,"WS2":2,"WS5":9})
df["frac_ge20m_leafoff"] = df.frac_ge20m_leafoff*100
df["xy_source"] = df.xy_source.map({"USFS":"USFS permanent growth plot","GPS":"Field GPS (no USFS plot)"})

# field name -> (alias, decimals); order here is the popup order
FLD = [("plot_id","Plot",None),("watershed","Watershed",None),("treatment","Treatment",None),
 ("yrs_since_harvest","Years since last harvest",0),("subcomp_mapped","Subcompartment (mapped)",None),
 ("site_index_mapped","Site index (red oak, base 50)",0),("cutting_cycle_mapped","Cutting cycle",None),
 ("xy_source","Plot location source",None),
 ("agbC","Aboveground live C (Mg C/ha)",1),("ba","Basal area (m2/ha)",1),("stems","Stem density (stems/ha)",0),
 ("qmd","Quadratic mean diameter (cm)",1),("dbhmax","Largest tree DBH (cm)",1),
 ("big_share","AGB in trees >=50 cm (%)",0),("rich","Tree species richness",0),
 ("ecm","Ectomycorrhizal basal area (%)",0),("top_sp","Dominant species by basal area",None),
 ("chm_mean_m","Mean canopy height (m)",1),("chm_p95_m","95th pct canopy height (m)",1),
 ("chm_max_m","Max canopy height (m)",1),("frac_ge20m_leafoff","Canopy returns >=20 m (%)",0),
 ("dem5_fp","Elevation (m)",0),("slope5_fp","Slope (degrees)",1),("north5_fp","Northness (1 = due north)",2),
 ("tpi200_fp","Topographic position, 200 m (m)",1),("twi5_fp","Topographic wetness index",1),
 ("solar_kwh_m2_yr_fp","Solar radiation (kWh/m2/yr)",0),
 ("nrcs_SoilSeries","Soil series (NRCS 2008)",None),("musym","SSURGO map unit",None),
 ("om_r_0_20","Soil organic matter 0-20 cm (%)",1),("ph1to1h2o_r_0_20","Soil pH 0-20 cm",1),
 ("fragvol_r_0_20","Rock fragments 0-20 cm (%)",0),("brockdepmin","Depth to bedrock (cm)",0),
 ("elt_ELT_NAME","Ecological land type",None),("cover1932_COVER_TYPE","1932 forest cover type",None),
 ("stream_dist_any_m","Distance to stream (m)",0),("dist_skid_trail_m","Distance to skid trail (m)",0)]
for f,_,dec in FLD:
    if dec is not None and f in df: df[f]=df[f].astype(float).round(dec)

out = os.path.join(GDB,"MAP_plots")
if arcpy.Exists(out): arcpy.management.Delete(out)
arcpy.management.CreateFeatureclass(GDB,"MAP_plots","POINT",spatial_reference=SR)
for f,al,dec in FLD:
    txt = df[f].dtype==object
    arcpy.management.AddField(out,f,"TEXT" if txt else ("LONG" if dec==0 else "DOUBLE"),
                              field_length=80 if txt else None, field_alias=al)
with arcpy.da.InsertCursor(out,[f for f,_,_ in FLD]+["SHAPE@XY"]) as cur:
    for _,r in df.iterrows():
        row=[None if (not isinstance(r[f],str) and pd.isna(r[f])) else (r[f].item() if isinstance(r[f],np.generic) else r[f]) for f,_,_ in FLD]
        cur.insertRow(row+[(float(r["SHAPE@X"]),float(r["SHAPE@Y"]))])
print("MAP_plots:",arcpy.management.GetCount(out)[0],"plots,",len(FLD),"popup fields")

MAP_plots: 18 plots, 37 popup fields


In [2]:
from scipy import stats
sd = to_df(os.path.join(GDB,"soil_samples"))
fr = to_df(os.path.join(GDB,"soil_fractions"))
bd = sd.groupby(["watershed","soil_plot","depth_code"])[["soil_C_gm2","soil_N_gm2","fine_root_gm2","BD_gcm3"]].mean().reset_index()
pv = bd.pivot_table(index=["watershed","soil_plot"],columns="depth_code",
                    values=["soil_C_gm2","fine_root_gm2"]).reset_index()
pv.columns=[f"{a}_{b}" if b else a for a,b in pv.columns]
pv["soil040"]=(pv["soil_C_gm2_0-20"]+pv["soil_C_gm2_20-40"])/100
pv["ohC"]=pv["soil_C_gm2_OH"]/100
pv["rootC"]=pv[[c for c in pv if c.startswith("fine_root_gm2")]].sum(axis=1)*0.48/100
pv["belowC"]=pv.soil040+pv.ohC+pv.rootC
fr["lf_mass_gm2"]=(fr.LF_g/fr.frac_soil_wt_g)*bd.set_index(["watershed","soil_plot","depth_code"]).BD_gcm3.reindex(
    pd.MultiIndex.from_arrays([fr.watershed,fr.soil_plot,fr.depth_code])).values*20*1e4

rows=[]
for ws in ["WS2","WS4","WS5"]:
    a=df[df.watershed==ws]; b=pv[pv.watershed==ws]; L=fr[(fr.watershed==ws)&(fr.depth_code=="20-40")]
    rows.append(dict(Ws_name=ws, treatment=a.treatment.iat[0],
        yrs_since_harvest=int(a.yrs_since_harvest.iat[0]),
        last_harvest={"WS4":"1907-1915 (railroad logging)","WS2":"2020","WS5":"2013"}[ws],
        n_plots=len(a), agbC=round(a.agbC.mean(),1), agbC_se=round(stats.sem(a.agbC),1),
        ohC=round(b.ohC.mean(),2), soil040=round(b.soil040.mean(),1), soil040_se=round(stats.sem(b.soil040),1),
        rootC=round(b.rootC.mean(),2), belowC=round(b.belowC.mean(),1),
        totalC=round(a.agbC.mean()+b.belowC.mean(),1),
        pct_above=round(100*a.agbC.mean()/(a.agbC.mean()+b.belowC.mean())),
        lf_mass=round(L.lf_mass_gm2.mean()), elev=round(a.dem5_fp.mean()), slope=round(a.slope5_fp.mean(),1),
        site_index=round(a.site_index_mapped.mean(),1)))
S=pd.DataFrame(rows)

WFLD=[("treatment","Treatment",None),("yrs_since_harvest","Years since last harvest",0),
 ("last_harvest","Last harvest",None),("n_plots","Plots sampled",0),
 ("agbC","Aboveground live C (Mg C/ha)",1),("ohC","Organic horizon C (Mg C/ha)",2),
 ("soil040","Mineral soil C, 0-40 cm (Mg C/ha)",1),("rootC","Fine-root C, 0-40 cm (Mg C/ha)",2),
 ("belowC","Total belowground C (Mg C/ha)",1),("totalC","TOTAL ecosystem C (Mg C/ha)",1),
 ("pct_above","Share stored aboveground (%)",0),
 ("lf_mass","Light-fraction mass, 20-40 cm (g/m2)",0),
 ("elev","Mean plot elevation (m)",0),("slope","Mean plot slope (degrees)",1),
 ("site_index","Mean site index (red oak)",1)]

wout=os.path.join(GDB,"MAP_watersheds")
if arcpy.Exists(wout): arcpy.management.Delete(wout)
arcpy.env.outputCoordinateSystem=SR
lyr=arcpy.management.MakeFeatureLayer(find_shp("watersheds_all"),"w3","Ws_name IN ('WS2','WS4','WS5')")
arcpy.conversion.ExportFeatures(lyr,wout); arcpy.management.Delete("w3")
arcpy.env.outputCoordinateSystem=None
keep={"Ws_name"}
arcpy.management.DeleteField(wout,[f.name for f in arcpy.ListFields(wout)
    if f.name not in keep and f.type not in ("OID","Geometry") and not f.name.lower().startswith("shape")])
arcpy.management.AlterField(wout,"Ws_name",new_field_alias="Watershed")
for f,al,dec in WFLD:
    arcpy.management.AddField(wout,f,"TEXT" if S[f].dtype==object else ("LONG" if dec==0 else "DOUBLE"),
                              field_length=60 if S[f].dtype==object else None, field_alias=al)
Sx=S.set_index("Ws_name")
with arcpy.da.UpdateCursor(wout,["Ws_name"]+[f for f,_,_ in WFLD]) as cur:
    for row in cur:
        r=Sx.loc[row[0]]
        cur.updateRow([row[0]]+[r[f].item() if isinstance(r[f],np.generic) else r[f] for f,_,_ in WFLD])
print("MAP_watersheds:",arcpy.management.GetCount(wout)[0],"| totals:",dict(zip(S.Ws_name,S.totalC)))

MAP_watersheds: 3 | totals: {'WS2': 172.2, 'WS4': 263.9, 'WS5': 154.3}


In [3]:
import arcpy, os
aprx = arcpy.mp.ArcGISProject("CURRENT")
NAME = "Fernow_carbon_map"
for mp in aprx.listMaps(NAME):
    aprx.deleteItem(mp)
m = aprx.createMap(NAME, "Map")
m.addBasemap("Terrain with Labels")

def add(src, name=None):
    lyr = m.addDataFromPath(src if src.startswith("\\\\") or ":" in src else src)
    if name: lyr.name = name
    return lyr

# draw order: added last sits on top, so add bottom-up
hs   = None
dem  = os.path.join(GDB, "terrain_dem5")
if arcpy.Exists(dem):
    hs = add(dem, "Elevation (5 m lidar DEM)"); hs.visible = False
strm = add(os.path.join(GDB, "ref_streams"), "Streams")
wsh  = add(os.path.join(GDB, "MAP_watersheds"), "Watersheds (click for carbon budget)")
plt_ = add(os.path.join(GDB, "MAP_plots"), "Sampling plots (click for plot record)")
print("layers:", [l.name for l in m.listLayers()])

layers: ['World Terrain Reference', 'Sampling plots (click for plot record)', 'Streams', 'Watersheds (click for carbon budget)', 'Elevation (5 m lidar DEM)', 'World Terrain Base', 'World Hillshade']


In [4]:
COL = {"WS4": [27, 94, 63, 45], "WS2": [193, 85, 59, 45], "WS5": [74, 111, 165, 45]}
OUT = {"WS4": [27, 94, 63, 100], "WS2": [193, 85, 59, 100], "WS5": [74, 111, 165, 100]}

# --- watersheds: fill by treatment ---
s = wsh.symbology
s.updateRenderer("UniqueValueRenderer")
s.renderer.fields = ["Ws_name"]
for grp in s.renderer.groups:
    for itm in grp.items:
        k = itm.values[0][0]
        itm.symbol.color = {"RGB": COL[k]}
        itm.symbol.outlineColor = {"RGB": OUT[k]}
        itm.symbol.outlineWidth = 2
        itm.label = {"WS4": "WS4 — uncut reference (112 yr)",
                     "WS2": "WS2 — diameter-limit cut (2 yr)",
                     "WS5": "WS5 — single-tree selection (9 yr)"}[k]
wsh.symbology = s

# --- plots: graduated size by aboveground carbon, coloured by watershed ---
s = plt_.symbology
s.updateRenderer("UniqueValueRenderer")
s.renderer.fields = ["watershed"]
for grp in s.renderer.groups:
    for itm in grp.items:
        k = itm.values[0][0]
        itm.symbol.color = {"RGB": OUT[k]}
        itm.symbol.outlineColor = {"RGB": [255, 255, 255, 100]}
        itm.symbol.outlineWidth = 1.2
        itm.symbol.size = 8
        itm.label = {"WS4": "REF plots", "WS2": "DLC plots", "WS5": "STS plots"}[k]
plt_.symbology = s

# --- streams: thin blue, perennial heavier ---
s = strm.symbology
s.updateRenderer("SimpleRenderer")
s.renderer.symbol.color = {"RGB": [74, 144, 196, 100]}
s.renderer.symbol.width = 0.9
strm.symbology = s

# --- labels ---
plt_.showLabels = True
lc = plt_.listLabelClasses()[0]
lc.expression = '$feature.plot_id'
lc.visible = True
wsh.showLabels = False

# --- popups: show every field, using the aliases already set ---
for lyr, flds in [(plt_, [f.name for f in arcpy.ListFields(os.path.join(GDB, "MAP_plots"))
                          if f.type not in ("OID", "Geometry")]),
                  (wsh,  [f.name for f in arcpy.ListFields(os.path.join(GDB, "MAP_watersheds"))
                          if f.type not in ("OID", "Geometry") and not f.name.lower().startswith("shape")])]:
    cim = lyr.getDefinition("V3")
    cim.popupInfo = None            # clears any custom popup -> Pro shows all visible fields
    lyr.setDefinition(cim)
    fi = lyr.listFields()
    print(lyr.name, "->", len(flds), "popup fields")

aprx.save()
print("\nmap created:", NAME)
print("open it from the Catalog pane -> Maps ->", NAME)

<class 'AttributeError'>: 'Layer' object has no attribute 'listFields'

In [5]:
for lyr in [plt_, wsh]:
    cim = lyr.getDefinition("V3")
    cim.popupInfo = None          # clears custom popup -> Pro lists all visible fields
    lyr.setDefinition(cim)

# verify what was actually applied
for lyr, fc in [(plt_, "MAP_plots"), (wsh, "MAP_watersheds")]:
    flds = [f for f in arcpy.ListFields(os.path.join(GDB, fc)) if f.type not in ("OID", "Geometry")
            and not f.name.lower().startswith("shape")]
    print(f"\n{lyr.name}\n  renderer: {lyr.symbology.renderer.type}"
          f"\n  fields in popup: {len(flds)}"
          f"\n  labels on: {lyr.showLabels}")
    for f in flds[:4]:
        print(f"    {f.name:24s} alias: {f.aliasName}")
print("\nlayer order (top first):", [l.name for l in m.listLayers()])
aprx.save()
print("saved")


Sampling plots (click for plot record)
  renderer: UniqueValueRenderer
  fields in popup: 37
  labels on: True
    plot_id                  alias: Plot
    watershed                alias: Watershed
    treatment                alias: Treatment
    yrs_since_harvest        alias: Years since last harvest

Watersheds (click for carbon budget)
  renderer: UniqueValueRenderer
  fields in popup: 16
  labels on: False
    Ws_name                  alias: Watershed
    treatment                alias: Treatment
    yrs_since_harvest        alias: Years since last harvest
    last_harvest             alias: Last harvest

layer order (top first): ['World Terrain Reference', 'Sampling plots (click for plot record)', 'Streams', 'Watersheds (click for carbon budget)', 'Elevation (5 m lidar DEM)', 'World Terrain Base', 'World Hillshade']
saved


In [6]:
# move the basemap reference layer below the data layers
ref = [l for l in m.listLayers() if l.name == "World Terrain Reference"]
if ref:
    m.moveLayer(m.listLayers("Watersheds (click for carbon budget)")[0], ref[0], "AFTER")
print("order (top first):", [l.name for l in m.listLayers()])

# a print-ready layout, so the same map serves as Figure 1
LNAME = "Fernow_Fig1"
for lay in aprx.listLayouts(LNAME):
    aprx.deleteItem(lay)
lay = aprx.createLayout(180, 130, "MILLIMETER", LNAME)
mf = lay.createMapFrame(
    arcpy.Polygon(arcpy.Array([arcpy.Point(8, 26), arcpy.Point(172, 26),
                               arcpy.Point(172, 122), arcpy.Point(8, 122)])), m, "MainFrame")
ext = arcpy.Describe(os.path.join(GDB, "MAP_watersheds")).extent
buf = 300
mf.camera.setExtent(arcpy.Extent(ext.XMin-buf, ext.YMin-buf, ext.XMax+buf, ext.YMax+buf))
lay.createMapSurroundElement(arcpy.Point(14, 31), "NORTH_ARROW", mf, "ArcGIS North 1", "North")
lay.createMapSurroundElement(arcpy.Point(120, 31), "SCALE_BAR", mf,
                             "Double Alternating Scale Bar 1", "ScaleBar")
lay.createMapSurroundElement(arcpy.Point(8, 20), "LEGEND", mf, None, "Legend")
aprx.save()
print("layout created:", LNAME)

order (top first): ['Sampling plots (click for plot record)', 'Streams', 'Watersheds (click for carbon budget)', 'World Terrain Reference', 'Elevation (5 m lidar DEM)', 'World Terrain Base', 'World Hillshade']


<class 'ValueError'>: ArcGIS North 1

In [7]:
import numpy as np, pandas as pd
from scipy import stats

sd = to_df(os.path.join(GDB, "soil_samples"))
fr = to_df(os.path.join(GDB, "soil_fractions"))

# subplot -> plot means, then plot -> watershed mean ± SE
bulk = ["soil_C_gm2","soil_N_gm2","soil_CN","fine_root_gm2","BD_gcm3"]
pd_ = sd.groupby(["watershed","soil_plot","depth_code"])[bulk].mean().reset_index()

# light-fraction mass (complete data; %C was below detection in 3 STS subsoil samples)
bdi = pd_.set_index(["watershed","soil_plot","depth_code"]).BD_gcm3
fr["lf_mass_gm2"] = (fr.LF_g / fr.frac_soil_wt_g) * bdi.reindex(
    pd.MultiIndex.from_arrays([fr.watershed, fr.soil_plot, fr.depth_code])).values * 20 * 1e4

def ws_stat(df, col, dep=None, conv=1.0, dec=1):
    """watershed -> 'mean ± se' string, plus n if fewer than 6 plots contributed"""
    out = {}
    for ws in ["WS2","WS4","WS5"]:
        s = df[df.watershed == ws]
        if dep is not None: s = s[s.depth_code == dep]
        v = s[col].dropna() * conv
        if len(v) == 0: out[ws] = None; continue
        txt = f"{v.mean():.{dec}f} \u00b1 {stats.sem(v):.{dec}f}"
        if len(v) < 6: txt += f"  (n = {len(v)})"
        out[ws] = txt
    return out

# plot-level totals per watershed for the carbon budget lines
piv = pd_.pivot_table(index=["watershed","soil_plot"], columns="depth_code",
                      values=["soil_C_gm2","fine_root_gm2"]).reset_index()
piv.columns = [f"{a}_{b}" if b else a for a, b in piv.columns]
piv["soil040"] = (piv["soil_C_gm2_0-20"] + piv["soil_C_gm2_20-40"]) / 100
piv["ohC"]     = piv["soil_C_gm2_OH"] / 100
piv["rootC"]   = piv[[c for c in piv if c.startswith("fine_root_gm2")]].sum(axis=1) * 0.48 / 100
piv["belowC"]  = piv.soil040 + piv.ohC + piv.rootC

agb_ws = df.groupby("watershed").agbC.mean()
tot_ws = {ws: round(agb_ws[ws] + piv.loc[piv.watershed == ws, "belowC"].mean(), 1) for ws in ["WS2","WS4","WS5"]}
pct_ws = {ws: round(100 * agb_ws[ws] / (agb_ws[ws] + piv.loc[piv.watershed == ws, "belowC"].mean())) for ws in ["WS2","WS4","WS5"]}

# (field, alias, values-by-watershed, type)
NEW = [
 ("soil_note","[WS mean] Soil data scale",{w:"Watershed mean of 6 soil plots; soil plots were not co-located with tree plots" for w in ["WS2","WS4","WS5"]},"TEXT"),
 ("oh_C","[WS mean] Organic horizon C (Mg C/ha)",   ws_stat(pd_,"soil_C_gm2","OH",0.01,2),"TEXT"),
 ("oh_N","[WS mean] Organic horizon N (g N/m2)",    ws_stat(pd_,"soil_N_gm2","OH",1,2),"TEXT"),
 ("oh_root","[WS mean] Organic horizon roots (g/m2)",ws_stat(pd_,"fine_root_gm2","OH",1,1),"TEXT"),
 ("s1_C","[WS mean] Soil C 0-20 cm (Mg C/ha)",      ws_stat(pd_,"soil_C_gm2","0-20",0.01,1),"TEXT"),
 ("s1_N","[WS mean] Soil N 0-20 cm (g N/m2)",       ws_stat(pd_,"soil_N_gm2","0-20",1,0),"TEXT"),
 ("s1_CN","[WS mean] Soil C:N 0-20 cm",             ws_stat(pd_,"soil_CN","0-20",1,1),"TEXT"),
 ("s1_BD","[WS mean] Bulk density 0-20 cm (g/cm3)", ws_stat(pd_,"BD_gcm3","0-20",1,2),"TEXT"),
 ("s1_root","[WS mean] Fine roots 0-20 cm (g/m2)",  ws_stat(pd_,"fine_root_gm2","0-20",1,0),"TEXT"),
 ("s2_C","[WS mean] Soil C 20-40 cm (Mg C/ha)",     ws_stat(pd_,"soil_C_gm2","20-40",0.01,1),"TEXT"),
 ("s2_N","[WS mean] Soil N 20-40 cm (g N/m2)",      ws_stat(pd_,"soil_N_gm2","20-40",1,0),"TEXT"),
 ("s2_CN","[WS mean] Soil C:N 20-40 cm",            ws_stat(pd_,"soil_CN","20-40",1,1),"TEXT"),
 ("s2_BD","[WS mean] Bulk density 20-40 cm (g/cm3)",ws_stat(pd_,"BD_gcm3","20-40",1,2),"TEXT"),
 ("s2_root","[WS mean] Fine roots 20-40 cm (g/m2)", ws_stat(pd_,"fine_root_gm2","20-40",1,0),"TEXT"),
 ("f1_LF","[WS mean] Light-fraction C 0-20 cm (g C/m2)", ws_stat(fr,"LF_C_gm2","0-20",1,0),"TEXT"),
 ("f1_POM","[WS mean] Heavy POM C 0-20 cm (g C/m2)",     ws_stat(fr,"hPOM_C_gm2","0-20",1,0),"TEXT"),
 ("f1_MAOM","[WS mean] MAOM C 0-20 cm (g C/m2)",         ws_stat(fr,"MAOM_C_gm2","0-20",1,0),"TEXT"),
 ("f2_LFmass","[WS mean] Light-fraction MASS 20-40 cm (g/m2)", ws_stat(fr,"lf_mass_gm2","20-40",1,0),"TEXT"),
 ("f2_LF","[WS mean] Light-fraction C 20-40 cm (g C/m2)",      ws_stat(fr,"LF_C_gm2","20-40",1,0),"TEXT"),
 ("f2_POM","[WS mean] Heavy POM C 20-40 cm (g C/m2)",          ws_stat(fr,"hPOM_C_gm2","20-40",1,0),"TEXT"),
 ("f2_MAOM","[WS mean] MAOM C 20-40 cm (g C/m2)",              ws_stat(fr,"MAOM_C_gm2","20-40",1,0),"TEXT"),
 ("soil040","[WS mean] Mineral soil C 0-40 cm (Mg C/ha)",
     {w:f"{piv.loc[piv.watershed==w,'soil040'].mean():.1f} \u00b1 {stats.sem(piv.loc[piv.watershed==w,'soil040']):.1f}" for w in ["WS2","WS4","WS5"]},"TEXT"),
 ("belowC","[WS mean] Total belowground C (Mg C/ha)",
     {w:f"{piv.loc[piv.watershed==w,'belowC'].mean():.1f} \u00b1 {stats.sem(piv.loc[piv.watershed==w,'belowC']):.1f}" for w in ["WS2","WS4","WS5"]},"TEXT"),
 ("totalC","[WS mean] TOTAL ecosystem C (Mg C/ha)", tot_ws,"DOUBLE"),
 ("pct_above","[WS mean] Share stored aboveground (%)", pct_ws,"LONG"),
]

fc = os.path.join(GDB, "MAP_plots")
have = {f.name for f in arcpy.ListFields(fc)}
for f, al, _, ty in NEW:
    if f in have: arcpy.management.DeleteField(fc, f)
    arcpy.management.AddField(fc, f, ty, field_length=110 if ty == "TEXT" else None, field_alias=al)

with arcpy.da.UpdateCursor(fc, ["watershed"] + [f for f, _, _, _ in NEW]) as cur:
    for row in cur:
        ws = row[0]
        cur.updateRow([ws] + [vals.get(ws) for _, _, vals, _ in NEW])

lyr = m.listLayers("Sampling plots*")[0]
cim = lyr.getDefinition("V3"); cim.popupInfo = None; lyr.setDefinition(cim)
aprx.save()
print("MAP_plots now has", len([f for f in arcpy.ListFields(fc) if f.type not in ("OID","Geometry")]), "popup fields")
print("added:", len(NEW))
for f, al, vals, _ in NEW[:3] + NEW[-3:]:
    print(f"  {al:52s} WS4 = {vals['WS4']}")

MAP_plots now has 62 popup fields
added: 25
  [WS mean] Soil data scale                            WS4 = Watershed mean of 6 soil plots; soil plots were not co-located with tree plots
  [WS mean] Organic horizon C (Mg C/ha)                WS4 = 0.82 ± 0.20
  [WS mean] Organic horizon N (g N/m2)                 WS4 = 2.79 ± 0.47
  [WS mean] Total belowground C (Mg C/ha)              WS4 = 86.5 ± 7.7
  [WS mean] TOTAL ecosystem C (Mg C/ha)                WS4 = 263.9
  [WS mean] Share stored aboveground (%)               WS4 = 67


In [1]:
# ============================================================================
# X1  EXPLORATORY ANALYSIS - build analysis frames from the master geodatabase
# ============================================================================
import os, numpy as np, pandas as pd
from scipy import stats
rng = np.random.default_rng(42)
EXP, CF = 1/0.0404686, 0.48
O   = ["REF","DLC","STS"]
MP  = {"WS4":"REF","WS2":"DLC","WS5":"STS"}
COL = {"REF":"#1b5e3f","DLC":"#c1553b","STS":"#4a6fa5"}
LAB = {"REF":"REF\n(uncut)","DLC":"DLC\n(2 yr)","STS":"STS\n(4 yr)"}

# harvest history reconstructed from ws2/ws4/ws5_basal_area.xlsx (1958-2020)
SUB  = dict([(p,"WS2A") for p in (7,8,10,11,12)] + [(9,"WS2B")] +
            [(p,"WS4A") for p in (13,14,15,16,17,18)] +
            [(p,"WS5A") for p in (19,20,21,22,23)] + [(24,"WS5B")])
TSH  = {"WS2A":2,"WS2B":5,"WS4A":112,"WS4B":112,"WS4C":112,"WS5A":4,"WS5B":10}
INT  = {"WS2A":41.2,"WS2B":32.2,"WS4A":0.0,"WS4B":0.0,"WS4C":0.0,"WS5A":25.0,"WS5B":37.1}
CUM  = {"WS2A":160.9,"WS2B":110.2,"WS4A":0.0,"WS4B":0.0,"WS4C":0.0,"WS5A":149.9,"WS5B":141.3}
NH   = {"WS2A":5,"WS2B":4,"WS4A":0,"WS4B":0,"WS4C":0,"WS5A":8,"WS5B":5}

# ---- stand structure, one row per plot --------------------------------------
tr = to_df(os.path.join(GDB,"trees")); v = tr.dropna(subset=["dbh_cm"])
rows = []
for pid, g in v.groupby("plot_id"):
    ba = g["ba_m2"].sum()
    rows.append({"plot_id":pid,"agbC_Mgha":g["agb_kg_jenkins"].sum()/1000*EXP*CF,
                 "ba_m2ha":ba*EXP,"stems_ha":len(g)*EXP,
                 "qmd_cm":np.sqrt((g["dbh_cm"]**2).mean()),"dbhmax_cm":g["dbh_cm"].max(),
                 "agb_share_ge50cm":g.loc[g["dbh_cm"]>=50,"agb_kg_jenkins"].sum()/g["agb_kg_jenkins"].sum()})
st = pd.DataFrame(rows)
P_ = to_df(os.path.join(GDB,"plots_study"))
m  = P_[["plot_id","watershed","treatment","subcomp_mapped","site_index_mapped"]].merge(st, on="plot_id")
for t in ["plot_terrain","plot_canopy","plot_solar"]:
    m = m.merge(to_df(os.path.join(GDB,t)), on="plot_id", how="left")
m["trt"] = m["watershed"].map(MP)

# ---- soil: subplot -> plot means, by depth ----------------------------------
s  = to_df(os.path.join(GDB,"soil_samples"))
BULK = [c for c in ["soil_C_gm2","soil_N_gm2","soil_CN","fine_root_gm2","BD_gcm3"] if c in s.columns]
sd = s.groupby(["watershed","soil_plot","depth_code"], as_index=False)[BULK].mean()
sd["trt"] = sd["watershed"].map(MP)

# ---- fractions; light-fraction MASS is complete where LF %C is censored -----
fr = to_df(os.path.join(GDB,"soil_fractions"))
fr["trt"] = fr["watershed"].map(MP)
bd  = sd.set_index(["watershed","soil_plot","depth_code"])["BD_gcm3"]
idx = pd.MultiIndex.from_arrays([fr["watershed"], fr["soil_plot"], fr["depth_code"]])
fr["lf_mass_gm2"] = (fr["LF_g"]/fr["frac_soil_wt_g"]) * bd.reindex(idx).values * 20 * 1e4
for d in (sd, fr):
    d["sub"]       = d["soil_plot"].map(SUB)
    d["tsh"]       = d["sub"].map(TSH)
    d["intensity"] = d["sub"].map(INT)
    d["cum_harv"]  = d["sub"].map(CUM)
    d["n_harv"]    = d["sub"].map(NH)
lf = fr[fr["depth_code"]=="20-40"].copy()

# ---- depth-distribution metrics, one row per soil plot ----------------------
pv = sd.pivot_table(index=["watershed","trt","soil_plot","sub","tsh","intensity"],
                    columns="depth_code", values=["soil_C_gm2","soil_N_gm2","fine_root_gm2"]).reset_index()
pv.columns = ["%s_%s"%(a,b) if b else a for a,b in pv.columns]
pv["frac_subsoil"]      = pv["soil_C_gm2_20-40"]/(pv["soil_C_gm2_0-20"]+pv["soil_C_gm2_20-40"])
pv["frac_root_subsoil"] = pv["fine_root_gm2_20-40"]/(pv["fine_root_gm2_0-20"]+pv["fine_root_gm2_20-40"])
pv["rootC"]      = (pv["fine_root_gm2_0-20"]+pv["fine_root_gm2_20-40"]+pv["fine_root_gm2_OH"].fillna(0))*CF/100
pv["soilC_0_40"] = (pv["soil_C_gm2_0-20"]+pv["soil_C_gm2_20-40"])/100

# ---- helpers ---------------------------------------------------------------
def gg(df, col, dep=None):
    out = []
    for t in O:
        x = df[df["trt"]==t]
        if dep is not None: x = x[x["depth_code"]==dep]
        out.append(x[col].dropna().values)
    return out

def perm_p(groups, n=20000):
    obs = stats.f_oneway(*groups)[0]
    pool = np.concatenate(groups); sizes = [len(g) for g in groups]; c = 0
    for _ in range(n):
        x = rng.permutation(pool); k = 0; parts = []
        for sz in sizes:
            parts.append(x[k:k+sz]); k += sz
        if stats.f_oneway(*parts)[0] >= obs: c += 1
    return (c+1)/(n+1)

def dunnett_p(groups):
    try:
        return list(stats.dunnett(groups[1], groups[2], control=groups[0]).pvalue)
    except AttributeError:
        p = [stats.ttest_ind(groups[i], groups[0], equal_var=False)[1] for i in (1,2)]
        return [min(1, 2*min(p)) if q == min(p) else min(1, q) for q in p]

def hedges_g(a, b):
    n1, n2 = len(a), len(b)
    sp = np.sqrt(((n1-1)*a.var(ddof=1) + (n2-1)*b.var(ddof=1))/(n1+n2-2))
    return (a.mean()-b.mean())/sp * (1 - 3/(4*(n1+n2)-9))

print("stand table:", m.shape, "| soil plot x depth:", sd.shape,
      "| fractions:", fr.shape, "| depth metrics:", pv.shape)
print("subsoil LF mass by treatment (g/m2):",
      {t: round(float(x.mean())) for t, x in zip(O, gg(lf,"lf_mass_gm2"))})

<class 'NameError'>: name 'to_df' is not defined

In [2]:
# ============================================================================
# X0  SETUP - run this first in any new session (kernel resets clear everything)
# ============================================================================
import arcpy, os, numpy as np, pandas as pd
arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False
aprx = arcpy.mp.ArcGISProject("CURRENT")
PROJ = os.path.dirname(aprx.filePath)
GDB  = os.path.join(PROJ, "Fernow_SOM_master.gdb")
SR   = arcpy.SpatialReference(26917)

def to_df(fc, xy=False):
    f = [x.name for x in arcpy.ListFields(fc)
         if x.type not in ("Geometry","OID","Blob","Raster","GlobalID")]
    q = f + (["SHAPE@X","SHAPE@Y"] if xy else [])
    return pd.DataFrame(arcpy.da.SearchCursor(fc, q), columns=q)

def _ftype(s):
    if pd.api.types.is_bool_dtype(s): return "SHORT"
    if pd.api.types.is_integer_dtype(s): return "LONG"
    if pd.api.types.is_float_dtype(s): return "DOUBLE"
    return "TEXT"

def write_df(df, name, xy=None):
    out = os.path.join(GDB, name)
    if arcpy.Exists(out): arcpy.management.Delete(out)
    if xy: arcpy.management.CreateFeatureclass(GDB, name, "POINT", spatial_reference=SR)
    else:  arcpy.management.CreateTable(GDB, name)
    cols, names = list(df.columns), []
    for c in cols:
        fn, ft = arcpy.ValidateFieldName(c, GDB), _ftype(df[c])
        L = max(16, int(df[c].astype(str).str.len().max()) + 8) if ft == "TEXT" else None
        arcpy.management.AddField(out, fn, ft, field_length=L); names.append(fn)
    def clean(v):
        if v is None or (not isinstance(v, str) and pd.isna(v)): return None
        if isinstance(v, (bool, np.bool_)): return int(v)
        return v.item() if isinstance(v, np.generic) else v
    with arcpy.da.InsertCursor(out, names + (["SHAPE@XY"] if xy else [])) as cur:
        for _, r in df.iterrows():
            row = [clean(r[c]) for c in cols]
            if xy: row.append((float(r[xy[0]]), float(r[xy[1]])))
            cur.insertRow(row)
    print(f"{name}: {arcpy.management.GetCount(out)[0]} rows, {len(names)} fields")

print("PROJ:", PROJ)
print("gdb exists:", arcpy.Exists(GDB),
      "| plots:", arcpy.management.GetCount(os.path.join(GDB,"plots_study"))[0],
      "| trees:", arcpy.management.GetCount(os.path.join(GDB,"trees"))[0])

PROJ: C:\Users\DCifelli2\OneDrive - mercercounty.org\Documents\ArcGIS\Projects\Fernow-FM-SOM-DC-092026
gdb exists: True | plots: 18 | trees: 712


In [3]:
# ============================================================================
# X1  EXPLORATORY ANALYSIS - build analysis frames from the master geodatabase
# ============================================================================
import os, numpy as np, pandas as pd
from scipy import stats
rng = np.random.default_rng(42)
EXP, CF = 1/0.0404686, 0.48
O   = ["REF","DLC","STS"]
MP  = {"WS4":"REF","WS2":"DLC","WS5":"STS"}
COL = {"REF":"#1b5e3f","DLC":"#c1553b","STS":"#4a6fa5"}
LAB = {"REF":"REF\n(uncut)","DLC":"DLC\n(2 yr)","STS":"STS\n(4 yr)"}

# harvest history reconstructed from ws2/ws4/ws5_basal_area.xlsx (1958-2020)
SUB  = dict([(p,"WS2A") for p in (7,8,10,11,12)] + [(9,"WS2B")] +
            [(p,"WS4A") for p in (13,14,15,16,17,18)] +
            [(p,"WS5A") for p in (19,20,21,22,23)] + [(24,"WS5B")])
TSH  = {"WS2A":2,"WS2B":5,"WS4A":112,"WS4B":112,"WS4C":112,"WS5A":4,"WS5B":10}
INT  = {"WS2A":41.2,"WS2B":32.2,"WS4A":0.0,"WS4B":0.0,"WS4C":0.0,"WS5A":25.0,"WS5B":37.1}
CUM  = {"WS2A":160.9,"WS2B":110.2,"WS4A":0.0,"WS4B":0.0,"WS4C":0.0,"WS5A":149.9,"WS5B":141.3}
NH   = {"WS2A":5,"WS2B":4,"WS4A":0,"WS4B":0,"WS4C":0,"WS5A":8,"WS5B":5}

# ---- stand structure, one row per plot --------------------------------------
tr = to_df(os.path.join(GDB,"trees")); v = tr.dropna(subset=["dbh_cm"])
rows = []
for pid, g in v.groupby("plot_id"):
    ba = g["ba_m2"].sum()
    rows.append({"plot_id":pid,"agbC_Mgha":g["agb_kg_jenkins"].sum()/1000*EXP*CF,
                 "ba_m2ha":ba*EXP,"stems_ha":len(g)*EXP,
                 "qmd_cm":np.sqrt((g["dbh_cm"]**2).mean()),"dbhmax_cm":g["dbh_cm"].max(),
                 "agb_share_ge50cm":g.loc[g["dbh_cm"]>=50,"agb_kg_jenkins"].sum()/g["agb_kg_jenkins"].sum()})
st = pd.DataFrame(rows)
P_ = to_df(os.path.join(GDB,"plots_study"))
m  = P_[["plot_id","watershed","treatment","subcomp_mapped","site_index_mapped"]].merge(st, on="plot_id")
for t in ["plot_terrain","plot_canopy","plot_solar"]:
    m = m.merge(to_df(os.path.join(GDB,t)), on="plot_id", how="left")
m["trt"] = m["watershed"].map(MP)

# ---- soil: subplot -> plot means, by depth ----------------------------------
s  = to_df(os.path.join(GDB,"soil_samples"))
BULK = [c for c in ["soil_C_gm2","soil_N_gm2","soil_CN","fine_root_gm2","BD_gcm3"] if c in s.columns]
sd = s.groupby(["watershed","soil_plot","depth_code"], as_index=False)[BULK].mean()
sd["trt"] = sd["watershed"].map(MP)

# ---- fractions; light-fraction MASS is complete where LF %C is censored -----
fr = to_df(os.path.join(GDB,"soil_fractions"))
fr["trt"] = fr["watershed"].map(MP)
bd  = sd.set_index(["watershed","soil_plot","depth_code"])["BD_gcm3"]
idx = pd.MultiIndex.from_arrays([fr["watershed"], fr["soil_plot"], fr["depth_code"]])
fr["lf_mass_gm2"] = (fr["LF_g"]/fr["frac_soil_wt_g"]) * bd.reindex(idx).values * 20 * 1e4
for d in (sd, fr):
    d["sub"]       = d["soil_plot"].map(SUB)
    d["tsh"]       = d["sub"].map(TSH)
    d["intensity"] = d["sub"].map(INT)
    d["cum_harv"]  = d["sub"].map(CUM)
    d["n_harv"]    = d["sub"].map(NH)
lf = fr[fr["depth_code"]=="20-40"].copy()

# ---- depth-distribution metrics, one row per soil plot ----------------------
pv = sd.pivot_table(index=["watershed","trt","soil_plot","sub","tsh","intensity"],
                    columns="depth_code", values=["soil_C_gm2","soil_N_gm2","fine_root_gm2"]).reset_index()
pv.columns = ["%s_%s"%(a,b) if b else a for a,b in pv.columns]
pv["frac_subsoil"]      = pv["soil_C_gm2_20-40"]/(pv["soil_C_gm2_0-20"]+pv["soil_C_gm2_20-40"])
pv["frac_root_subsoil"] = pv["fine_root_gm2_20-40"]/(pv["fine_root_gm2_0-20"]+pv["fine_root_gm2_20-40"])
pv["rootC"]      = (pv["fine_root_gm2_0-20"]+pv["fine_root_gm2_20-40"]+pv["fine_root_gm2_OH"].fillna(0))*CF/100
pv["soilC_0_40"] = (pv["soil_C_gm2_0-20"]+pv["soil_C_gm2_20-40"])/100

# ---- helpers ---------------------------------------------------------------
def gg(df, col, dep=None):
    out = []
    for t in O:
        x = df[df["trt"]==t]
        if dep is not None: x = x[x["depth_code"]==dep]
        out.append(x[col].dropna().values)
    return out

def perm_p(groups, n=20000):
    obs = stats.f_oneway(*groups)[0]
    pool = np.concatenate(groups); sizes = [len(g) for g in groups]; c = 0
    for _ in range(n):
        x = rng.permutation(pool); k = 0; parts = []
        for sz in sizes:
            parts.append(x[k:k+sz]); k += sz
        if stats.f_oneway(*parts)[0] >= obs: c += 1
    return (c+1)/(n+1)

def dunnett_p(groups):
    try:
        return list(stats.dunnett(groups[1], groups[2], control=groups[0]).pvalue)
    except AttributeError:
        p = [stats.ttest_ind(groups[i], groups[0], equal_var=False)[1] for i in (1,2)]
        return [min(1, 2*min(p)) if q == min(p) else min(1, q) for q in p]

def hedges_g(a, b):
    n1, n2 = len(a), len(b)
    sp = np.sqrt(((n1-1)*a.var(ddof=1) + (n2-1)*b.var(ddof=1))/(n1+n2-2))
    return (a.mean()-b.mean())/sp * (1 - 3/(4*(n1+n2)-9))

print("stand table:", m.shape, "| soil plot x depth:", sd.shape,
      "| fractions:", fr.shape, "| depth metrics:", pv.shape)
print("subsoil LF mass by treatment (g/m2):",
      {t: round(float(x.mean())) for t, x in zip(O, gg(lf,"lf_mass_gm2"))})

stand table: (18, 40) | soil plot x depth: (54, 14) | fractions: (36, 41) | depth metrics: (18, 19)
subsoil LF mass by treatment (g/m2): {'REF': 1155, 'DLC': 3056, 'STS': 1317}


In [4]:
# ==========================================================================
#  EXPLORATORY ANALYSIS CELLS - paste each block into a separate ArcGIS Pro
#  notebook cell, AFTER your setup cell (which defines GDB, PROJ, to_df).
# ==========================================================================

# ---------------------------- CELL X1 --------------------------------------
# ============================================================================
# X1  EXPLORATORY ANALYSIS - build analysis frames from the master geodatabase
# ============================================================================
import os, numpy as np, pandas as pd
from scipy import stats
rng = np.random.default_rng(42)
EXP, CF = 1/0.0404686, 0.48
O   = ["REF","DLC","STS"]
MP  = {"WS4":"REF","WS2":"DLC","WS5":"STS"}
COL = {"REF":"#1b5e3f","DLC":"#c1553b","STS":"#4a6fa5"}
LAB = {"REF":"REF\n(uncut)","DLC":"DLC\n(2 yr)","STS":"STS\n(4 yr)"}

# harvest history reconstructed from ws2/ws4/ws5_basal_area.xlsx (1958-2020)
SUB  = dict([(p,"WS2A") for p in (7,8,10,11,12)] + [(9,"WS2B")] +
            [(p,"WS4A") for p in (13,14,15,16,17,18)] +
            [(p,"WS5A") for p in (19,20,21,22,23)] + [(24,"WS5B")])
TSH  = {"WS2A":2,"WS2B":5,"WS4A":112,"WS4B":112,"WS4C":112,"WS5A":4,"WS5B":10}
INT  = {"WS2A":41.2,"WS2B":32.2,"WS4A":0.0,"WS4B":0.0,"WS4C":0.0,"WS5A":25.0,"WS5B":37.1}
CUM  = {"WS2A":160.9,"WS2B":110.2,"WS4A":0.0,"WS4B":0.0,"WS4C":0.0,"WS5A":149.9,"WS5B":141.3}
NH   = {"WS2A":5,"WS2B":4,"WS4A":0,"WS4B":0,"WS4C":0,"WS5A":8,"WS5B":5}

# ---- stand structure, one row per plot --------------------------------------
tr = to_df(os.path.join(GDB,"trees")); v = tr.dropna(subset=["dbh_cm"])
rows = []
for pid, g in v.groupby("plot_id"):
    ba = g["ba_m2"].sum()
    rows.append({"plot_id":pid,"agbC_Mgha":g["agb_kg_jenkins"].sum()/1000*EXP*CF,
                 "ba_m2ha":ba*EXP,"stems_ha":len(g)*EXP,
                 "qmd_cm":np.sqrt((g["dbh_cm"]**2).mean()),"dbhmax_cm":g["dbh_cm"].max(),
                 "agb_share_ge50cm":g.loc[g["dbh_cm"]>=50,"agb_kg_jenkins"].sum()/g["agb_kg_jenkins"].sum()}) 
st = pd.DataFrame(rows)
P_ = to_df(os.path.join(GDB,"plots_study"))
m  = P_[["plot_id","watershed","treatment","subcomp_mapped","site_index_mapped"]].merge(st, on="plot_id")
for t in ["plot_terrain","plot_canopy","plot_solar"]:
    d = to_df(os.path.join(GDB,t))
    m = m.merge(d, on="plot_id", how="left")
m["trt"] = m["watershed"].map(MP)

# ---- soil: subplot -> plot means, by depth ----------------------------------
s  = to_df(os.path.join(GDB,"soil_samples"))
BULK = [c for c in ["soil_C_gm2","soil_N_gm2","soil_CN","fine_root_gm2","BD_gcm3"] if c in s.columns]
sd = s.groupby(["watershed","soil_plot","depth_code"], as_index=False)[BULK].mean()
sd["trt"] = sd["watershed"].map(MP)

# ---- fractions; light-fraction MASS is complete where LF %C is censored -----
fr = to_df(os.path.join(GDB,"soil_fractions"))
fr["trt"] = fr["watershed"].map(MP)
bd = sd.set_index(["watershed","soil_plot","depth_code"])["BD_gcm3"]
idx = pd.MultiIndex.from_arrays([fr["watershed"], fr["soil_plot"], fr["depth_code"]])
fr["lf_mass_gm2"] = (fr["LF_g"]/fr["frac_soil_wt_g"]) * bd.reindex(idx).values * 20 * 1e4
for d in (sd, fr):
    d["sub"]       = d["soil_plot"].map(SUB)
    d["tsh"]       = d["sub"].map(TSH)
    d["intensity"] = d["sub"].map(INT)
    d["cum_harv"]  = d["sub"].map(CUM)
    d["n_harv"]    = d["sub"].map(NH)
lf = fr[fr["depth_code"]=="20-40"].copy()

# ---- depth-distribution metrics, one row per soil plot ----------------------
pv = sd.pivot_table(index=["watershed","trt","soil_plot","sub","tsh","intensity"],
                    columns="depth_code", values=["soil_C_gm2","soil_N_gm2","fine_root_gm2"]).reset_index()
pv.columns = ["%s_%s"%(a,b) if b else a for a,b in pv.columns]
pv["frac_subsoil"]      = pv["soil_C_gm2_20-40"]/(pv["soil_C_gm2_0-20"]+pv["soil_C_gm2_20-40"])
pv["frac_root_subsoil"] = pv["fine_root_gm2_20-40"]/(pv["fine_root_gm2_0-20"]+pv["fine_root_gm2_20-40"])
pv["rootC"]  = (pv["fine_root_gm2_0-20"]+pv["fine_root_gm2_20-40"]+pv["fine_root_gm2_OH"].fillna(0))*CF/100
pv["soilC_0_40"] = (pv["soil_C_gm2_0-20"]+pv["soil_C_gm2_20-40"])/100

# ---- helpers ---------------------------------------------------------------
def gg(df, col, dep=None):
    """values by treatment, in REF/DLC/STS order"""
    out = []
    for t in O:
        x = df[df["trt"]==t]
        if dep is not None: x = x[x["depth_code"]==dep]
        out.append(x[col].dropna().values)
    return out

def perm_p(groups, n=20000):
    """permutation P for one-way difference; exact-style, assumes nothing"""
    obs = stats.f_oneway(*groups)[0]
    pool = np.concatenate(groups); sizes = [len(g) for g in groups]; c = 0
    for _ in range(n):
        x = rng.permutation(pool); k = 0; parts = []
        for sz in sizes:
            parts.append(x[k:k+sz]); k += sz
        if stats.f_oneway(*parts)[0] >= obs: c += 1
    return (c+1)/(n+1)

def dunnett_p(groups):
    """Dunnett vs REF; falls back to Holm-adjusted Welch t if scipy is older"""
    try:
        return list(stats.dunnett(groups[1], groups[2], control=groups[0]).pvalue)
    except AttributeError:
        p = [stats.ttest_ind(groups[i], groups[0], equal_var=False)[1] for i in (1,2)]
        return [min(1, 2*min(p)) if q == min(p) else min(1, q) for q in p]

def hedges_g(a, b):
    n1, n2 = len(a), len(b)
    sp = np.sqrt(((n1-1)*a.var(ddof=1) + (n2-1)*b.var(ddof=1))/(n1+n2-2))
    return (a.mean()-b.mean())/sp * (1 - 3/(4*(n1+n2)-9))

print("stand table:", m.shape, "| soil plot x depth:", sd.shape,
      "| fractions:", fr.shape, "| depth metrics:", pv.shape)
print("subsoil LF mass by treatment (g/m2):",
      {t: round(float(x.mean())) for t, x in zip(O, gg(lf,"lf_mass_gm2"))})

# ---------------------------- CELL X2 --------------------------------------
# ============================================================================
# X2  EXPLORATORY TESTS
# ============================================================================
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf, statsmodels.api as sm
pd.set_option("display.width", 250)

print("="*96)
print("1  HARVEST TIMING: is subsoil particulate C a decaying slash pulse?")
print("="*96)
print(lf[["trt","soil_plot","sub","tsh","intensity","lf_mass_gm2","LF_C_gm2","hPOM_C_gm2","MAOM_C_gm2"]]
      .sort_values(["trt","tsh"]).round(1).to_string(index=False))
for t in ("DLC","STS"):
    g  = lf[lf["trt"]==t]
    rc = g[g["tsh"]==g["tsh"].min()]; ol = g[g["tsh"]==g["tsh"].max()]
    rank = int((g["lf_mass_gm2"] < ol["lf_mass_gm2"].iloc[0]).sum()) + 1
    print("\n  %s: %d plots at %d yr = %.0f g/m2 | 1 plot at %d yr = %.0f g/m2 | older plot ranks %d of %d"
          % (t, len(rc), rc["tsh"].iloc[0], rc["lf_mass_gm2"].mean(),
             ol["tsh"].iloc[0], ol["lf_mass_gm2"].iloc[0], rank, len(g)))
print("\n  (a decaying pulse predicts the older plot ranks LOWEST)")
h = lf[lf["trt"]!="REF"]
for x, lab, d in [("tsh","years since harvest (12 harvested plots)",h),
                  ("intensity","removal intensity per entry (12 harvested plots)",h),
                  ("cum_harv","cumulative harvested C, 1958-2020 (18 plots)",lf),
                  ("n_harv","number of harvest entries (18 plots)",lf)]:
    r, p = stats.pearsonr(d[x], d["lf_mass_gm2"])
    print("  LF mass vs %-52s r = %+.2f, P = %.3f" % (lab, r, p))

print("\n"+"="*96)
print("2  PERMANOVA: does SOM composition differ where single pools do not?")
print("="*96)
def permanova(X, labels, n=9999):
    X = np.asarray(X, float); labels = np.asarray(labels)
    Z = (X - X.mean(0))/X.std(0, ddof=1)
    D = np.sqrt(((Z[:,None,:]-Z[None,:,:])**2).sum(-1))
    N = len(Z); grps = np.unique(labels)
    def F(lb):
        SST = (D**2).sum()/(2*N)
        SSW = sum((D[np.ix_(np.where(lb==g)[0], np.where(lb==g)[0])]**2).sum()/(2*(lb==g).sum()) for g in grps)
        SSA = SST - SSW
        return (SSA/(len(grps)-1))/(SSW/(N-len(grps))), SSA/SST
    obs, R2 = F(labels)
    c = sum(1 for _ in range(n) if F(rng.permutation(labels))[0] >= obs)
    return obs, R2, (c+1)/(n+1)
def permdisp(X, labels, n=9999):
    X = np.asarray(X, float); labels = np.asarray(labels)
    Z = (X - X.mean(0))/X.std(0, ddof=1)
    d, lb = [], []
    for g in np.unique(labels):
        Zi = Z[labels==g]; d.append(np.sqrt(((Zi-Zi.mean(0))**2).sum(1))); lb.append(labels[labels==g])
    d = np.concatenate(d); lb = np.concatenate(lb)
    obs = stats.f_oneway(*[d[lb==g] for g in np.unique(lb)])[0]
    c = sum(1 for _ in range(n) if stats.f_oneway(*[rng.permutation(d)[lb==g] for g in np.unique(lb)])[0] >= obs)
    return (c+1)/(n+1)
for dep in ("0-20","20-40"):
    d = fr[fr["depth_code"]==dep]
    for cols, lab in [(["LF_C_gm2","hPOM_C_gm2","MAOM_C_gm2"], "C fractions"),
                      (["LF_C_gm2","hPOM_C_gm2","MAOM_C_gm2","LF_N_gm2","hPOM_N_gm2","MAOM_N_gm2"], "C + N fractions")]:
        k = d.dropna(subset=cols)
        if k.groupby("trt").size().min() < 3: continue
        F_, R2, p = permanova(k[cols], k["trt"])
        print("  %-6s %-18s n=%2d  pseudo-F=%5.2f  R2=%.2f  P=%.4f  | dispersion P=%.3f"
              % (dep, lab, len(k), F_, R2, p, permdisp(k[cols], k["trt"])))
k = lf.dropna(subset=["lf_mass_gm2"])
cols = ["lf_mass_gm2","hPOM_C_gm2","MAOM_C_gm2"]
F_, R2, p = permanova(k[cols], k["trt"])
print("  20-40  LF MASS + hPOM + MAOM  n=%d  pseudo-F=%5.2f  R2=%.2f  P=%.4f  | dispersion P=%.3f"
      % (len(k), F_, R2, p, permdisp(k[cols], k["trt"])))
for a, b in (("REF","DLC"),("REF","STS"),("DLC","STS")):
    kk = k[k["trt"].isin([a,b])]
    F_, R2, p = permanova(kk[cols], kk["trt"], n=4999)
    print("     pairwise %s vs %s: pseudo-F=%5.2f  R2=%.2f  P=%.4f" % (a, b, F_, R2, p))
print("  NOTE a significant dispersion test means the result is partly unequal SPREAD, not location")

print("\n"+"="*96)
print("3  DEPTH DISTRIBUTION AND ROOT ALLOCATION")
print("="*96)
for c, lab in [("frac_subsoil","soil C below 20 cm (fraction of 0-40)"),
               ("frac_root_subsoil","fine roots below 20 cm (fraction of 0-40)")]:
    g = gg(pv, c); du = dunnett_p(g)
    print("  %-40s REF=%.3f DLC=%.3f STS=%.3f | ANOVA P=%.4f perm P=%.4f KW P=%.4f | Dunnett %.4f / %.4f"
          % (lab, g[0].mean(), g[1].mean(), g[2].mean(), stats.f_oneway(*g)[1], perm_p(g),
             stats.kruskal(*g)[1], du[0], du[1]))
    if c == "frac_root_subsoil":
        for t, x in zip(("DLC","STS"), (g[1], g[2])):
            bs = [hedges_g(rng.choice(x,len(x)), rng.choice(g[0],len(g[0]))) for _ in range(10000)]
            print("      %s vs REF: Hedges' g = %+.2f (95%% CI %+.2f to %+.2f)"
                  % (t, hedges_g(x,g[0]), np.percentile(bs,2.5), np.percentile(bs,97.5)))
g = gg(pv,"rootC")
print("  %-40s REF=%.2f DLC=%.2f STS=%.2f | ANOVA P=%.4f perm P=%.4f"
      % ("total fine-root C (Mg C/ha)", g[0].mean(), g[1].mean(), g[2].mean(), stats.f_oneway(*g)[1], perm_p(g)))
print("  root:shoot ratio is DESCRIPTIVE ONLY - every plot in a watershed shares one")
print("  aboveground denominator, so a test on the ratio has no plot-level error term:")
for t in O:
    print("      %s: root C %.2f / AGB C %.0f  ->  %.4f"
          % (t, pv.loc[pv["trt"]==t,"rootC"].mean(), m.loc[m["trt"]==t,"agbC_Mgha"].mean(),
             pv.loc[pv["trt"]==t,"rootC"].mean()/m.loc[m["trt"]==t,"agbC_Mgha"].mean()))

print("\n"+"="*96)
print("4  IS THE SUBSOIL SIGNAL MEDIATED BY ROOTS?")
print("="*96)
j = lf.merge(pv[["soil_plot","frac_root_subsoil","rootC"]], on="soil_plot")
j = j.merge(sd[sd["depth_code"]=="20-40"][["soil_plot","fine_root_gm2","soil_C_gm2"]], on="soil_plot")
for x, lab in [("fine_root_gm2","subsoil fine-root biomass"),("frac_root_subsoil","subsoil root fraction"),
               ("soil_C_gm2","subsoil bulk soil C")]:
    r, p = stats.pearsonr(j[x], j["lf_mass_gm2"]); rs, ps = stats.spearmanr(j[x], j["lf_mass_gm2"])
    print("  LF mass vs %-28s r=%+.2f P=%.3f | Spearman rho=%+.2f P=%.3f" % (lab, r, p, rs, ps))
f1 = smf.ols("lf_mass_gm2 ~ C(trt) + fine_root_gm2", j).fit(); a1 = sm.stats.anova_lm(f1, typ=2)
print("  model LF mass ~ treatment + subsoil roots: treatment P=%.4f, roots P=%.4f, R2=%.2f"
      % (a1.loc["C(trt)","PR(>F)"], a1.loc["fine_root_gm2","PR(>F)"], f1.rsquared))

print("\n"+"="*96)
print("5  FDR CORRECTION ACROSS ALL EXPLORATORY TESTS")
print("="*96)
res = []
def add(lab, g):
    du = dunnett_p(g)
    res.append({"test":lab,"REF":g[0].mean(),"DLC":g[1].mean(),"STS":g[2].mean(),
                "P":stats.f_oneway(*g)[1],"KW":stats.kruskal(*g)[1],"D_DLC":du[0],"D_STS":du[1]})
for dep in ("0-20","20-40"):
    d = fr[fr["depth_code"]==dep].copy()
    d["MAOM_CN"] = d["MAOM_C_gm2"]/d["MAOM_N_gm2"]
    d["hPOM_CN"] = d["hPOM_C_gm2"]/d["hPOM_N_gm2"]
    for c, lab in [("MAOM_CN","MAOM C:N"),("hPOM_CN","heavy POM C:N"),
                   ("MAOM_C_gm2","MAOM C"),("hPOM_C_gm2","heavy POM C")]:
        g = gg(d, c)
        if min(len(x) for x in g) >= 3: add("%s %s" % (dep, lab), g)
add("20-40 LF mass",           gg(lf,"lf_mass_gm2"))
add("subsoil root fraction",   gg(pv,"frac_root_subsoil"))
add("subsoil soil-C fraction", gg(pv,"frac_subsoil"))
add("fine roots 20-40 cm",     gg(sd,"fine_root_gm2","20-40"))
add("total fine-root C",       gg(pv,"rootC"))
R = pd.DataFrame(res)
R["q_FDR"] = multipletests(R["P"], method="fdr_bh")[1]
print(R.round(4).to_string(index=False))
print("\n  q < 0.05:", R.loc[R["q_FDR"]<0.05,"test"].tolist())
print("  q < 0.10:", R.loc[R["q_FDR"]<0.10,"test"].tolist())
R.to_csv(os.path.join(PROJ,"exploratory_results.csv"), index=False)
print("\n  saved:", os.path.join(PROJ,"exploratory_results.csv"))

# ---------------------------- CELL X3 --------------------------------------
# ============================================================================
# X3  EXPLORATORY FIGURE  (saves PNG + PDF next to the project)
# ============================================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size":8,"axes.linewidth":0.7,"axes.spines.top":False,
    "axes.spines.right":False,"legend.frameon":False,"xtick.labelsize":7.5,
    "ytick.labelsize":7.5,"legend.fontsize":7.5,"savefig.dpi":400,"pdf.fonttype":42})

def dotbar(ax, groups, ylab, ptxt=None, ppos="right"):
    for i, (t, v) in enumerate(zip(O, groups)):
        ax.bar(i, v.mean(), 0.62, color=COL[t], alpha=0.85, yerr=stats.sem(v),
               capsize=3, ecolor="0.25", error_kw={"lw":0.8, "zorder":4}, zorder=1)
        ax.scatter(i + rng.uniform(-0.15, 0.15, len(v)), v, s=11, facecolor="white",
                   edgecolor="0.3", lw=0.5, zorder=3)
    ax.set_xticks(range(3)); ax.set_xticklabels([LAB[t] for t in O]); ax.set_ylabel(ylab)
    if ptxt:
        x, ha = (0.97, "right") if ppos == "right" else (0.03, "left")
        ax.text(x, 0.96, ptxt, transform=ax.transAxes, ha=ha, va="top", fontsize=7.5)
def tag(ax, L, dx=-0.22):
    ax.text(dx, 1.07, L, transform=ax.transAxes, fontsize=10, fontweight="bold", va="top")

fig = plt.figure(figsize=(7.2, 4.9))
gs  = fig.add_gridspec(2, 3, hspace=0.46, wspace=0.42, left=0.085, right=0.985, top=0.93, bottom=0.11)

# a  subsoil light-fraction mass (complete data)
ax = fig.add_subplot(gs[0,0]); tag(ax,"a")
g  = gg(lf, "lf_mass_gm2")
dotbar(ax, g, "Light-fraction mass, 20–40 cm\n(g m$^{-2}$)", "$P$ = 0.003", "left")
ax.text(1, 3900, "2.6× REF", ha="center", fontsize=7, color=COL["DLC"], fontweight="bold")
ax.set_ylim(0, 5200)

# b  fine roots below 20 cm  (NEW)
ax = fig.add_subplot(gs[0,1]); tag(ax,"b")
g  = gg(pv, "frac_root_subsoil")
dotbar(ax, g, "Fine roots below 20 cm\n(fraction of 0–40 cm)", "$P$ = 0.017", "left")
ax.set_ylim(0, 0.62)
ax.annotate("", xy=(1,0.50), xytext=(0,0.50), arrowprops=dict(arrowstyle="-", lw=0.7, color="0.4"))
ax.text(0.5, 0.515, "both harvests ≈ 2× REF", ha="center", fontsize=6.5, color="0.35")

# c  total fine-root C is unchanged
ax = fig.add_subplot(gs[0,2]); tag(ax,"c")
dotbar(ax, gg(pv,"rootC"), "Total fine-root C, 0–40 cm\n(Mg C ha$^{-1}$)", "$P$ = 0.47", "left")
ax.set_ylim(0, 5)

# d  MAOM C:N  (NEW)
ax = fig.add_subplot(gs[1,0]); tag(ax,"d")
wd = 0.26
for k, t in enumerate(O):
    mu, se = [], []
    for dep in ("0-20","20-40"):
        d = fr[(fr["trt"]==t) & (fr["depth_code"]==dep)]
        r = (d["MAOM_C_gm2"]/d["MAOM_N_gm2"]).dropna()
        mu.append(r.mean()); se.append(stats.sem(r))
    ax.bar(np.arange(2)+(k-1)*wd, mu, wd*0.92, yerr=se, color=COL[t], alpha=0.85,
           capsize=2, ecolor="0.3", error_kw={"lw":0.7}, label=t)
ax.set_xticks([0,1]); ax.set_xticklabels(["0–20 cm","20–40 cm"])
ax.set_ylabel("MAOM C:N"); ax.set_ylim(0,21)
ax.text(0, 17.6, "$P$ = 0.022", ha="center", fontsize=7)
ax.text(1, 17.6, "$P$ = 0.044", ha="center", fontsize=7)
ax.legend(loc="upper left", bbox_to_anchor=(-0.02,1.02), handlelength=1.1, borderpad=0.2, labelspacing=0.25, ncol=3, columnspacing=0.7, handletextpad=0.4)

# e  timing hypothesis fails
ax = fig.add_subplot(gs[1,1]); tag(ax,"e")
for t in ("DLC","STS"):
    d = lf[lf["trt"]==t]
    ax.scatter(d["tsh"], d["lf_mass_gm2"], s=22, color=COL[t], edgecolor="white", lw=0.4, label=t, zorder=3)
d = lf[lf["trt"]=="REF"]
ax.scatter([13]*len(d), d["lf_mass_gm2"], s=22, color=COL["REF"], edgecolor="white", lw=0.4,
           label="REF (uncut, plotted at 13)", zorder=3)
ax.set_xlabel("Years since last harvest"); ax.set_ylabel("Light-fraction mass, 20–40 cm\n(g m$^{-2}$)")
ax.set_xlim(0, 14.5); ax.set_ylim(0, 5200)
ax.text(0.97, 0.96, "harvested plots only:\n$r$ = −0.22, $P$ = 0.49", transform=ax.transAxes,
        ha="right", va="top", fontsize=6.8)
ax.legend(loc="lower left", fontsize=6.3, handlelength=0.8, borderpad=0.2, labelspacing=0.2)

# f  removal intensity is the better predictor
ax = fig.add_subplot(gs[1,2]); tag(ax,"f")
for t in O:
    d = lf[lf["trt"]==t]
    ax.scatter(d["intensity"], d["lf_mass_gm2"], s=22, color=COL[t], edgecolor="white", lw=0.4, zorder=3)
h = lf[lf["trt"]!="REF"]
b = np.polyfit(h["intensity"], h["lf_mass_gm2"], 1)
xx = np.linspace(h["intensity"].min(), h["intensity"].max(), 20)
ax.plot(xx, np.polyval(b, xx), "--", color="0.45", lw=1.0, zorder=2)
ax.set_xlabel("C removed per harvest entry (%)"); ax.set_ylabel("Light-fraction mass, 20–40 cm\n(g m$^{-2}$)")
ax.set_ylim(0, 5200)
ax.text(0.97, 0.96, "harvested plots only:\n$r$ = +0.49, $P$ = 0.11", transform=ax.transAxes,
        ha="right", va="top", fontsize=6.8)

for ext in ("png","pdf"):
    fig.savefig(os.path.join(PROJ, "Fig6_exploratory."+ext), bbox_inches="tight")
print("saved:", os.path.join(PROJ, "Fig6_exploratory.png"))

stand table: (18, 40) | soil plot x depth: (54, 14) | fractions: (36, 41) | depth metrics: (18, 19)
subsoil LF mass by treatment (g/m2): {'REF': 1155, 'DLC': 3056, 'STS': 1317}
1  HARVEST TIMING: is subsoil particulate C a decaying slash pulse?
trt  soil_plot  sub  tsh  intensity  lf_mass_gm2  LF_C_gm2  hPOM_C_gm2  MAOM_C_gm2
DLC          7 WS2A    2       41.2       3759.5     954.9       158.1      1068.5
DLC          8 WS2A    2       41.2       4795.4    1628.5       281.7      2171.1
DLC         10 WS2A    2       41.2       1532.4     645.6       221.5      1700.2
DLC         11 WS2A    2       41.2       2391.1     917.2       225.3      1157.9
DLC         12 WS2A    2       41.2       1502.6     605.9       137.9      1377.2
DLC          9 WS2B    5       32.2       4356.4    1306.0      2114.2      2017.1
REF         13 WS4A  112        0.0       1010.0     436.7       180.4      1040.2
REF         14 WS4A  112        0.0       1534.0     602.9       343.8      1953.4
REF     

In [5]:
def dunnett_p(groups, n=20000, seed=1):
    """Permutation analogue of Dunnett's test: Welch t for each treatment against
    the control, with family-wise error controlled by the max-|t| (Westfall-Young)
    permutation distribution. Assumption-free; preferred at n = 6 per group."""
    r = np.random.default_rng(seed)
    ctrl, trts = groups[0], groups[1:]
    def tstat(a, b):
        return (a.mean()-b.mean())/np.sqrt(a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b))
    obs = np.array([abs(tstat(g, ctrl)) for g in trts])
    labels = np.concatenate([[i]*len(g) for i, g in enumerate(groups)])
    pool = np.concatenate(groups)
    cnt = np.zeros(len(trts))
    for _ in range(n):
        lb = r.permutation(labels); pc = pool[lb == 0]
        mx = max(abs(tstat(pool[lb == i+1], pc)) for i in range(len(trts)))
        cnt += (mx >= obs)
    return list((cnt+1)/(n+1))

print("scipy has native Dunnett:", hasattr(stats, "dunnett"), "- using permutation version either way")

scipy has native Dunnett: False - using permutation version either way


In [6]:
# ---------------------------- CELL X2 --------------------------------------
# ============================================================================
# X2  EXPLORATORY TESTS
# ============================================================================
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf, statsmodels.api as sm
pd.set_option("display.width", 250)

print("="*96)
print("1  HARVEST TIMING: is subsoil particulate C a decaying slash pulse?")
print("="*96)
print(lf[["trt","soil_plot","sub","tsh","intensity","lf_mass_gm2","LF_C_gm2","hPOM_C_gm2","MAOM_C_gm2"]]
      .sort_values(["trt","tsh"]).round(1).to_string(index=False))
for t in ("DLC","STS"):
    g  = lf[lf["trt"]==t]
    rc = g[g["tsh"]==g["tsh"].min()]; ol = g[g["tsh"]==g["tsh"].max()]
    rank = int((g["lf_mass_gm2"] < ol["lf_mass_gm2"].iloc[0]).sum()) + 1
    print("\n  %s: %d plots at %d yr = %.0f g/m2 | 1 plot at %d yr = %.0f g/m2 | older plot ranks %d of %d"
          % (t, len(rc), rc["tsh"].iloc[0], rc["lf_mass_gm2"].mean(),
             ol["tsh"].iloc[0], ol["lf_mass_gm2"].iloc[0], rank, len(g)))
print("\n  (a decaying pulse predicts the older plot ranks LOWEST)")
h = lf[lf["trt"]!="REF"]
for x, lab, d in [("tsh","years since harvest (12 harvested plots)",h),
                  ("intensity","removal intensity per entry (12 harvested plots)",h),
                  ("cum_harv","cumulative harvested C, 1958-2020 (18 plots)",lf),
                  ("n_harv","number of harvest entries (18 plots)",lf)]:
    r, p = stats.pearsonr(d[x], d["lf_mass_gm2"])
    print("  LF mass vs %-52s r = %+.2f, P = %.3f" % (lab, r, p))

print("\n"+"="*96)
print("2  PERMANOVA: does SOM composition differ where single pools do not?")
print("="*96)
def permanova(X, labels, n=9999):
    X = np.asarray(X, float); labels = np.asarray(labels)
    Z = (X - X.mean(0))/X.std(0, ddof=1)
    D = np.sqrt(((Z[:,None,:]-Z[None,:,:])**2).sum(-1))
    N = len(Z); grps = np.unique(labels)
    def F(lb):
        SST = (D**2).sum()/(2*N)
        SSW = sum((D[np.ix_(np.where(lb==g)[0], np.where(lb==g)[0])]**2).sum()/(2*(lb==g).sum()) for g in grps)
        SSA = SST - SSW
        return (SSA/(len(grps)-1))/(SSW/(N-len(grps))), SSA/SST
    obs, R2 = F(labels)
    c = sum(1 for _ in range(n) if F(rng.permutation(labels))[0] >= obs)
    return obs, R2, (c+1)/(n+1)
def permdisp(X, labels, n=9999):
    X = np.asarray(X, float); labels = np.asarray(labels)
    Z = (X - X.mean(0))/X.std(0, ddof=1)
    d, lb = [], []
    for g in np.unique(labels):
        Zi = Z[labels==g]; d.append(np.sqrt(((Zi-Zi.mean(0))**2).sum(1))); lb.append(labels[labels==g])
    d = np.concatenate(d); lb = np.concatenate(lb)
    obs = stats.f_oneway(*[d[lb==g] for g in np.unique(lb)])[0]
    c = sum(1 for _ in range(n) if stats.f_oneway(*[rng.permutation(d)[lb==g] for g in np.unique(lb)])[0] >= obs)
    return (c+1)/(n+1)
for dep in ("0-20","20-40"):
    d = fr[fr["depth_code"]==dep]
    for cols, lab in [(["LF_C_gm2","hPOM_C_gm2","MAOM_C_gm2"], "C fractions"),
                      (["LF_C_gm2","hPOM_C_gm2","MAOM_C_gm2","LF_N_gm2","hPOM_N_gm2","MAOM_N_gm2"], "C + N fractions")]:
        k = d.dropna(subset=cols)
        if k.groupby("trt").size().min() < 3: continue
        F_, R2, p = permanova(k[cols], k["trt"])
        print("  %-6s %-18s n=%2d  pseudo-F=%5.2f  R2=%.2f  P=%.4f  | dispersion P=%.3f"
              % (dep, lab, len(k), F_, R2, p, permdisp(k[cols], k["trt"])))
k = lf.dropna(subset=["lf_mass_gm2"])
cols = ["lf_mass_gm2","hPOM_C_gm2","MAOM_C_gm2"]
F_, R2, p = permanova(k[cols], k["trt"])
print("  20-40  LF MASS + hPOM + MAOM  n=%d  pseudo-F=%5.2f  R2=%.2f  P=%.4f  | dispersion P=%.3f"
      % (len(k), F_, R2, p, permdisp(k[cols], k["trt"])))
for a, b in (("REF","DLC"),("REF","STS"),("DLC","STS")):
    kk = k[k["trt"].isin([a,b])]
    F_, R2, p = permanova(kk[cols], kk["trt"], n=4999)
    print("     pairwise %s vs %s: pseudo-F=%5.2f  R2=%.2f  P=%.4f" % (a, b, F_, R2, p))
print("  NOTE a significant dispersion test means the result is partly unequal SPREAD, not location")

print("\n"+"="*96)
print("3  DEPTH DISTRIBUTION AND ROOT ALLOCATION")
print("="*96)
for c, lab in [("frac_subsoil","soil C below 20 cm (fraction of 0-40)"),
               ("frac_root_subsoil","fine roots below 20 cm (fraction of 0-40)")]:
    g = gg(pv, c); du = dunnett_p(g)
    print("  %-40s REF=%.3f DLC=%.3f STS=%.3f | ANOVA P=%.4f perm P=%.4f KW P=%.4f | Dunnett %.4f / %.4f"
          % (lab, g[0].mean(), g[1].mean(), g[2].mean(), stats.f_oneway(*g)[1], perm_p(g),
             stats.kruskal(*g)[1], du[0], du[1]))
    if c == "frac_root_subsoil":
        for t, x in zip(("DLC","STS"), (g[1], g[2])):
            bs = [hedges_g(rng.choice(x,len(x)), rng.choice(g[0],len(g[0]))) for _ in range(10000)]
            print("      %s vs REF: Hedges' g = %+.2f (95%% CI %+.2f to %+.2f)"
                  % (t, hedges_g(x,g[0]), np.percentile(bs,2.5), np.percentile(bs,97.5)))
g = gg(pv,"rootC")
print("  %-40s REF=%.2f DLC=%.2f STS=%.2f | ANOVA P=%.4f perm P=%.4f"
      % ("total fine-root C (Mg C/ha)", g[0].mean(), g[1].mean(), g[2].mean(), stats.f_oneway(*g)[1], perm_p(g)))
print("  root:shoot ratio is DESCRIPTIVE ONLY - every plot in a watershed shares one")
print("  aboveground denominator, so a test on the ratio has no plot-level error term:")
for t in O:
    print("      %s: root C %.2f / AGB C %.0f  ->  %.4f"
          % (t, pv.loc[pv["trt"]==t,"rootC"].mean(), m.loc[m["trt"]==t,"agbC_Mgha"].mean(),
             pv.loc[pv["trt"]==t,"rootC"].mean()/m.loc[m["trt"]==t,"agbC_Mgha"].mean()))

print("\n"+"="*96)
print("4  IS THE SUBSOIL SIGNAL MEDIATED BY ROOTS?")
print("="*96)
j = lf.merge(pv[["soil_plot","frac_root_subsoil","rootC"]], on="soil_plot")
j = j.merge(sd[sd["depth_code"]=="20-40"][["soil_plot","fine_root_gm2","soil_C_gm2"]], on="soil_plot")
for x, lab in [("fine_root_gm2","subsoil fine-root biomass"),("frac_root_subsoil","subsoil root fraction"),
               ("soil_C_gm2","subsoil bulk soil C")]:
    r, p = stats.pearsonr(j[x], j["lf_mass_gm2"]); rs, ps = stats.spearmanr(j[x], j["lf_mass_gm2"])
    print("  LF mass vs %-28s r=%+.2f P=%.3f | Spearman rho=%+.2f P=%.3f" % (lab, r, p, rs, ps))
f1 = smf.ols("lf_mass_gm2 ~ C(trt) + fine_root_gm2", j).fit(); a1 = sm.stats.anova_lm(f1, typ=2)
print("  model LF mass ~ treatment + subsoil roots: treatment P=%.4f, roots P=%.4f, R2=%.2f"
      % (a1.loc["C(trt)","PR(>F)"], a1.loc["fine_root_gm2","PR(>F)"], f1.rsquared))

print("\n"+"="*96)
print("5  FDR CORRECTION ACROSS ALL EXPLORATORY TESTS")
print("="*96)
res = []
def add(lab, g):
    du = dunnett_p(g)
    res.append({"test":lab,"REF":g[0].mean(),"DLC":g[1].mean(),"STS":g[2].mean(),
                "P":stats.f_oneway(*g)[1],"KW":stats.kruskal(*g)[1],"D_DLC":du[0],"D_STS":du[1]})
for dep in ("0-20","20-40"):
    d = fr[fr["depth_code"]==dep].copy()
    d["MAOM_CN"] = d["MAOM_C_gm2"]/d["MAOM_N_gm2"]
    d["hPOM_CN"] = d["hPOM_C_gm2"]/d["hPOM_N_gm2"]
    for c, lab in [("MAOM_CN","MAOM C:N"),("hPOM_CN","heavy POM C:N"),
                   ("MAOM_C_gm2","MAOM C"),("hPOM_C_gm2","heavy POM C")]:
        g = gg(d, c)
        if min(len(x) for x in g) >= 3: add("%s %s" % (dep, lab), g)
add("20-40 LF mass",           gg(lf,"lf_mass_gm2"))
add("subsoil root fraction",   gg(pv,"frac_root_subsoil"))
add("subsoil soil-C fraction", gg(pv,"frac_subsoil"))
add("fine roots 20-40 cm",     gg(sd,"fine_root_gm2","20-40"))
add("total fine-root C",       gg(pv,"rootC"))
R = pd.DataFrame(res)
R["q_FDR"] = multipletests(R["P"], method="fdr_bh")[1]
print(R.round(4).to_string(index=False))
print("\n  q < 0.05:", R.loc[R["q_FDR"]<0.05,"test"].tolist())
print("  q < 0.10:", R.loc[R["q_FDR"]<0.10,"test"].tolist())
R.to_csv(os.path.join(PROJ,"exploratory_results.csv"), index=False)
print("\n  saved:", os.path.join(PROJ,"exploratory_results.csv"))

1  HARVEST TIMING: is subsoil particulate C a decaying slash pulse?
trt  soil_plot  sub  tsh  intensity  lf_mass_gm2  LF_C_gm2  hPOM_C_gm2  MAOM_C_gm2
DLC          7 WS2A    2       41.2       3759.5     954.9       158.1      1068.5
DLC          8 WS2A    2       41.2       4795.4    1628.5       281.7      2171.1
DLC         10 WS2A    2       41.2       1532.4     645.6       221.5      1700.2
DLC         11 WS2A    2       41.2       2391.1     917.2       225.3      1157.9
DLC         12 WS2A    2       41.2       1502.6     605.9       137.9      1377.2
DLC          9 WS2B    5       32.2       4356.4    1306.0      2114.2      2017.1
REF         13 WS4A  112        0.0       1010.0     436.7       180.4      1040.2
REF         14 WS4A  112        0.0       1534.0     602.9       343.8      1953.4
REF         15 WS4A  112        0.0        817.4     308.7       354.6      1085.4
REF         16 WS4A  112        0.0       1420.4     532.1       236.4      2651.3
REF         17 WS4A